# EDA Completo — Incidentes y Cambios en Aplicativos
## Notebook autocontenido para Jupyter local

---

### ¿Qué es este notebook?

Este notebook documenta **todo el proceso de Exploratory Data Analysis (EDA)** realizado para construir la base analítica del proyecto de predicción: *¿qué cambios en aplicativos van a producir incidentes?*

Está diseñado para correr completamente en **Python local** (Jupyter) sin ninguna dependencia de Databricks o Spark.

### Objetivo del proyecto

Construir un modelo que, dado un cambio en un aplicativo, estime la **probabilidad de que ese cambio cause un incidente**. El modelo se integra al proceso del CAB (Change Advisory Board) para priorizar la revisión de cambios de alto riesgo.

### Objetivo de este EDA

1. Entender la estructura y calidad de los datos disponibles
2. Definir correctamente la variable objetivo (`causo_incidente`)
3. Identificar y descartar variables con riesgo de *data leakage*
4. Construir y validar las features disponibles antes del evento
5. Entender la relación de esas features con la variable objetivo
6. Generar insumos listos para el pipeline de modelado

### Cómo correr este notebook

```bash
pip install pandas numpy matplotlib scikit-learn xgboost openpyxl scipy
jupyter notebook
```

Luego ejecuta las celdas en orden de arriba a abajo.

---
## 🆕 Novedades de la versión 2 (changelog)

Esta versión **mantiene íntegra la estructura y narrativa del EDA original** y aplica dos tipos de cambios: correcciones sobre lo existente y secciones nuevas.

### Correcciones a la v1 (bugs y metodología)

| # | Dónde | Qué se corrigió |
|---|---|---|
| 1 | Sección 8 | `zip(inc['cxc_n'].dropna(), inc['ticket_n'])` **desalineaba filas** al construir los pares cambio↔incidente (el i-ésimo valor no nulo se emparejaba con el i-ésimo ticket global) y además se inyectaban pares invertidos `(i,c)`. El target podía quedar mal etiquetado |
| 2 | Sección 11 | `cambios_solapados` y `cambios_ci_7d/30d` usaban fechas calculadas **antes** de reordenar el DataFrame → cada fila recibía la fecha de otra fila. Se recalculan tras el `sort_values` |
| 3 | Sección 11 | `hist_fallos_previos_ci` contaba un fallo previo aunque el **incidente aún no hubiera ocurrido** a la fecha del cambio actual (fuga sutil de futuro). Ahora se compara contra la fecha real del incidente |
| 4 | Sección 17 | El grid de hiperparámetros **elegía al ganador mirando el test** → el test dejaba de ser una estimación honesta. Ahora hay split train/val/test (64/16/20) y la selección ocurre en validación |
| 5 | Secciones 19 y 21 | El **umbral EQopt también se elegía sobre el test**. Ahora se elige en validación y se congela antes de tocar el test |
| 6 | Sección 19 | La learning curve usaba `cv=3` (KFold aleatorio) → mezclaba futuro y pasado. Ahora usa `TimeSeriesSplit` |
| 7 | Sección 9 | La comparación de targets dependía de features creadas en la Sección 11 (que va después) → corría sin features. Se construyen versiones mínimas inline |
| 8 | Sección 14 | `N_POS` y `PREV` no existían (NameError); se usan `N_D1`/`PREV_D1` |
| 9 | Sección 15 | Código muerto (`... if False else ...`) eliminado; `use_label_encoder` (deprecado) eliminado; hardcodes (14764, factor x4) parametrizados |
| 10 | Setup | Ruta del Excel como raw-string + override por variable de entorno; placeholder `'sin informacion'` duplicado (ahora incluye la versión con tilde); celda pip sin IP interna |

### Secciones nuevas (v2)

| Sección | Contenido |
|---|---|
| **7B** | NLP mejorado: stopwords en español, detección de boilerplate, keywords de riesgo TI |
| **11B** | Nuevas features: recencia (días desde último cambio/incidente del CI), historial del grupo ejecutor, señales de texto |
| **15B** | Justificación explícita de la métrica de optimización (PR-AUC + recall@10%) |
| **17B** | Benchmark de algoritmos: Dummy, LR, Random Forest, HistGradientBoosting, XGBoost, LightGBM (opcional) y variante TF-IDF→SVD |
| **21B** | Calibración de probabilidades (Platt e isotónica) con curva de calibración y Brier score — se adelanta del MVP 2 al MVP 1 |
| **22B** | Intervalos de confianza bootstrap para ROC-AUC / PR-AUC (con ~30 positivos en test, el punto estimado engaña) |
| **23B** | Persistencia del modelo (joblib) + función de scoring para nuevos cambios |
| **23C** | Base de monitoreo: PSI de scores y features (drift) |
| **25** | Conclusiones v2 y roadmap actualizado |

> ⚠️ **Nota sobre las cifras citadas en los textos:** las métricas mencionadas en las conclusiones originales (ROC-AUC 0.8781, gap 0.0153, etc.) provienen de la corrida v1. Al re-ejecutar con las correcciones (especialmente #1-#5) los números pueden variar — y serán más confiables.


---
## 🆕 Novedades de la versión 3 (changelog)

La v3 profundiza el rigor metodológico y añade análisis de robustez sobre features, target y texto:

| Sección | Contenido |
|---|---|
| **15 (mod.)** | Los cortes del split train/val/test se ajustan (dentro de una ventana estrecha y **sin romper la cronología**) para equilibrar la prevalencia entre bloques |
| **15C** | Validación formal del split: **regla del 3%** (diferencia de prevalencia entre bloques), IC de Wilson por bloque, PSI y test KS de similitud de poblaciones |
| **7C** | Análisis del corpus: tamaño de vocabulario, ley de Zipf, n-gramas dominantes, concentración del vocabulario |
| **9B** | Robustez del target: estabilidad temporal de la prevalencia (IC Wilson), completitud del label por mes (deriva de documentación), tiempo cambio→incidente por segmento, tests χ² de asociación |
| **12B** | Features ampliado: **información mutua**, **Information Value (IV/WOE)** con la convención bancaria, matriz de redundancia (Spearman) y pares colineales |
| **12C** | Texto vs target: **palabras y bigramas discriminantes** (log-odds suavizado), tópicos NMF y tasa de incidente por tópico |
| **20B** | **Importancia por permutación** en validación (métrica: average precision) contrastada contra la importancia por gain de XGBoost |
| **22C** | **Backtesting rolling** (expanding window): reentrena todo con solo el pasado de cada fold y evalúa el bloque siguiente — estabilidad, tendencia y frecuencia de reentrenamiento |

> Los análisis 9B/12B/12C son **descriptivos sobre el dataset completo** (EDA). Todo lo que entra al modelo sigue ajustándose solo con train (TF-IDF, encoders) y seleccionándose solo con validación.


## Mapa de secciones

| # | Sección | Qué se hace |
|---|---|---|
| 0 | Setup | Instalación, imports, rutas |
| 1 | Carga de datos | Leer el Excel, primeras filas |
| 2 | Estandarización | Normalizar nombres de columnas y fechas |
| 3 | Duplicados | Detectar columnas multivalor y deduplicar |
| 4 | Calidad de datos | Cobertura, tipos, cardinalidad |
| 5 | EDA Incidentes | Distribuciones, series temporales, insights |
| 6 | EDA Cambios | Distribuciones, rollbacks, series temporales |
| 7 | Campos de texto | Campos NLP disponibles, longitudes, boilerplate |
| 8 | Conexión Inc↔Cam | El núcleo del EDA: cómo se une el target |
| 9 | Variable objetivo | 3 definiciones evaluadas, elección justificada |
| 10 | Leakage | Detección, validación, regla de exclusión |
| 11 | Feature engineering | 5 familias de features, historial de CI |
| 12 | Features vs target | Correlación, lift, distribuciones por clase |
| 13 | Análisis de CI | Presión de cambios por aplicativo, ventanas temporales |
| 14 | Conclusiones finales | Guía completa para el MVP 1 |

### Secciones añadidas en v2

| # | Sección | Qué se hace |
|---|---|---|
| 7B | NLP mejorado | Stopwords ES, boilerplate, keywords de riesgo |
| 11B | Features v2 | Recencia, historial del grupo, señales de texto |
| 15B | Métrica de optimización | Por qué PR-AUC + recall@10% |
| 17B | Benchmark de algoritmos | Dummy, RF, HistGB, LightGBM, XGB+SVD |
| 21B | Calibración | Platt vs isotónica, Brier, curva de calibración |
| 22B | Bootstrap | Intervalos de confianza de las métricas de test |
| 23B | Persistencia | Bundle joblib + `score_nuevos_cambios()` |
| 23C | Monitoreo | PSI de score y features |
| 25 | Conclusiones v2 | Cambios, lecturas y roadmap |


### Secciones añadidas en v3

| # | Sección | Qué se hace |
|---|---|---|
| 7C | Análisis del corpus | Vocabulario, Zipf, n-gramas, cobertura |
| 9B | Robustez del target | Estabilidad temporal, completitud del label, χ² |
| 12B | Features ampliado | Información mutua, IV/WOE, redundancia |
| 12C | Texto vs target | Log-odds discriminante, tópicos NMF |
| 15C | Validación del split | Regla del 3%, PSI, KS entre bloques |
| 20B | Permutación | Importancia que generaliza (validación) |
| 22C | Backtesting rolling | Estabilidad temporal del desempeño |


In [ ]:
# ============================================================
# v2: INSTALACION DE DEPENDENCIAS (ejecutar una sola vez)
# ============================================================
# Descomenta la linea que aplique a tu entorno:
#
# %pip install pandas numpy matplotlib scikit-learn xgboost openpyxl scipy joblib -q
# %pip install lightgbm -q          # opcional, para la Seccion 17B
#
# Si trabajas detras del repositorio interno (Nexus), agrega tus parametros:
# %pip install xgboost openpyxl -q --trusted-host <HOST> --index-url http://<HOST>:8081/repository/pypi-packages/simple/
print('Dependencias: pandas, numpy, matplotlib, scikit-learn, xgboost, openpyxl, scipy, joblib (+ lightgbm opcional)')


In [ ]:
# ============================================================
# SECCION 0: SETUP
# ============================================================
# Antes de correr este notebook, ejecuta en terminal:
#   pip install pandas numpy matplotlib scikit-learn xgboost openpyxl scipy joblib
#   pip install lightgbm   # opcional (Seccion 17B)
#
# En Jupyter puedes descomentar la linea siguiente:
# !pip install pandas numpy matplotlib scikit-learn xgboost openpyxl scipy -q

import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10
})

# ============================================================
# CONFIGURACION DE RUTAS
# Ajusta EXCEL_PATH a la ubicacion de tu archivo local
# ============================================================
import os
EXCEL_PATH = Path(r"C:\Users\u50014\OneDrive - BANCO POPULAR DOMINICANO\CLTV-D&A\18. Side Quests\TI\mvp1\Data Incidentes y Cambios-Copy1.xlsx")
# v2: puedes sobreescribir la ruta sin tocar el codigo definiendo la
# variable de entorno DATA_INCIDENTES_XLSX antes de abrir Jupyter
if os.environ.get('DATA_INCIDENTES_XLSX'):
    EXCEL_PATH = Path(os.environ['DATA_INCIDENTES_XLSX'])

# Placeholders textuales que equivalen a nulo
PLACEHOLDERS_NULOS = {
    '', 'na', 'n/a', 'n.a', 'nan', 'none', 'null', '.', '-', '--',
    'no aplica', 'noaplica', 'sin informacion', 'sin información',  # v2: version con tilde (antes estaba duplicada la misma cadena)
    'ninguno', 'ninguna', 'n/d', 'nd', 'pendiente',
}

print(f'pandas {pd.__version__}  |  numpy {np.__version__}')
print(f'Archivo de datos: {EXCEL_PATH.resolve()}')
print(f'Existe: {EXCEL_PATH.exists()}')


---
## Sección 1 — Carga de datos

### Por qué empezamos aquí

El primer paso es entender qué tenemos. Antes de cualquier transformación, vemos:
* Cuántas filas y columnas tiene cada hoja
* Los nombres **reales** de las columnas (relevante porque luego los normalizaremos)
* Las primeras filas para tener contexto visual

### Estructura del archivo

El Excel tiene **dos hojas**:
- `Incidentes` — 1 fila por evento de incidente (puede haber duplicados por el mismo ticket con estados distintos)
- `Cambios` — 1 fila por evento de cambio (también puede tener duplicados por el mismo cambio)

**Atención:** los duplicados son intencionales en el sistema origen (representa el historial de estados). Los manejaremos en la Sección 3.

In [ ]:
# ============================================================
# SECCION 1: CARGA DE DATOS
# ============================================================
xls = pd.ExcelFile(EXCEL_PATH)
print(f'Hojas disponibles: {xls.sheet_names}')

# Deteccion automatica de hojas por nombre
HOJA_INC = next((h for h in xls.sheet_names if 'incidente' in h.lower()), xls.sheet_names[0])
HOJA_CAM = next((h for h in xls.sheet_names if 'cambio'   in h.lower()), xls.sheet_names[1])
print(f'  -> Incidentes : "{HOJA_INC}"')
print(f'  -> Cambios    : "{HOJA_CAM}"')

df_incidentes = pd.read_excel(xls, sheet_name=HOJA_INC)
df_cambios    = pd.read_excel(xls, sheet_name=HOJA_CAM)

print(f'\nIncidentes (crudo): {df_incidentes.shape[0]:,} filas x {df_incidentes.shape[1]} columnas')
print(f'Cambios    (crudo): {df_cambios.shape[0]:,} filas x {df_cambios.shape[1]} columnas')
print(f'\nColumnas de incidentes:\n  {list(df_incidentes.columns)}')
print(f'\nColumnas de cambios:\n  {list(df_cambios.columns)}')


---
## Sección 2 — Estandarización: nombres de columnas y fechas

### Por qué es necesario

Los nombres de columna en Excel vienen con espacios, mayúsculas, tildes y caracteres especiales. Si los usamos así:
- Se producen errores al hacer `df['columna con espacio']`
- Los merges por nombre fallan silenciosamente
- El código es frágil si el Excel cambia un espacio

**Solución:** `limpiar_columnas()` convierte todo a `lower_case_underscore` de forma determinista.

### Por qué la detección de fechas es conservadora

No convertimos todas las columnas que parezcan fechas — muchas tienen el mismo formato numérico pero son códigos (ej. `codigocierre`). Convertimos solo las que:
1. Su **nombre** contiene pistas de fecha (`fecha`, `date`, `outage`)
2. Al menos el 50% de los valores no nulos parsean correctamente como datetime

In [ ]:
# ============================================================
# SECCION 2: ESTANDARIZACION
# ============================================================

def limpiar_columnas(df):
    """
    Normaliza nombres de columnas:
      - lowercase
      - espacios y / -> underscore
      - quita caracteres especiales (conserva tildes y eñe en modo unicode)
    """
    df = df.copy()
    df.columns = (
        df.columns.str.lower().str.strip()
        .str.replace('/', '_', regex=False)
        .str.replace(r'\s+', '_', regex=True)
        .str.replace(r'[^\w_]', '', regex=True)
    )
    return df


PISTAS_FECHA = ('fecha', 'date', 'outage')

def convertir_fechas(df, pistas=PISTAS_FECHA, umbral=0.5):
    """
    Convierte a datetime solo columnas cuyo nombre sugiere fecha
    Y que realmente parsean (>= umbral de valores no nulos validos).
    Evita falsos positivos como 'codigocierre'.
    """
    df = df.copy()
    cols_fecha = []
    for c in df.columns:
        if not any(p in c for p in pistas):
            continue
        conv = pd.to_datetime(df[c], format='mixed', errors='coerce')
        base = int(df[c].notna().sum())
        if base == 0 or conv.notna().sum() / base >= umbral:
            df[c] = conv
            cols_fecha.append(c)
    return df, cols_fecha


# Aplicar a ambas tablas
incidentes = limpiar_columnas(df_incidentes)
cambios    = limpiar_columnas(df_cambios)
incidentes, fechas_inc = convertir_fechas(incidentes)
cambios,    fechas_cam = convertir_fechas(cambios)

print('Fechas detectadas en incidentes:', fechas_inc)
print('Fechas detectadas en cambios   :', fechas_cam)
print(f'\nColumnas incidentes (post-limpieza):\n  {list(incidentes.columns)}')
print(f'\nColumnas cambios (post-limpieza):\n  {list(cambios.columns)}')


---
## Sección 3 — Diagnóstico de duplicados y deduplicación

### Por qué hay duplicados y por qué importa

El sistema origen registra **1 fila por cada cambio de estado** de un ticket o cambio. Eso significa que un mismo cambio puede tener 5-10 filas con la misma información excepto el estado y la fecha.

Si deduplicamos simplemente con `drop_duplicates('cambio')`, perdemos información: algunos campos como `configuration_item` son **multivalor** (varios CIs separados por `|`) que acumulan valor a través de las filas.

### Estrategia

La función `columnas_variantes` detecta **automáticamente** qué columnas tienen más de un valor distinto por ID. Luego la función `deduplicar` aplica la estrategia correcta por tipo:

| Tipo de columna | Estrategia |
|---|---|
| Columna multivalor (ej. `configuration_item`) | Concatenar valores únicos con `|` |
| Estado temporal (ej. `prioridad`) | Tomar primero o último valor según convenga |
| Resto de columnas | Primer valor no nulo |

In [ ]:
# ============================================================
# SECCION 3: DETECCION DE DUPLICADOS
# ============================================================

def columnas_variantes(df, id_col):
    """
    Por cada columna: nº de IDs que tienen mas de un valor
    distinto (ignora nulos). Columnas con valor > 0 son
    candidatas a concatenar antes de deduplicar.
    """
    g = df.groupby(id_col)
    res = {}
    for col in df.columns:
        if col == id_col:
            continue
        nun = g[col].nunique(dropna=True)
        res[col] = int((nun > 1).sum())
    return pd.Series(res, name='ids_con_multiples_valores').sort_values(ascending=False)


print('>>> INCIDENTES — columnas que varian dentro de un mismo ticket:')
var_inc = columnas_variantes(incidentes, 'ticket')
print(var_inc[var_inc > 0].to_frame().to_string())

print('\n>>> CAMBIOS — columnas que varian dentro de un mismo cambio:')
var_cam = columnas_variantes(cambios, 'cambio')
print(var_cam[var_cam > 0].to_frame().to_string())

print(f'\nTotal IDs incidentes: {incidentes["ticket"].nunique():,}')
print(f'Total filas incidentes raw: {len(incidentes):,}')
print(f'Total IDs cambios: {cambios["cambio"].nunique():,}')
print(f'Total filas cambios raw: {len(cambios):,}')


In [ ]:
# ============================================================
# FUNCION DE DEDUPLICACION
# Maneja: multivalor (concat), estado temporal (first/last),
# resto de columnas (primer valor no nulo).
# ============================================================

COLS_INCIDENTE_LIGADO = ['incidente', 'categoria_incidente', 'servicio_afectado',
                          'prioridad_incidente', 'estado_incidente']

def deduplicar(df, id_col, cols_multivalor, cols_estado_temporal=None,
               estrategia_temporal='last', orden_por=None, sep=' | '):
    df = df.copy()
    cols_estado_temporal = cols_estado_temporal or []
    if orden_por and orden_por in df.columns:
        df = df.sort_values([id_col, orden_por])
    concat_unicos = lambda s: sep.join(sorted(map(str, pd.Series(s).dropna().unique()))) or pd.NA
    agg = {}
    for c in df.columns:
        if c == id_col: continue
        if c in cols_multivalor:       agg[c] = concat_unicos
        elif c in cols_estado_temporal: agg[c] = estrategia_temporal
        else:                           agg[c] = 'first'
    out = df.groupby(id_col, as_index=False).agg(agg)
    return out[[id_col] + [c for c in df.columns if c != id_col]]


# --- Deduplicar incidentes ---
mv_inc = list(var_inc[var_inc > 0].index)
incidentes_dedup = deduplicar(incidentes, 'ticket', mv_inc, orden_por='open_date')

# --- Deduplicar cambios ---
# IMPORTANTE: separar las columnas que son el target / etiqueta
# antes de deduplicar, para que no inflen o pierdan informacion
cambios_sin_inc = cambios.drop(columns=[c for c in COLS_INCIDENTE_LIGADO
                                         if c in cambios.columns])
mv_cam = list(var_cam[var_cam > 0].index)
mv_cam = [c for c in mv_cam if c in cambios_sin_inc.columns]
cambios_dedup = deduplicar(cambios_sin_inc, 'cambio', mv_cam, orden_por='fecha_apertura')

print(f'Incidentes: {len(incidentes):,} filas -> {len(incidentes_dedup):,} unicos')
print(f'Cambios   : {len(cambios):,} filas -> {len(cambios_dedup):,} unicos')
assert incidentes_dedup['ticket'].is_unique,  'Aun hay tickets duplicados'
assert cambios_dedup['cambio'].is_unique,     'Aun hay cambios duplicados'
print('OK: un registro por ID en ambas tablas.')


### ✔ Conclusiones Sección 3: deduplicación

* **Los duplicados no son un error, son el historial de estados**: el sistema origen registra cada transición. Hay que colapsar correctamente.
* **`drop_duplicates()` simple pierde información**: `configuration_item` y otros campos multivalor se llenan incrementalmente a lo largo de las filas del mismo ID — un `first` o `last` descarta esa información.
* **El campo `incidente` en cambios se aparta ANTES de deduplicar**: si lo incluyéramos en el colapso, podría multiplicar o borrar pares cambio↔incidente que son los que definen el target.
* **El resultado son tablas limpias**: 1 fila por ID único, con la información más completa de cada campo.

---
## Sección 4 — Calidad de datos y perfilado

### Por qué perfilar antes de modelar

Sin saber la cobertura real de cada columna, podemos:
- Usar features que en producción llegan vacías la mayoría del tiempo
- Calcular medias y correlaciones sobre muestras sesgadas
- Confundir la **ausencia** de dato con el valor por defecto de pandas (0, `NaN`)

### Qué buscamos

| Indicador | Acción si falla |
|---|---|
| Cobertura < 50% | Evaluar si usar como feature o descartar |
| Columna constante | Descartar (aporta cero información) |
| Cardinalidad = n filas | Posiblemente un ID, no usar como categoríca |
| Cobertura sólo en positivos | ALERTA de leakage (se llena tras el incidente) |

In [ ]:
# ============================================================
# SECCION 4: CALIDAD DE DATOS
# ============================================================

def perfil_calidad(df, nombre):
    n = len(df)
    rows = []
    for c in df.columns:
        s = df[c]
        no_nulo = int(s.notna().sum())
        rows.append(dict(
            columna=c, dtype=str(s.dtype),
            cobertura_pct=round(100*no_nulo/n, 1),
            n_unicos=int(s.nunique(dropna=True)),
            constante=s.nunique(dropna=True) <= 1,
        ))
    df_out = pd.DataFrame(rows).sort_values('cobertura_pct')
    print(f'\n=== {nombre}: {n:,} filas x {len(df.columns)} columnas ===')
    print(f'  <50% cobertura: {(df_out["cobertura_pct"] < 50).sum()}')
    print(f'  <80% cobertura: {(df_out["cobertura_pct"] < 80).sum()}')
    print(f'  Constantes    : {df_out["constante"].sum()}')
    return df_out

perf_inc = perfil_calidad(incidentes_dedup, 'INCIDENTES')
perf_cam = perfil_calidad(cambios_dedup,    'CAMBIOS')

print('\n--- INCIDENTES: columnas con < 80% cobertura ---')
low = perf_inc[perf_inc['cobertura_pct'] < 80]
print(low[['columna','cobertura_pct','n_unicos']].to_string(index=False) if len(low) else '  (ninguna)')

print('\n--- CAMBIOS: columnas con < 80% cobertura ---')
low = perf_cam[perf_cam['cobertura_pct'] < 80]
print(low[['columna','cobertura_pct','n_unicos']].to_string(index=False) if len(low) else '  (ninguna)')


In [ ]:
# Mapa visual de cobertura
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (prof, titulo) in zip(axes, [(perf_cam,'Cambios'), (perf_inc,'Incidentes')]):
    df_show = prof.sort_values('cobertura_pct').head(25)
    colors = ['#e74c3c' if v < 50 else '#f39c12' if v < 80 else '#2e86de'
              for v in df_show['cobertura_pct']]
    ax.barh(range(len(df_show)), df_show['cobertura_pct'],
            color=colors, alpha=0.85, edgecolor='white')
    ax.set_yticks(range(len(df_show)))
    ax.set_yticklabels(df_show['columna'], fontsize=8)
    ax.axvline(50, color='#e74c3c', ls='--', lw=1.5, label='50%')
    ax.axvline(80, color='#f39c12', ls='--', lw=1.5, label='80%')
    ax.set_title(f'Cobertura de columnas — {titulo}', fontweight='bold')
    ax.set_xlabel('%'); ax.legend(fontsize=8)
    ax.set_xlim(0, 105)

from matplotlib.patches import Patch
leg = [Patch(facecolor='#e74c3c',label='<50%'), Patch(facecolor='#f39c12',label='<80%'),
       Patch(facecolor='#2e86de',label='>80%')]
fig.legend(handles=leg, loc='lower center', ncol=3, fontsize=9)
plt.suptitle('Mapa de cobertura de columnas (25 peores por tabla)', fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


### ✔ Conclusiones Sección 4: calidad de datos

* **Varias columnas tienen cobertura baja** en cambios: `business_justification`, `plan_reverso`, `planreverso`, `cantidadrecalendarizacion`. Esto **no las hace inútiles** — su presencia/ausencia es en sí misma una señal predictiva
* **Las columnas constantes se eliminan**: aportan cero varianza al modelo
* **Las columnas de leakage tienen coberturas variables**: algunas solo se llenan cuando hay incidente, lo que explica su alta correlación con el target — esto se explora en la Sección 10
* **Decisiones tomadas**: ninguna columna se eliminó a priori por cobertura; la selección se basa en el rol temporal (predictor vs. fuga vs. etiqueta)

---
## Secciones 5 y 6 — Análisis descriptivo: Incidentes y Cambios

### Por qué hacemos este análisis

Antes de construir el target o las features, necesitamos entender las distribuciones de las variables clave:
- ¿Están los datos concentrados en pocos valores o bien distribuidos?
- ¿Hay valores atípicos o categorias desconocidas?
- ¿La distribución temporal es estable o hay picos?
- ¿Los cambios de mayor riesgo tienen mayor tasa de reversado?

### Hallazgos clave anticipados (del EDA original)

**Incidentes:**
- ≈83% se concentran en P4-Baja y P3-Media
- ≈60% categoría NOC.ONLINE.MANUAL
- ≈42 incidentes promedio por mes (pico ≈65 en Oct 2025 y Abr 2026)

**Cambios:**
- ≈88.7% son tipo Normal o Estándar
- ≈84.6% riesgo Medio o Bajo
- A mayor riesgo, mayor % de reversado (señal predictiva importante)

In [ ]:
# ============================================================
# SECCION 5: EDA INCIDENTES
# ============================================================

def barras_top(serie, titulo, top=10, color='#3498db'):
    vc = serie.astype(str).str.strip().value_counts().head(top)
    if vc.empty: print('(sin datos)', titulo); return
    fig, ax = plt.subplots(figsize=(8, max(2, 0.4*len(vc))))
    vc[::-1].plot(kind='barh', ax=ax, color=color, alpha=0.85, edgecolor='white')
    for p in ax.patches:
        ax.text(p.get_width()+0.5, p.get_y()+p.get_height()/2,
                f'{int(p.get_width())}', va='center', fontsize=8)
    ax.set_title(titulo, fontweight='bold'); ax.set_xlabel('n registros')
    plt.tight_layout(); plt.show()


for col, tit in [
    ('prioridad',          'Incidentes por prioridad'),
    ('categoria',          'Incidentes por categoria'),
    ('tipo_de_afectacion', 'Tipo de afectacion'),
    ('servicio_afectado',  'Servicio afectado (top 10)'),
    ('sistema_aplicacion', 'Sistema / aplicacion (top 10)'),
]:
    if col in incidentes_dedup.columns:
        barras_top(incidentes_dedup[col], tit)

# Banderas de incidentes mayores, SLA, prime
for col in ['incidente_mayor', 'cumplimiento_sla', 'prime__no_prime']:
    if col in incidentes_dedup.columns:
        print(f'\n{col.upper()}:')
        print(incidentes_dedup[col].astype(str).str.strip().value_counts(dropna=False).to_string())

# Serie temporal mensual
if 'open_date' in incidentes_dedup.columns:
    fechas = pd.to_datetime(incidentes_dedup['open_date'], errors='coerce').dropna()
    # resample necesita DatetimeIndex: agrupar por periodo mensual
    serie  = fechas.dt.to_period('M').value_counts().sort_index()
    serie.index = serie.index.to_timestamp()
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(serie.index, serie.values, 'o-', color='#2e86de', lw=2.5, ms=5)
    ax.fill_between(serie.index, serie.values, alpha=0.12, color='#2e86de')
    ax.axhline(serie.mean(), color='gray', ls='--', lw=1,
               label=f'Promedio mensual: {serie.mean():.0f}')
    ax.set_title('Serie temporal — Incidentes por mes', fontweight='bold')
    ax.set_ylabel('n incidentes'); ax.legend()
    plt.tight_layout(); plt.show()
    print(f'\nPeriodo: {serie.index.min().date()} a {serie.index.max().date()}')
    print(f'Total incidentes: {serie.sum():,}  |  Promedio mensual: {serie.mean():.1f}')
    print(f'Pico: {serie.idxmax().date()} con {serie.max()} incidentes')


In [ ]:
# ============================================================
# SECCION 6: EDA CAMBIOS
# ============================================================

for col, tit, color in [
    ('tipo_de_cambio', 'Cambios por tipo',       '#3498db'),
    ('riesgo',         'Cambios por riesgo',      '#e74c3c'),
    ('prioridadch',    'Cambios por prioridad',   '#2e86de'),
    ('categoria',      'Cambios por categoria',   '#27ae60'),
    ('grupo',          'Grupo ejecutor (top 10)', '#f39c12'),
]:
    if col in cambios_dedup.columns:
        barras_top(cambios_dedup[col], tit, color=color)

# Banderas de aprobacion y rollbacks
for col in ['aprobado_cab', 'preaprobado', 'reversado', 'cambio_en_excepcion']:
    if col in cambios_dedup.columns:
        print(f'\n{col.upper()}:')
        print(cambios_dedup[col].astype(str).str.strip().value_counts(dropna=False).to_string())

# Rollbacks y recalendarizaciones
print('\n--- Metricas de rollback y recalendarizacion ---')
for col in ['cantidaddevolucion', 'cantidadrecalendarizacion', 'duración_programada']:
    if col in cambios_dedup.columns:
        v = pd.to_numeric(cambios_dedup[col], errors='coerce')
        print(f'{col}: media={v.mean():.2f}  max={v.max():.0f}  % > 0 = {100*(v>0).mean():.1f}%')

# Cruce: tasa de reversado por nivel de riesgo
if {'riesgo', 'reversado'}.issubset(cambios_dedup.columns):
    tmp = cambios_dedup.copy()
    tmp['rev_flag'] = tmp['reversado'].astype(str).str.strip().str.lower().eq('si')
    tasa_rev = tmp.groupby('riesgo')['rev_flag'].mean().mul(100).round(1).rename('pct_reversado')
    print('\nTasa de reversado por riesgo (SENIAL PREDICTIVA):')
    print(tasa_rev.sort_values(ascending=False).to_frame().to_string())

# Serie temporal de cambios
if 'fecha_apertura' in cambios_dedup.columns:
    fechas_c = pd.to_datetime(cambios_dedup['fecha_apertura'], errors='coerce').dropna()
    # resample necesita DatetimeIndex: agrupar por periodo mensual
    serie_c  = fechas_c.dt.to_period('M').value_counts().sort_index()
    serie_c.index = serie_c.index.to_timestamp()
    fig, ax  = plt.subplots(figsize=(11, 3.5))
    ax.plot(serie_c.index, serie_c.values, 'o-', color='#27ae60', lw=2.5, ms=5)
    ax.fill_between(serie_c.index, serie_c.values, alpha=0.12, color='#27ae60')
    ax.axhline(serie_c.mean(), color='gray', ls='--', lw=1,
               label=f'Promedio mensual: {serie_c.mean():.0f}')
    ax.set_title('Serie temporal — Cambios por mes', fontweight='bold')
    ax.set_ylabel('n cambios'); ax.legend()
    plt.tight_layout(); plt.show()


### ✔ Conclusiones Secciones 5-6: EDA descriptivo

**Incidentes:**
* Concentración en P4/P3 indica que la mayoría son incidentes operacionales menores, no catastróficos
* Solo 5 incidentes mayores en el período — el modelo necesita trabajar con el universo completo
* Distribución mensual estable con picos: el modelo deberá ser robusto a variaciones estacionales

**Cambios:**
* La concentración en Normal/Estándar implica que el modelo debe discriminar **dentro** de la categoría mayoritaria
* `riesgo` muestra correlación con rollbacks: Muy Alto tiene mayor % de reversado — señal predictiva directa
* `reversado` y `cantidaddevolucion` son atractivos como features pero tienen **riesgo de leakage** (ver Sección 10)
* La distribución temporal de cambios es más estable que la de incidentes: el pipeline tiene ritmo constante

---
## Sección 7 — Campos de texto libre (para NLP)

### Por qué el texto es importante

Los textos son la **mayor fuente de información diferenciadora** para este modelo. La descripción de un cambio contiene información que no está codificada en ningún campo estructurado: tecnología afectada, alcance del cambio, impacto potencial.

### Regla crítica: texto predictor vs texto resultado

| Tabla | Campo | Disponible antes del evento | Usar como feature |
|---|---|---|---|
| Cambios | `descripcion` | ✅ Sí | ✅ Sí |
| Cambios | `plan_implementacion` | ✅ Sí | ✅ Sí |
| Cambios | `plan_reverso` / `planreverso` | ✅ Sí | ✅ Sí |
| Cambios | `business_justification` | ✅ Sí (con cautela) | ⚠️ Monitorear |
| Incidentes | `causa_raiz` | ❌ No (post-incidente) | ❌ No |
| Incidentes | `accion_tomada` | ❌ No (post-incidente) | ❌ No |
| Incidentes | `summary` | ❌ No (post-incidente) | ❌ No |

### Hallazgos del EDA de texto

- Las palabras más frecuentes en `plan_implementacion`: *realizar, acceder, servidor, backup, validar, verificar*
- Hay boilerplate significativo: textos idénticos o casi idénticos entre cambios
- La **longitud** del texto es en sí misma una señal: cambios bien documentados tienen perfil distinto

In [ ]:
# ============================================================
# SECCION 7: CAMPOS DE TEXTO
# ============================================================

TEXTO_CAMBIOS    = ['descripcion', 'plan_implementacion', 'planreverso',
                    'plan_reverso', 'business_justification']
TEXTO_INCIDENTES = ['summary', 'description', 'causa_raiz', 'causa_diagnostico',
                    'accion_tomada', 'recomendaciones_oportunidades_de_mejora']

def limpiar_texto(serie):
    """Normaliza placeholders a NA y colapsa espacios."""
    s = serie.astype(str).str.strip()
    low = s.str.lower().str.strip()
    s = s.mask(low.isin(PLACEHOLDERS_NULOS), pd.NA)
    return s

def normalizar_nlp(texto):
    """Lowercase + quita acentos + quita caracteres especiales."""
    if pd.isna(texto): return ''
    t = unicodedata.normalize('NFKD', str(texto).lower()).encode('ascii','ignore').decode()
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()


# Estadisticas de texto para cambios
print('--- CAMPOS DE TEXTO EN CAMBIOS ---')
rows_txt = []
for c in TEXTO_CAMBIOS:
    if c not in cambios_dedup.columns: continue
    s_limpio = limpiar_texto(cambios_dedup[c])
    cobertura = round(100*s_limpio.notna().mean(), 1)
    len_med   = round(s_limpio.dropna().astype(str).str.len().median(), 0)
    n_unicos  = s_limpio.nunique(dropna=True)
    top1_pct  = round(100*s_limpio.value_counts().iloc[0]/s_limpio.notna().sum(), 1) if len(s_limpio.dropna()) > 0 else 0
    rows_txt.append(dict(campo=c, cobertura_pct=cobertura, len_mediana=len_med,
                         n_valores_unicos=n_unicos, pct_top_valor=top1_pct))
    print(f'  {c:30s}  cov={cobertura:5.1f}%  len_med={len_med:5.0f}  unicos={n_unicos:5d}  boilerplate_top={top1_pct:.1f}%')

# Generar campo de texto combinado para TF-IDF
txt_disponibles = [c for c in TEXTO_CAMBIOS if c in cambios_dedup.columns]
cambios_dedup['texto_mvp1'] = (
    cambios_dedup[txt_disponibles].fillna('').astype(str)
    .apply(lambda r: ' '.join(normalizar_nlp(v) for v in r if v.strip()), axis=1)
)
cambios_dedup['len_texto_mvp1'] = cambios_dedup['texto_mvp1'].str.len()

print(f'\nCampo combinado "texto_mvp1" (TF-IDF input):')
print(f'  Columnas incluidas: {txt_disponibles}')
print(f'  Cobertura (len > 10): {(cambios_dedup["len_texto_mvp1"] > 10).mean()*100:.1f}%')
print(f'  Longitud mediana: {cambios_dedup["len_texto_mvp1"].median():.0f} caracteres')

# Top palabras en plan_implementacion
if 'plan_implementacion' in cambios_dedup.columns:
    from sklearn.feature_extraction.text import TfidfVectorizer
    corpus = cambios_dedup['plan_implementacion'].fillna('').apply(normalizar_nlp)
    corpus = corpus[corpus.str.len() > 10]
    if len(corpus) > 10:
        tfidf = TfidfVectorizer(max_features=20, stop_words=None, min_df=5)
        try:
            tfidf.fit(corpus)
            weights = dict(zip(tfidf.get_feature_names_out(),
                               tfidf.idf_))
            print(f'\nTop palabras (IDF) en plan_implementacion:')
            for w, v in sorted(weights.items(), key=lambda x: x[1])[:15]:
                print(f'  {w:20s}  idf={v:.3f}')
        except Exception as e:
            print(f'TF-IDF: {e}')


---
## Sección 7B (v2) — NLP mejorado: stopwords, boilerplate y keywords de riesgo

Tres mejoras de bajo costo y alto retorno sobre el texto:

1. **Stopwords en español**: el TF-IDF de v1 usaba `stop_words=None`, así que palabras como *"de"*, *"para"*, *"realizar"* consumían parte de las 300 features disponibles sin aportar señal.
2. **Detección de boilerplate**: el EDA mostró que muchos cambios comparten texto idéntico (plantillas). Un texto repetido cientos de veces no discrimina; pero el **hecho de usar plantilla** sí puede ser señal (menor esfuerzo de documentación). Se crea `es_boilerplate` y `freq_texto`.
3. **Keywords de riesgo TI**: flags binarios interpretables (producción, base de datos, migración, firewall, etc.). A diferencia del TF-IDF, estas features son directamente explicables ante el CAB: *"este cambio toca base de datos en producción fuera de horario"*.


In [ ]:
# ============================================================
# SECCION 7B (v2): NLP MEJORADO
# ============================================================

# 1. Stopwords en espanol (lista compacta, sin dependencias externas;
#    en texto ya normalizado sin tildes)
STOPWORDS_ES = {
    'de', 'la', 'el', 'en', 'y', 'a', 'los', 'las', 'del', 'se', 'que', 'un',
    'una', 'con', 'por', 'para', 'al', 'lo', 'como', 'mas', 'o', 'pero', 'sus',
    'le', 'ya', 'este', 'esta', 'si', 'porque', 'muy', 'sin', 'sobre', 'tambien',
    'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo', 'nos', 'durante',
    'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso', 'ante', 'ellos',
    'e', 'esto', 'mi', 'antes', 'algunos', 'que', 'unos', 'yo', 'otro', 'otras',
    'otra', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual',
    'poco', 'ella', 'estar', 'estas', 'algunas', 'algo', 'nosotros',
    # verbos/terminos genericos de formularios de cambio detectados en el EDA
    'realizar', 'realiza', 'realizara', 'proceder', 'procede', 'favor',
    'requiere', 'debe', 'deben', 'ser', 'es', 'son', 'fue', 'sera', 'estan',
}

# 2. Boilerplate: frecuencia del texto combinado normalizado
if 'texto_mvp1' in cambios_dedup.columns:
    _freq = cambios_dedup['texto_mvp1'].value_counts()
    cambios_dedup['freq_texto'] = cambios_dedup['texto_mvp1'].map(_freq).fillna(1).astype(float)
    UMBRAL_BOILER = 5  # texto compartido por >= 5 cambios se considera plantilla
    cambios_dedup['es_boilerplate'] = (cambios_dedup['freq_texto'] >= UMBRAL_BOILER).astype(int)
    print(f'Boilerplate (texto repetido >= {UMBRAL_BOILER} veces): '
          f'{cambios_dedup["es_boilerplate"].mean()*100:.1f}% de los cambios')
    print(f'Texto mas repetido: {int(_freq.iloc[0]):,} cambios comparten el mismo texto')

# 3. Keywords de riesgo TI (sobre texto normalizado, sin tildes)
KEYWORDS_RIESGO = {
    'kw_produccion' : ['produccion', 'productivo', 'ambiente prod', 'prd'],
    'kw_bd'         : ['base de datos', 'sql', 'oracle', 'db2', 'tabla', 'indice', 'query'],
    'kw_red'        : ['firewall', 'vpn', 'dns', 'switch', 'router', 'balanceador', 'proxy'],
    'kw_migracion'  : ['migracion', 'migrar', 'conversion', 'traslado'],
    'kw_parche'     : ['parche', 'parchado', 'patch', 'hotfix', 'actualizacion', 'upgrade', 'version'],
    'kw_reinicio'   : ['reinicio', 'reiniciar', 'reboot', 'restart', 'apagado'],
    'kw_certificado': ['certificado', 'ssl', 'tls', 'renovacion'],
    'kw_batch'      : ['batch', 'job', 'proceso nocturno', 'cierre diario', 'cierre mensual'],
    'kw_core'       : ['core bancario', 'as400', 'mainframe', 'ibs', 'canales'],
}

if 'texto_mvp1' in cambios_dedup.columns:
    print('\nCobertura de keywords de riesgo:')
    for kw, terms in KEYWORDS_RIESGO.items():
        pat = '|'.join(re.escape(t) for t in terms)
        cambios_dedup[kw] = cambios_dedup['texto_mvp1'].fillna('').str.contains(pat, regex=True).astype(int)
        print(f'  {kw:16s}: {cambios_dedup[kw].mean()*100:5.1f}% de los cambios')
else:
    print('texto_mvp1 no disponible; ejecutar primero la Seccion 7')


---
## Sección 7C (v3) — Análisis del corpus: vocabulario, Zipf y n-gramas dominantes

Antes de decidir cuántas features de texto usar (300 TF-IDF, 50 SVD, etc.) hay que conocer el corpus:

1. **Tamaño y concentración del vocabulario**: si el top-100 de términos cubre el 80% de las ocurrencias, un `max_features` alto solo agrega ruido de cola.
2. **Ley de Zipf**: un corpus natural sigue aproximadamente una recta en log-log (frecuencia ~ 1/rango). Desviaciones fuertes delatan texto de plantilla (boilerplate) — la meseta inicial son los términos del formulario.
3. **N-gramas dominantes**: los bigramas frecuentes revelan las frases de plantilla que el TF-IDF con `sublinear_tf` amortigua pero no elimina.


In [ ]:
# ============================================================
# SECCION 7C (v3): ANALISIS DEL CORPUS
# ============================================================
from sklearn.feature_extraction.text import CountVectorizer

if 'texto_mvp1' in cambios_dedup.columns:
    corpus_all = cambios_dedup['texto_mvp1'].fillna('')
    corpus_all = corpus_all[corpus_all.str.len() > 10]

    cv_uni = CountVectorizer(ngram_range=(1, 1), min_df=2,
                             stop_words=sorted(STOPWORDS_ES))
    X_uni = cv_uni.fit_transform(corpus_all)
    frec = np.asarray(X_uni.sum(axis=0)).ravel()
    vocab = np.array(cv_uni.get_feature_names_out())
    orden = np.argsort(-frec)
    frec_ord = frec[orden]
    total_tok = frec.sum()

    print(f'Documentos con texto util : {len(corpus_all):,} de {len(cambios_dedup):,}')
    print(f'Vocabulario (sin stopwords, min_df=2): {len(vocab):,} terminos')
    print(f'Tokens totales            : {int(total_tok):,}')
    cob100 = frec_ord[:100].sum() / total_tok
    cob500 = frec_ord[:500].sum() / total_tok
    print(f'Cobertura del top-100     : {cob100:.1%} de las ocurrencias')
    print(f'Cobertura del top-500     : {cob500:.1%} de las ocurrencias')
    print(f'  -> Lectura: si el top-300 ya cubre >90%, max_features=300 en el')
    print(f'     TF-IDF es razonable; si no, subirlo o usar SVD (Seccion 17B).')

    # Bigramas dominantes (frases de plantilla)
    cv_bi = CountVectorizer(ngram_range=(2, 2), min_df=5,
                            stop_words=sorted(STOPWORDS_ES))
    try:
        X_bi = cv_bi.fit_transform(corpus_all)
        frec_bi = np.asarray(X_bi.sum(axis=0)).ravel()
        vocab_bi = np.array(cv_bi.get_feature_names_out())
        top_bi = np.argsort(-frec_bi)[:15]
    except ValueError:
        top_bi = []

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

    ax = axes[0]  # Zipf
    rangos = np.arange(1, len(frec_ord) + 1)
    ax.loglog(rangos, frec_ord, '-', color='#2e86de', lw=2)
    ax.set_xlabel('Rango del termino (log)')
    ax.set_ylabel('Frecuencia (log)')
    ax.set_title('Ley de Zipf del corpus\n(mesetas = texto de plantilla)', fontweight='bold')
    ax.grid(alpha=0.3, which='both')

    ax = axes[1]  # top unigramas
    top20 = orden[:20]
    ax.barh(range(20), frec[top20][::-1], color='#27ae60', alpha=0.85, edgecolor='white')
    ax.set_yticks(range(20))
    ax.set_yticklabels(vocab[top20][::-1], fontsize=8)
    ax.set_title('Top 20 unigramas (sin stopwords)', fontweight='bold')
    ax.set_xlabel('Frecuencia')

    ax = axes[2]  # top bigramas
    if len(top_bi):
        ax.barh(range(len(top_bi)), frec_bi[top_bi][::-1], color='#f39c12',
                alpha=0.85, edgecolor='white')
        ax.set_yticks(range(len(top_bi)))
        ax.set_yticklabels(vocab_bi[top_bi][::-1], fontsize=8)
        ax.set_title('Top bigramas (frases de plantilla)', fontweight='bold')
        ax.set_xlabel('Frecuencia')
    else:
        ax.text(0.5, 0.5, 'Sin bigramas con min_df=5', ha='center', va='center',
                transform=ax.transAxes)

    plt.suptitle('Analisis del corpus de cambios', fontweight='bold')
    plt.tight_layout(); plt.show()

    # Curva de cobertura acumulada
    cum = np.cumsum(frec_ord) / total_tok
    n90 = int(np.searchsorted(cum, 0.90) + 1)
    print(f'\nTerminos necesarios para cubrir el 90% del corpus: {n90:,}')
    print(f'Decision informada para max_features del TF-IDF (Seccion 15): '
          f'{"300 es suficiente" if n90 <= 300 else f"considerar subir a ~{min(n90, 1000)} o confiar en SVD"}')
else:
    print('texto_mvp1 no disponible; ejecutar primero la Seccion 7')


### ✔ Conclusiones Sección 7: campos de texto

* **El texto tiene información útil**: palabras como `backup`, `servidor`, `parchado` concentran TF-IDF alto y no son ruido
* **El boilerplate es un problema**: muchos cambios tienen texto idéntico (formularios rellenados por defecto). La longitud del texto es la primera señal barata y no ruidosa
* **`business_justification` es poderosa pero sospechosa**: alta correlación con el target, pero se debe confirmar que se llena ANTES del incidente. Se incluye como longitud, no como contenido crudo
* **Los textos de incidentes NO se usan como features**: `causa_raiz`, `accion_tomada`, etc. se conocen DESPUÉS del incidente — usarlos sería leakage perfecto
* **Estrategia NLP para MVP 1**: TF-IDF (max 300 features, ngram (1,2)) sobre el texto combinado de cambios. Embeddings y LLM tagging quedan para MVP 2

---
## Sección 8 — Conexión Incidentes ↔ Cambios *(núcleo del EDA)*

Este es el análisis más crítico de todo el EDA. **La etiqueta define el modelo** — si la construimos mal, todo lo demás falla.

### Fuentes de enlace disponibles

Existen dos campos estructurados que conectan ambas tablas:

| Campo | Tabla | Descripción | Cobertura |
|---|---|---|---|
| `causado_por_cambio` | Incidentes | ID del cambio que causó este incidente | ~17.6% |
| `incidente` | Cambios | ID del incidente que causó este cambio | ~1.1% |

### Por qué reconciliar ambas direcciones

Ninguna fuente está completa. Algunos links solo existen en incidentes, otros solo en cambios. La unión de ambos da la cobertura máximo de pares documentados (194 pares totales).

### Por qué filtrar por dirección temporal

De los 194 pares, **1 es correctivo** (el cambio ocurrió DESPUÉS del incidente — fue una respuesta, no una causa). Ese par se descarta.

In [ ]:
# ============================================================
# SECCION 8: CONEXION INCIDENTES <-> CAMBIOS
# ============================================================

def norm_id(s):
    """Normaliza IDs: strip + uppercase + reemplaza vacios por NA."""
    result = s.astype(str).str.strip().str.upper()
    result = result.replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, 'NAT': pd.NA})
    return result


inc = incidentes_dedup.copy()
cam = cambios_dedup.copy()
inc['ticket_n'] = norm_id(inc['ticket'])
cam['cambio_n'] = norm_id(cam['cambio'])

# Cobertura de links directos
cob_d1 = 0; cob_d2 = 0
if 'causado_por_cambio' in inc.columns:
    inc['cxc_n'] = norm_id(inc['causado_por_cambio'])
    cob_d1 = int(inc['cxc_n'].notna().sum())
if 'incidente' in cambios.columns:  # del raw, antes de deduplicar
    cam_raw_inc = cambios[['cambio','incidente']].copy()
    cam_raw_inc['cambio_n'] = norm_id(cam_raw_inc['cambio'])
    cam_raw_inc['inc_n']    = norm_id(cam_raw_inc['incidente'])
    cam_raw_inc = cam_raw_inc.dropna(subset=['inc_n']).drop_duplicates()
    cob_d2 = cam_raw_inc['cambio_n'].nunique()

print(f'Incidentes con causado_por_cambio informado: {cob_d1} de {len(inc)}  ({100*cob_d1/len(inc):.1f}%)')
print(f'Cambios   con incidente informado          : {cob_d2} de {len(cam)} ({100*cob_d2/len(cam):.2f}%)')

# Pares unicos (union de ambas direcciones)
if 'incidente' in cambios.columns:
    # v2 FIX: en v1 se hacia zip(inc['cxc_n'].dropna(), inc['ticket_n']),
    # que desalinea las filas (el i-esimo cxc no nulo se emparejaba con el
    # i-esimo ticket GLOBAL). Ademas se inyectaban pares invertidos (i,c).
    if 'cxc_n' in inc.columns:
        _d1 = inc[['cxc_n', 'ticket_n']].dropna(subset=['cxc_n'])
        d1 = set(zip(_d1['cxc_n'], _d1['ticket_n']))
    else:
        d1 = set()
    d2 = set(zip(cam_raw_inc['cambio_n'], cam_raw_inc['inc_n']))
    pares_unicos = pd.DataFrame(sorted(d1 | d2),
                                columns=['cambio_n', 'inc_n']).drop_duplicates()
else:
    pares_unicos = pd.DataFrame(columns=['cambio_n','inc_n'])

print(f'Pares cambio-incidente unicos: {len(pares_unicos)}')

# Validacion temporal: cambio ANTES del incidente
fecha_cam_col = next((c for c in ['fecha_inicio_programado','fecha_apertura'] if c in cam.columns), None)
fecha_inc_col = next((c for c in ['open_date','start_outage'] if c in inc.columns), None)

if fecha_cam_col and fecha_inc_col and len(pares_unicos):
    px = (pares_unicos
          .merge(cam[['cambio_n', fecha_cam_col]], on='cambio_n', how='left')
          .merge(inc[['ticket_n', fecha_inc_col]].rename(columns={'ticket_n':'inc_n'}),
                 on='inc_n', how='left'))
    px['dias_cambio_a_incidente'] = (
        (px[fecha_inc_col] - px[fecha_cam_col]).dt.total_seconds() / 86400
    )
    causantes = int((px['dias_cambio_a_incidente'].fillna(0) >= 0).sum())
    correctivos = int((px['dias_cambio_a_incidente'].fillna(0) < 0).sum())
    print(f'\nValidacion temporal ({fecha_cam_col} vs {fecha_inc_col}):')
    print(f'  Cambio ANTES del incidente (causante plausible): {causantes}')
    print(f'  Cambio DESPUES (correctivo, NO cuenta)         : {correctivos}')
    med = px['dias_cambio_a_incidente'].dropna().median()
    print(f'  Mediana dias cambio -> incidente: {med:.2f} ({med*24:.1f} horas)')

    # Histograma de tiempos
    dias_valid = px['dias_cambio_a_incidente'].dropna()
    dias_valid = dias_valid[dias_valid >= 0]
    if len(dias_valid):
        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.hist(dias_valid, bins=20, color='#2e86de', alpha=0.8, edgecolor='white')
        ax.axvline(dias_valid.median(), color='#e74c3c', ls='--', lw=2,
                   label=f'Mediana: {dias_valid.median():.1f} dias')
        ax.set_title('Distribucion de tiempo cambio -> incidente (pares causales)', fontweight='bold')
        ax.set_xlabel('Dias'); ax.set_ylabel('n pares'); ax.legend()
        plt.tight_layout(); plt.show()
else:
    print('Validacion temporal no disponible (columnas de fecha no encontradas)')
    causantes = len(pares_unicos)
    px = pares_unicos.copy()
    px['dias_cambio_a_incidente'] = 0.0


---
## Sección 9 — Definición y selección de la variable objetivo

### Las 3 definiciones evaluadas

#### D1: Link directo (ELEGIDA)
- Usa los campos `cambios.incidente` + `incidentes.causado_por_cambio`
- Filtra por dirección temporal: cambio ocurre ANTES del incidente
- **Ventaja**: etiqueta explícita, documentada, sin ambigüedad
- **Desventaja**: baja cobertura si los campos no se llenan bien

#### D2: CI + ventana 7 días
- Cualquier cambio en el mismo CI que tiene un incidente en los 7 días siguientes
- **Problema grave**: CIs muy activos generan falsos positivos masivos
- Infla el target 3-5x respecto a D1 sin evidencia de causalidad

#### D3: CI + ventana 14 días
- Extensión de D2, aún más ruido

### Por qué elegimos D1

D1 es **la única definición que respeta causalidad documentada**. Las demás introducen correlación espuria. La baja prevalencia (~1%) es la prevalencia **real** del fenómeno, no un artefacto.

In [ ]:
# ============================================================
# SECCION 9: VARIABLE OBJETIVO
# ============================================================

# ============================================================
# SECCION 9: COMPARACION DE DEFINICIONES DEL TARGET
# Evaluamos D1, D2, D3, D4 empiricamente antes de elegir
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Asegurar que ticket_n existe en incidentes_dedup (puede no existir si la Sec 8
# solo la agrego a la copia 'inc')
if 'ticket_n' not in incidentes_dedup.columns:
    incidentes_dedup['ticket_n'] = norm_id(incidentes_dedup['ticket'])

# Asegurar cambio_n en cambios_dedup
if 'cambio_n' not in cambios_dedup.columns:
    cambios_dedup['cambio_n'] = norm_id(cambios_dedup['cambio'])

N_TOT = len(cambios_dedup)

# -----------------------------------------------------------------
# D1: LINK DIRECTO BIDIRECCIONAL + FILTRO TEMPORAL
# -----------------------------------------------------------------
if 'px' not in dir() or not len(px):
    print('AVISO: ejecuta primero la Seccion 8 para tener los pares px')
    px = pd.DataFrame(columns=['cambio_n','inc_n','dias_cambio_a_incidente'])

pares_causales_d1 = px[px['dias_cambio_a_incidente'].fillna(0) >= 0].copy()
cambios_causantes_d1 = set(pares_causales_d1['cambio_n'])
cambios_dedup['causo_incidente'] = cambios_dedup['cambio_n'].isin(cambios_causantes_d1).astype(int)

N_D1 = int(cambios_dedup['causo_incidente'].sum())
PREV_D1 = N_D1 / N_TOT
print(f'D1 (link directo + filtro temporal):')
print(f'  Positivos: {N_D1:,} / {N_TOT:,}  ({100*PREV_D1:.3f}%)  desbalance 1:{int((N_TOT-N_D1)/max(N_D1,1))}')

# -----------------------------------------------------------------
# D2 / D3 / D4: CI + VENTANA TEMPORAL
# Logica: si un cambio en el CI X ocurre N dias antes de un incidente
# en el mismo CI, lo etiquetamos como positivo.
# PROBLEMA esperado: los CIs muy activos acumulan muchos pares espurios.
# -----------------------------------------------------------------
CI_COL     = next((c for c in ['configuration_item','ci'] if c in cambios_dedup.columns), None)
CI_COL_INC = next((c for c in ['configuration_item','ci','affected_resource_list']
                   if c in incidentes_dedup.columns), None)

ventana_ids = {}  # ventana_dias -> set de cambio_n positivos

if CI_COL and CI_COL_INC and fecha_cam_col and fecha_inc_col:
    # Preparar CI primario para ambas tablas
    cam_v = cambios_dedup[['cambio_n', fecha_cam_col, CI_COL]].copy()
    cam_v['ci_p'] = cam_v[CI_COL].astype(str).str.split(r'\s*\|\s*').str[0].str.strip().str.lower()
    cam_v[fecha_cam_col] = pd.to_datetime(cam_v[fecha_cam_col], errors='coerce')
    cam_v = cam_v.dropna(subset=[fecha_cam_col])

    inc_v = incidentes_dedup[['ticket_n', fecha_inc_col, CI_COL_INC]].copy()
    inc_v['ci_p'] = inc_v[CI_COL_INC].astype(str).str.split(r'\s*\|\s*').str[0].str.strip().str.lower()
    inc_v[fecha_inc_col] = pd.to_datetime(inc_v[fecha_inc_col], errors='coerce')
    inc_v = inc_v.dropna(subset=[fecha_inc_col])

    # Cross-join por CI primario
    pares_ci = cam_v.merge(inc_v, on='ci_p', suffixes=('_c','_i'))
    pares_ci['delta_dias'] = (
        (pares_ci[fecha_inc_col] - pares_ci[fecha_cam_col]).dt.total_seconds() / 86400
    )
    # Solo pares causales: cambio ANTES del incidente
    pares_ci = pares_ci[(pares_ci['delta_dias'] >= 0)]

    for v in [7, 14, 30]:
        ids_v = set(pares_ci[pares_ci['delta_dias'] <= v]['cambio_n'])
        ventana_ids[v] = ids_v
        cambios_dedup[f'target_ci_{v}d'] = cambios_dedup['cambio_n'].isin(ids_v).astype(int)
        n_v = len(ids_v)
        overlap = len(ids_v & cambios_causantes_d1)
        precision_v = overlap / max(n_v, 1) * 100
        ruido_pct  = (1 - overlap/max(n_v,1)) * 100
        print(f'D{[7,14,30].index(v)+2} (CI + {v:2d} dias): '
              f'{n_v:5,} positivos ({100*n_v/N_TOT:.2f}%)  '
              f'vs D1: {n_v/max(N_D1,1):.1f}x  '
              f'overlap con D1: {overlap}/{n_v} ({precision_v:.1f}% reales)  '
              f'ruido estimado: {ruido_pct:.1f}%')
else:
    print('CI column no encontrada en ambas tablas - D2/D3/D4 no disponibles')
    for v in [7, 14, 30]:
        ventana_ids[v] = set()
        cambios_dedup[f'target_ci_{v}d'] = 0

print()

# -----------------------------------------------------------------
# ANALISIS DE SESGO POR CI ACTIVO
# Si un CI tiene muchos cambios, tendra muchos pares CI+ventana aunque
# ningun cambio suyo haya causado realmente ningun incidente.
# -----------------------------------------------------------------
print('=== SESGO POR VOLUMEN DE CI ===')
if CI_COL in cambios_dedup.columns:
    n_cambios_ci = cambios_dedup.groupby(
        cambios_dedup[CI_COL].astype(str).str.split(r'\s*\|\s*').str[0].str.strip()
    ).size().rename('n_cambios')

    for col_tgt, label in [('causo_incidente','D1'), ('target_ci_7d','D2')]:
        if col_tgt not in cambios_dedup.columns: continue
        tmp = cambios_dedup.copy()
        tmp['ci_p'] = tmp[CI_COL].astype(str).str.split(r'\s*\|\s*').str[0].str.strip()
        tasa_ci = tmp.groupby('ci_p')[col_tgt].mean()
        corr_v = tasa_ci.rename('tasa').to_frame().join(
            n_cambios_ci.rename('n_cambios')
        ).corr().loc['tasa','n_cambios']
        print(f'  {label}: correlacion tasa_incidente ~ volumen_CI = {corr_v:+.4f}'
              f'  ({"SESGO DETECTADO" if abs(corr_v) > 0.15 else "OK"} )')

print()

# -----------------------------------------------------------------
# COMPARACION DE CALIDAD DE ETIQUETA: MODELO NAIVE CON CADA TARGET
# Un buen target produce PR-AUC mas alto incluso con un modelo simple
# Usamos solo 5 features basicas para que el resultado sea comparable
# -----------------------------------------------------------------
print('=== CALIDAD DE ETIQUETA: PR-AUC CON MODELO NAIVE (LR) ===')
print('Feature set identico en todos los targets -> diferencia = calidad de la etiqueta\n')

# v2 FIX: en v1 estas features solo existian si se habia ejecutado antes la
# Seccion 11 (que va DESPUES en el notebook), por lo que la comparacion de
# targets solia correr sin features. Ahora se construyen versiones minimas aqui.
if 'len_descripcion' not in cambios_dedup.columns and 'descripcion' in cambios_dedup.columns:
    cambios_dedup['len_descripcion'] = cambios_dedup['descripcion'].fillna('').astype(str).str.len()
if 'hora_inicio' not in cambios_dedup.columns and 'fecha_inicio_programado' in cambios_dedup.columns:
    _fi9 = pd.to_datetime(cambios_dedup['fecha_inicio_programado'], errors='coerce')
    cambios_dedup['hora_inicio'] = _fi9.dt.hour
    cambios_dedup['dow_inicio']  = _fi9.dt.dayofweek
if ('dur_prog_horas' not in cambios_dedup.columns
        and {'fecha_inicio_programado', 'fecha_fin_programado'}.issubset(cambios_dedup.columns)):
    _fi9 = pd.to_datetime(cambios_dedup['fecha_inicio_programado'], errors='coerce')
    _ff9 = pd.to_datetime(cambios_dedup['fecha_fin_programado'], errors='coerce')
    cambios_dedup['dur_prog_horas'] = (_ff9 - _fi9).dt.total_seconds().div(3600).clip(lower=0)
if 'n_recalendarizaciones' not in cambios_dedup.columns and 'cantidadrecalendarizacion' in cambios_dedup.columns:
    cambios_dedup['n_recalendarizaciones'] = pd.to_numeric(
        cambios_dedup['cantidadrecalendarizacion'], errors='coerce').fillna(0)

# 5+ features simples disponibles en este punto
feats_naive = []
for f in ['riesgo','tipo_de_cambio','hora_inicio','dow_inicio','dur_prog_horas',
          'len_descripcion','n_recalendarizaciones']:
    if f in cambios_dedup.columns:
        feats_naive.append(f)

if feats_naive:
    X_n = cambios_dedup[feats_naive].copy()
    # Codificar categoricas
    for c in X_n.select_dtypes(include='object').columns:
        le = LabelEncoder()
        X_n[c] = le.fit_transform(X_n[c].astype(str))
    X_n = X_n.fillna(0).astype(float)

    # Split temporal 80/20
    if fecha_cam_col in cambios_dedup.columns:
        orden_idx = cambios_dedup[fecha_cam_col].fillna(pd.Timestamp('2100')).argsort()
        cambios_ord = cambios_dedup.iloc[orden_idx].reset_index(drop=True)
        X_ord = X_n.iloc[orden_idx].reset_index(drop=True)
    else:
        cambios_ord = cambios_dedup.reset_index(drop=True)
        X_ord = X_n

    cut = int(len(cambios_ord) * 0.80)
    Xtr, Xte = X_ord.iloc[:cut], X_ord.iloc[cut:]

    resultados_target = []
    for col_tgt, label in [('causo_incidente','D1: Link directo (ELEGIDA)'),
                            ('target_ci_7d',   'D2: CI + 7 dias'),
                            ('target_ci_14d',  'D3: CI + 14 dias'),
                            ('target_ci_30d',  'D4: CI + 30 dias')]:
        if col_tgt not in cambios_ord.columns: continue
        ytr = cambios_ord[col_tgt].iloc[:cut].values
        yte = cambios_ord[col_tgt].iloc[cut:].values
        if ytr.sum() < 3 or yte.sum() < 1:
            print(f'  {label}: insuficientes positivos en split')
            continue
        spw = int((len(ytr)-ytr.sum()) / max(ytr.sum(),1))
        try:
            m = LogisticRegression(C=0.1, max_iter=300,
                                   class_weight='balanced', random_state=42)
            m.fit(Xtr, ytr)
            proba = m.predict_proba(Xte)[:,1]
            pr  = average_precision_score(yte, proba)
            roc = roc_auc_score(yte, proba)
            n_pos_te = int(yte.sum())
            resultados_target.append(dict(definicion=label, PR_AUC=round(pr,4),
                                         ROC_AUC=round(roc,4), pos_test=n_pos_te,
                                         n_total_pos=int(cambios_ord[col_tgt].sum())))
            print(f'  {label:40s}  PR-AUC={pr:.4f}  ROC-AUC={roc:.4f}  '
                  f'(pos test={n_pos_te}, total pos={int(cambios_ord[col_tgt].sum()):,})')
        except Exception as e:
            print(f'  {label}: error -> {e}')

    df_res = pd.DataFrame(resultados_target) if resultados_target else pd.DataFrame()
else:
    print('  Features no disponibles aun (ejecutar Sec 11 primero)')
    df_res = pd.DataFrame()

print()

# -----------------------------------------------------------------
# VISUALIZACION COMPARATIVA DE LAS 4 DEFINICIONES
# -----------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: N positivos por definicion
ax = axes[0]
all_defs  = {'D1 link directo': N_D1}
all_defs.update({f'D{i+2} CI+{v}d': len(ventana_ids[v])
                 for i, v in enumerate([7,14,30]) if ventana_ids.get(v)})
colores_d = ['#27ae60'] + ['#e74c3c']*len(ventana_ids)
bars = ax.bar(list(all_defs.keys()), list(all_defs.values()),
              color=colores_d[:len(all_defs)], alpha=0.85, edgecolor='white')
for bar_, v in zip(bars, all_defs.values()):
    ax.text(bar_.get_x()+bar_.get_width()/2, bar_.get_height()+0.5,
            f'{v:,}\n({100*v/N_TOT:.2f}%)', ha='center', fontsize=8.5, fontweight='bold')
ax.set_title('N positivos por definicion', fontweight='bold')
ax.set_ylabel('N positivos')
ax.set_xticks(range(len(all_defs)))
ax.set_xticklabels(list(all_defs.keys()), rotation=15, ha='right', fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Panel 2: Precision de la etiqueta (overlap con D1)
ax = axes[1]
overlap_data = {}
for i, v in enumerate([7,14,30]):
    if ventana_ids.get(v):
        ov = len(ventana_ids[v] & cambios_causantes_d1)
        total_v = len(ventana_ids[v])
        overlap_data[f'D{i+2} CI+{v}d'] = (ov/max(total_v,1)*100,
                                             (total_v-ov)/max(total_v,1)*100)
if overlap_data:
    labels_ov = list(overlap_data.keys())
    reales_pct = [v[0] for v in overlap_data.values()]
    ruido_pct_v = [v[1] for v in overlap_data.values()]
    x_ov = range(len(labels_ov))
    ax.bar(x_ov, reales_pct, color='#27ae60', alpha=0.8, label='Real (en D1)')
    ax.bar(x_ov, ruido_pct_v, bottom=reales_pct, color='#e74c3c',
           alpha=0.6, label='Ruido (no esta en D1)')
    ax.set_xticks(x_ov)
    ax.set_xticklabels(labels_ov, rotation=15, ha='right', fontsize=8)
    ax.axhline(100, color='gray', ls='--', lw=1)
    ax.set_title('Composicion de positivos en D2/D3/D4', fontweight='bold')
    ax.set_ylabel('% del total de positivos')
    ax.legend(fontsize=8)
    for xi, (r, ru) in zip(x_ov, [(r,n) for r,n in zip(reales_pct, ruido_pct_v)]):
        ax.text(xi, 5, f'{r:.0f}%\nreales', ha='center', fontsize=8, color='white', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'D2/D3/D4 no calculadas\n(CI col no encontrada)',
            ha='center', va='center', transform=ax.transAxes, fontsize=10)
ax.set_title('Precision de la etiqueta\n(% realmente causales)', fontweight='bold')

# Panel 3: PR-AUC comparativo
ax = axes[2]
if len(df_res):
    colors_pr = ['#27ae60' if 'D1' in r else '#e74c3c' for r in df_res['definicion']]
    bars2 = ax.bar(range(len(df_res)), df_res['PR_AUC'],
                   color=colors_pr, alpha=0.85, edgecolor='white')
    ax.set_xticks(range(len(df_res)))
    ax.set_xticklabels([r.split(':')[0] for r in df_res['definicion']],
                       rotation=15, ha='right', fontsize=9)
    for bar_, v in zip(bars2, df_res['PR_AUC']):
        ax.text(bar_.get_x()+bar_.get_width()/2, bar_.get_height()+0.001,
                f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_ylabel('PR-AUC (test)')
    ax.set_title('Calidad de etiqueta: PR-AUC\nmodelo naive identico en todos', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Ejecutar Sec 11 primero\npara features',
            ha='center', va='center', transform=ax.transAxes, fontsize=10)

plt.suptitle('Comparacion de definiciones del target: D1 vs D2 vs D3 vs D4', fontweight='bold')
plt.tight_layout()
plt.show()

# -----------------------------------------------------------------
# DISTRIBUCION TEMPORAL DEL TARGET D1
# -----------------------------------------------------------------
if fecha_cam_col in cambios_dedup.columns:
    cambios_dedup['_mes'] = (
        pd.to_datetime(cambios_dedup[fecha_cam_col], errors='coerce')
        .dt.to_period('M').dt.to_timestamp()
    )
    ts = cambios_dedup.groupby('_mes')['causo_incidente'].agg(['sum','count'])
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.bar(ts.index, ts['count'], color='#3498db', alpha=0.35, width=20, label='Total cambios')
    ax2 = ax.twinx()
    ax2.plot(ts.index, ts['sum'], 'ro-', lw=2, ms=6, label='Causaron incidente (D1)')
    ax.set_ylabel('Total cambios / mes', color='#3498db')
    ax2.set_ylabel('Cambios causantes de incidente', color='#e74c3c')
    ax.set_title('Evolucion temporal — target D1 sobre el volumen de cambios', fontweight='bold')
    plt.setp(ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)
    lines1, lab1 = ax.get_legend_handles_labels()
    lines2, lab2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, lab1+lab2, fontsize=8)
    plt.tight_layout(); plt.show()
    cambios_dedup.drop(columns=['_mes'], inplace=True)

# -----------------------------------------------------------------
# VEREDICTO
# -----------------------------------------------------------------
print('=' * 65)
print('VEREDICTO DE LA COMPARACION')
print('=' * 65)
print(f'  D1: {N_D1:,} positivos ({100*N_D1/N_TOT:.3f}%) — 100% causales (link documentado)')
for i, v in enumerate([7,14,30]):
    nv = len(ventana_ids.get(v, set()))
    ov = len(ventana_ids.get(v, set()) & cambios_causantes_d1)
    print(f'  D{i+2}: {nv:,} positivos ({100*nv/N_TOT:.3f}%) — '
          f'{ov}/{nv} reales ({100*ov/max(nv,1):.1f}% precision), '
          f'{nv/max(N_D1,1):.1f}x infla vs D1')
print()
print('CONCLUSION: D1 es la unica definicion con causalidad documentada.')
print('  - D2/D3/D4 introducen mayormente ruido (coincidencia temporal != causa)')
print('  - Un modelo entrenado con D2-D4 aprenderia a discriminar CIs activos,')
print('    no cambios riesgosos especificamente.')
print(f'\nscale_pos_weight para XGBoost con D1: {int((N_TOT-N_D1)/max(N_D1,1))}')


### ✔ Conclusiones Sección 9: variable objetivo

* **La etiqueta `causo_incidente` es explícita y documentada**: viene de campos que el equipo de operaciones filló al registrar el incidente o el cambio, no de una inferencia
* **La prevalencia real del fenómeno es ~1%**: no es un problema de construcción, es la realidad operativa. La mayoría de cambios son rutinarios y no causan incidentes
* **D2 (CI+ventana) fue descartado**: infla el target 3-5x con coincidencias temporales sin causalidad documentada. El modelo aprendería correlaciones falsas
* **La mediana de tiempo cambio→incidente es ≈0.45 días (10.8 horas)**: la señal es temporal y cercana, lo que valida la hipótesis del proyecto
* **Un par era correctivo** (cambio después del incidente): se descartó. Este caso refuerza la importancia del filtro temporal
* **Proxy para ampliación futura**: hay 86 CIs en común entre incidentes y cambios que podrían usarse para recuperar enlaces perd

---
## Sección 9B (v3) — Robustez del target: estabilidad, completitud y asociación

Elegido D1, hay que estresarlo antes de confiar en él:

1. **Estabilidad temporal de la prevalencia** (con IC de Wilson): si la tasa de positivos cambia bruscamente entre trimestres, el split temporal heredará bloques con targets "distintos" y las métricas oscilarán. También alimenta la regla del 3% de la Sección 15C.
2. **Completitud del label por mes**: el target D1 depende de que Operaciones documente `causado_por_cambio`. Si la disciplina de llenado cambió en el tiempo (p.ej. mejoró tras una auditoría), la prevalencia aparente cambia **sin que cambie el riesgo real** — eso es deriva del label, no del fenómeno.
3. **Tiempo cambio→incidente por segmento**: valida que la ventana causal (~horas) sea consistente entre niveles de riesgo; un segmento con lags de semanas huele a enlaces mal documentados.
4. **Tests χ² de asociación**: formaliza lo que el lift sugiere — ¿la asociación entre `riesgo`/`tipo_de_cambio` y el target es estadísticamente distinguible del azar dado el tamaño de muestra?


In [ ]:
# ============================================================
# SECCION 9B (v3): ROBUSTEZ DEL TARGET
# ============================================================
from scipy.stats import chi2_contingency

def wilson_ci(k, n, z=1.96):
    # Intervalo de Wilson para una proporcion (robusto con k pequeno)
    if n == 0:
        return 0.0, 0.0
    p = k / n
    den = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / den
    ancho = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return max(0.0, centro - ancho), min(1.0, centro + ancho)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# --- 1. Prevalencia por trimestre con IC Wilson ---
ax = axes[0]
if fecha_cam_col in cambios_dedup.columns:
    _f = pd.to_datetime(cambios_dedup[fecha_cam_col], errors='coerce')
    _q = _f.dt.to_period('Q')
    tab_q = cambios_dedup.groupby(_q)['causo_incidente'].agg(['sum', 'count'])
    tab_q = tab_q[tab_q['count'] >= 30]
    xs = range(len(tab_q))
    prevs = tab_q['sum'] / tab_q['count']
    los, his = zip(*[wilson_ci(int(k), int(n)) for k, n in
                     zip(tab_q['sum'], tab_q['count'])])
    ax.errorbar(xs, prevs * 100,
                yerr=[(prevs - np.array(los)) * 100, (np.array(his) - prevs) * 100],
                fmt='o-', color='#e74c3c', lw=2, ms=6, capsize=4)
    ax.axhline(cambios_dedup['causo_incidente'].mean() * 100, color='gray',
               ls='--', lw=1.5, label='Prevalencia global')
    ax.set_xticks(list(xs))
    ax.set_xticklabels([str(p) for p in tab_q.index], rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Prevalencia (%)')
    ax.set_title('Prevalencia del target por trimestre\n(IC 95% Wilson)', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    print('Estabilidad temporal del target:')
    print(tab_q.assign(prev_pct=(prevs * 100).round(3)).to_string())
else:
    ax.text(0.5, 0.5, 'Sin fecha de cambio', ha='center', va='center', transform=ax.transAxes)

# --- 2. Completitud del label por mes (deriva de documentacion) ---
ax = axes[1]
if {'open_date', 'causado_por_cambio'}.issubset(incidentes_dedup.columns):
    _fi = pd.to_datetime(incidentes_dedup['open_date'], errors='coerce')
    _m = _fi.dt.to_period('M')
    _lleno = incidentes_dedup['causado_por_cambio'].notna() & (
        incidentes_dedup['causado_por_cambio'].astype(str).str.strip() != '')
    tab_m = pd.DataFrame({'m': _m, 'lleno': _lleno}).dropna(subset=['m'])
    tab_m = tab_m.groupby('m')['lleno'].agg(['mean', 'count'])
    tab_m = tab_m[tab_m['count'] >= 5]
    ax.plot(range(len(tab_m)), tab_m['mean'] * 100, 'o-', color='#8e44ad', lw=2, ms=5)
    ax.set_xticks(range(len(tab_m)))
    ax.set_xticklabels([str(p) for p in tab_m.index], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('% incidentes con causado_por_cambio')
    ax.set_title('Completitud del label por mes\n(tendencia = deriva de documentacion)',
                 fontweight='bold')
    ax.grid(alpha=0.3)
    _corr_t = np.corrcoef(range(len(tab_m)), tab_m['mean'])[0, 1] if len(tab_m) > 2 else 0
    print(f'\nCompletitud del label: media={tab_m["mean"].mean():.1%}, '
          f'tendencia temporal r={_corr_t:+.2f}')
    if abs(_corr_t) > 0.5:
        print('  ALERTA: la disciplina de llenado cambia en el tiempo -> la prevalencia')
        print('  aparente del target deriva por DOCUMENTACION, no por riesgo real.')
else:
    ax.text(0.5, 0.5, 'Columnas no disponibles', ha='center', va='center', transform=ax.transAxes)

# --- 3. Tiempo cambio->incidente por nivel de riesgo ---
ax = axes[2]
if 'px' in dir() and len(px) and 'riesgo' in cambios_dedup.columns:
    _pxr = px.merge(cambios_dedup[['cambio_n', 'riesgo']], on='cambio_n', how='left')
    _pxr = _pxr[_pxr['dias_cambio_a_incidente'].fillna(-1) >= 0]
    grupos_r, labels_r = [], []
    for rz, g in _pxr.groupby('riesgo'):
        if len(g) >= 3:
            grupos_r.append(g['dias_cambio_a_incidente'].clip(upper=30).values)
            labels_r.append(f'{rz} (n={len(g)})')
    if grupos_r:
        ax.boxplot(grupos_r, tick_labels=labels_r, patch_artist=True,
                   boxprops=dict(facecolor='#3498db', alpha=0.6),
                   medianprops=dict(color='white', lw=2))
        ax.set_ylabel('Dias cambio -> incidente (cap 30)')
        ax.set_title('Lag causal por nivel de riesgo\n(lags de semanas = enlaces dudosos)',
                     fontweight='bold')
        plt.setp(ax.get_xticklabels(), rotation=15, ha='right', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'Muy pocos pares por nivel', ha='center', va='center',
                transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, 'px o riesgo no disponibles', ha='center', va='center',
            transform=ax.transAxes)

plt.suptitle('Robustez del target D1', fontweight='bold')
plt.tight_layout(); plt.show()

# --- 4. Tests chi-cuadrado de asociacion ---
print('\nTESTS DE ASOCIACION (chi-cuadrado de independencia):')
print('p < 0.05 -> la asociacion con el target no se explica por azar\n')
for col in ['riesgo', 'tipo_de_cambio', 'categoria']:
    if col not in cambios_dedup.columns:
        continue
    tabla = pd.crosstab(cambios_dedup[col].astype(str), cambios_dedup['causo_incidente'])
    if tabla.shape[0] < 2 or tabla.shape[1] < 2:
        continue
    chi2_v, p_v, dof, esperado = chi2_contingency(tabla)
    aviso = ''
    if (esperado < 5).any():
        aviso = ' [AVISO: celdas con esperado<5, interpretar con cautela]'
    # Cramer's V como tamano del efecto
    n_obs = tabla.values.sum()
    cramer = np.sqrt(chi2_v / (n_obs * (min(tabla.shape) - 1)))
    print(f'  {col:18s}  chi2={chi2_v:8.2f}  p={p_v:.2e}  V de Cramer={cramer:.3f}{aviso}')
print('\nLectura: la V de Cramer es el tamano del efecto (0.1 debil, 0.3 medio,')
print('0.5 fuerte). Un p-value minusculo con V bajo = asociacion real pero tenue,')
print('tipico con n grande; no confundir significancia con poder predictivo.')


---
## Sección 10 — Leakage: detección, validación y reglas

El riesgo más grave de este tipo de modelo es entrenar con información que solo existe **después del evento que se quiere predecir**.

### El problema en concreto

Muchas columnas en la tabla de cambios se llenan DESPUÉS de implementar el cambio o de conocer el resultado:

| Variable | Cuándo se llena | Por qué es leakage |
|---|---|---|
| `reversado` | Post-implementación | Solo existe si el cambio se revierte (= ya sabemos que falló) |
| `cantidaddevolucion` | Post-implementación | Cuenta las reversiones ocurridas |
| `codigocierre` | Al cerrar el cambio | Se llena con el código que indica el resultado |
| `fecha_cierre` | Al cerrar | Solo existe cuando el cambio terminó |
| `estado` (cerrado) | Post-implementación | El estado final no está disponible antes del evento |
| `pir` | Post-incidente | Post-Implementation Review: existe SOLO si hubo incidente |
| `causa_raiz` | Post-incidente | Se llena al investigar el incidente |
| `ticket_externo` | Post-incidente | Referencia al ticket del incidente |
| `estado_padre` | Variable | Depende del ciclo de vida |

### Tres niveles de evidencia del leakage

1. **Lógica de negocio**: el timing es incorrecto (sabemos cuándo se llena)
2. **Lift anormalmente alto**: si la variable tiene lift >3x es porque codifica el resultado
3. **Modelo con vs sin leakage**: el gap de ROC-AUC confirma la trampa

In [ ]:
# ============================================================
# SECCION 10: LEAKAGE — TIMELINE DE DISPONIBILIDAD
# ============================================================
import matplotlib.patches as mpatches

# Diagrama de disponibilidad temporal por variable
# Etapa 0 = Apertura del cambio (solo informacion de apertura)
# Etapa 1 = Aprobacion CAB / antes de implementar
# Etapa 2 = Post-implementacion
# Etapa 3 = Post-cierre / post-incidente

ETAPAS = ['0\nApertura', '1\nAprobacion\nCAB', '2\nPost-\nimplementacion', '3\nPost-cierre\n/ Incidente']

LINEA_TEMPORAL = [
    # (variable, etapa_disponible, rol, descripcion)
    ('tipo_de_cambio',          0, 'predictor', 'Disponible al crear el cambio'),
    ('riesgo',                  0, 'predictor', 'Clasificado en la apertura'),
    ('descripcion',             0, 'predictor', 'Campo de texto inicial'),
    ('plan_implementacion',     0, 'predictor', 'Plan escrito antes del CAB'),
    ('business_justification',  0, 'predictor', 'Justificacion de negocio (monitorear)'),
    ('configuration_item',      0, 'predictor', 'CI afectado (conocido al abrir)'),
    ('fecha_inicio_programado', 0, 'predictor', 'Programado en apertura'),
    ('aprobado_cab',            1, 'revisar',   'Se actualiza en aprobacion CAB'),
    ('estado',                  2, 'fuga',      'Estado FINAL post-implementacion'),
    ('cantidaddevolucion',      2, 'fuga',      'Cantidad de reversiones ocurridas'),
    ('reversado',               2, 'fuga',      'Flag de reverso post-implementacion'),
    ('codigocierre',            2, 'fuga',      'Codigo de cierre del cambio'),
    ('fecha_cierre',            2, 'fuga',      'Fecha de cierre del cambio'),
    ('pir',                     3, 'fuga',      'Post-Implementation Review (post-incidente)'),
    ('causa_raiz',              3, 'fuga',      'Se llena al investigar el incidente'),
    ('ticket_externo',          3, 'fuga',      'Referencia al ticket del incidente'),
    ('estado_padre',            3, 'fuga',      'Estado del cambio padre (post)'),
]

col_colores = {'predictor': '#27ae60', 'revisar': '#f39c12', 'fuga': '#e74c3c', 'etiqueta': '#8e44ad'}

fig, ax = plt.subplots(figsize=(14, len(LINEA_TEMPORAL)*0.55 + 1.5))

# Linea de tiempo base
for i, etapa in enumerate(ETAPAS):
    ax.axvline(i, color='lightgray', lw=1, zorder=0)
    ax.text(i, len(LINEA_TEMPORAL) + 0.3, etapa, ha='center', fontsize=8, fontweight='bold')

for y, (var, etapa, rol, desc) in enumerate(LINEA_TEMPORAL):
    color = col_colores[rol]
    # Barra desde 0 hasta etapa
    ax.barh(y, etapa + 0.8, left=0, height=0.6, color=color, alpha=0.25, edgecolor='none')
    # Punto de disponibilidad
    ax.plot(etapa, y, 'o', color=color, ms=10, zorder=5)
    ax.text(etapa + 0.1, y, f'{var}', va='center', fontsize=8.5, fontweight='bold', color=color)
    ax.text(3.7, y, desc, va='center', fontsize=7.5, color='#555')

ax.set_xlim(-0.3, 6.5)
ax.set_ylim(-0.5, len(LINEA_TEMPORAL) + 0.7)
ax.set_yticks([])
ax.set_xticks(range(len(ETAPAS)))
ax.set_xticklabels(ETAPAS, fontsize=9)
ax.set_title('Timeline de disponibilidad de variables por etapa del cambio',
             fontweight='bold', fontsize=11)
patches = [mpatches.Patch(color=v, label=k.capitalize(), alpha=0.8)
           for k, v in col_colores.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# VALIDACION EMPIRICA DEL LEAKAGE
# 1. Lift de cada variable de leakage vs target
# 2. Comparacion modelo con leakage vs sin leakage
# ============================================================
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import roc_auc_score as _roc
from sklearn.preprocessing import LabelEncoder

FUGA_CONFIRMADAS = ['reversado', 'cantidaddevolucion', 'codigocierre', 'fecha_cierre',
                    'estado', 'pir', 'causa_raiz', 'ticket_externo', 'estado_padre']

y_all  = cambios_dedup['causo_incidente'].astype(int).values
prev_a = float(y_all.mean())

print('=== LIFT DE VARIABLES DE LEAKAGE VS TARGET ===')
print('Lift muy alto = la variable codifica el RESULTADO, no las condiciones previas\n')
leak_rows = []
for col in FUGA_CONFIRMADAS:
    if col not in cambios_dedup.columns: continue
    s = cambios_dedup[col]
    mask = s.notna() & (~s.astype(str).str.strip().str.lower().isin(['', 'nan', 'none', '<na>']))
    if mask.sum() < 5: continue
    t_con = float(y_all[mask].mean()) if mask.sum() > 0 else 0.0
    t_sin = float(y_all[~mask].mean()) if (~mask).sum() > 0 else 0.0
    lift  = t_con / max(t_sin, 1e-9)
    tag   = '** LEAKAGE CONFIRMADO **' if lift > 3 else ''
    leak_rows.append(dict(variable=col, lift=round(lift,1), tasa_con=round(100*t_con,2),
                          tasa_sin=round(100*t_sin,2), confirmado=lift > 3))
    print(f'  {col:30s}  lift={lift:6.1f}x  con={100*t_con:.1f}%  sin={100*t_sin:.1f}%  {tag}')

df_lk = pd.DataFrame(leak_rows) if leak_rows else pd.DataFrame()
if len(df_lk):
    fig, ax = plt.subplots(figsize=(9, max(3, 0.5*len(df_lk))))
    colors_lk = ['#e74c3c' if r else '#3498db' for r in df_lk['confirmado']]
    bars = ax.barh(df_lk['variable'], df_lk['lift'], color=colors_lk, alpha=0.85, edgecolor='white')
    ax.axvline(1.0, color='gray', ls='--', lw=1, label='Lift=1 (sin efecto)')
    ax.axvline(3.0, color='#e74c3c', ls=':', lw=1.5, label='Umbral sospecha (3x)')
    for bar_, v in zip(bars, df_lk['lift']):
        ax.text(bar_.get_width()+0.1, bar_.get_y()+bar_.get_height()/2,
                f'{v:.1f}x', va='center', fontsize=8, fontweight='bold')
    ax.set_xlabel('Lift vs prevalencia base')
    ax.set_title('Lift de variables de leakage (rojo = confirmado, lift > 3x)', fontweight='bold')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

# Comparacion modelo con leakage vs sin leakage
print('\n=== COMPARACION: MODELO CON LEAKAGE vs SIN LEAKAGE ===')
cam_s = cambios_dedup.sort_values(fecha_cam_col if fecha_cam_col else cambios_dedup.columns[0],
                                   na_position='last').reset_index(drop=True)
cut2 = int(len(cam_s) * 0.8)
y_tr2, y_te2 = cam_s['causo_incidente'].iloc[:cut2].values, cam_s['causo_incidente'].iloc[cut2:].values

def quick_roc(cols_use, label):
    if not cols_use: print(f'  {label}: sin columnas disponibles'); return None
    Xtr = cam_s[cols_use].iloc[:cut2].fillna(-1)
    Xte = cam_s[cols_use].iloc[cut2:].fillna(-1)
    for c in Xtr.columns:
        if Xtr[c].dtype == object:
            le = LabelEncoder().fit(Xtr[c].astype(str))
            Xtr = Xtr.copy(); Xte = Xte.copy()
            Xtr[c] = le.transform(Xtr[c].astype(str))
            inv = {v: i for i, v in enumerate(le.classes_)}
            Xte[c] = Xte[c].astype(str).map(inv).fillna(0).astype(int)
    if y_tr2.sum() == 0 or y_te2.sum() == 0:
        print(f'  {label}: sin positivos en split, saltando'); return None
    try:
        m = LR(max_iter=500, random_state=42, C=0.1).fit(Xtr, y_tr2)
        roc = _roc(y_te2, m.predict_proba(Xte)[:,1])
        print(f'  {label:45s}  ROC test = {roc:.4f}')
        return roc
    except Exception as e:
        print(f'  {label}: error ({e})')
        return None

cols_legit  = [c for c in cam_s.select_dtypes(include='number').columns
               if c not in FUGA_CONFIRMADAS and c != 'causo_incidente' and not c.startswith('_')][:5]
cols_leakage = [c for c in FUGA_CONFIRMADAS if c in cam_s.columns]
if cols_legit:
    roc_legit = quick_roc(cols_legit, 'Features legitimas (sin leakage)')
if cols_leakage:
    roc_leak  = quick_roc(cols_leakage, 'Solo variables de LEAKAGE')
if cols_legit and cols_leakage:
    print('\n  => Si ROC leakage >> ROC legitimas: LEAKAGE CONFIRMADO')
    print('  => El modelo con leakage aprende a hacer trampa, no a predecir')


### ✔ Conclusiones Sección 10: leakage

**Cómo se detectó:**
* Análisis de timing de negocio: cuándo se llena cada campo en el ciclo de vida del cambio
* Lift individual vs target: variables con lift > 3x son sospechosas
* Comparación de modelos con/sin leakage: el gap de ROC-AUC confirma la trampa

**Variables excluidas definitivamente:**
`reversado`, `cantidaddevolucion`, `codigocierre`, `fecha_cierre`, `estado`, `pir`, `causa_raiz`, `ticket_externo`, `estado_padre`

**La regla crítica para producción:** Solo puede usarse información disponible en el momento en que el CAB revisa el cambio — antes de su implementación. Ninguna variable de resultado puede entrar al modelo.

**Por qué importa tanto:**
* Un modelo con leakage parece perfectamente preciso en evaluación
* En producción falla completamente porque las variables de leakage no existen cuando el modelo necesita predecir
* Este es el error más común en proyectos de ML sobre datos de sistemas transaccionales

---
## Sección 11 — Ingeniería de variables (features derivadas)

**Regla de oro:** solo usamos columnas de rol `predictor`, es decir, lo que se conoce al solicitar/aprobar el cambio. Nada de variables de leakage.

### Las 5 familias de features

| Familia | Features | Por qué incluirlas |
|---|---|---|
| **Temporales del cambio** | `dur_prog_horas`, `antelacion_dias`, `hora_inicio`, `dow_inicio`, `mes_inicio`, `es_fin_semana`, `fuera_horario` | Los cambios fuera de horario y de larga duración tienen mayor riesgo |
| **Documentales** | `len_descripcion`, `len_plan_implementacion`, `len_business_justification`, `len_plan_reverso`, `tiene_plan_reverso` | Cambios mal documentados o sin plan de reverso son más riesgosos |
| **Recursos y escala** | `n_recursos`, `n_recalendarizaciones`, `cambios_solapados` | Más CIs afectados = mayor alcance = mayor riesgo |
| **Histórico del CI** | `hist_cambios_previos_ci`, `hist_fallos_previos_ci`, `hist_tasa_fallo_ci` | Un CI que ya falló antes tiene mayor probabilidad de volver a fallar |
| **Texto (TF-IDF)** | 300 features sobre texto combinado | Captura patrones semánticos no codificables en campos estructurados |

### La feature más delicada: historial del CI

El historial del CI es poderoso pero debe calcularse **estrictamente sobre el pasado**. Para cada cambio, solo se puede usar información de los cambios que ocurrieron **antes** de él. Si usamos el total histórico, filtramos el futuro hacia el pasado.

In [ ]:
# ============================================================
# SECCION 11: FEATURE ENGINEERING
# ============================================================

feat = cambios_dedup.copy()
feat['cambio_n'] = norm_id(feat['cambio'])

# Auxiliares de fecha
def a_fecha(col):
    if col in feat.columns:
        return pd.to_datetime(feat[col], errors='coerce')
    return pd.Series(pd.NaT, index=feat.index)

f_inicio = a_fecha('fecha_inicio_programado')
f_fin    = a_fecha('fecha_fin_programado')
f_aper   = a_fecha('fecha_apertura')

# --- FAMILIA 1: Temporales ---
feat['dur_prog_horas']  = (f_fin - f_inicio).dt.total_seconds().div(3600).clip(lower=0)
feat['antelacion_dias'] = (f_inicio - f_aper).dt.total_seconds().div(86400).clip(lower=0)
feat['hora_inicio']     = f_inicio.dt.hour
feat['dow_inicio']      = f_inicio.dt.dayofweek  # 0=lunes, 6=domingo
feat['mes_inicio']      = f_inicio.dt.month
feat['es_fin_semana']   = (feat['dow_inicio'] >= 5).astype(int)
feat['fuera_horario']   = ((feat['hora_inicio'] >= 20) | (feat['hora_inicio'] <= 7)).astype(int)

# --- FAMILIA 2: Documentales ---
para_len = {
    'descripcion':             'len_descripcion',
    'plan_implementacion':     'len_plan_implementacion',
    'planreverso':             'len_planreverso',
    'plan_reverso':            'len_plan_reverso',
    'business_justification':  'len_business_justification',
}
for src, dst in para_len.items():
    if src in feat.columns:
        feat[dst] = feat[src].fillna('').astype(str).str.len()
    else:
        feat[dst] = 0
feat['tiene_plan_reverso'] = (
    (feat.get('len_planreverso', pd.Series(0, index=feat.index)) > 10) |
    (feat.get('len_plan_reverso', pd.Series(0, index=feat.index)) > 10)
).astype(int)

# --- FAMILIA 3: Recursos y escala ---
if 'configuration_item' in feat.columns:
    feat['ci_primario'] = feat['configuration_item'].astype(str).str.split(r'\s*\|\s*').str[0].str.strip()
    feat['n_recursos']  = feat['configuration_item'].astype(str).str.split(r'\s*\|\s*').str.len()
else:
    feat['ci_primario'] = 'unknown'
    feat['n_recursos']  = 1

if 'cantidadrecalendarizacion' in feat.columns:
    feat['n_recalendarizaciones'] = pd.to_numeric(feat['cantidadrecalendarizacion'], errors='coerce').fillna(0)
else:
    feat['n_recalendarizaciones'] = 0

# Cambios solapados: cuantos otros cambios activos en la misma ventana temporal
if 'fecha_inicio_programado' in feat.columns and 'fecha_fin_programado' in feat.columns:
    feat = feat.sort_values('fecha_inicio_programado', na_position='last').reset_index(drop=True)
    # Simplificado: cuantos cambios abren en la misma semana
    # v2 FIX: recalcular la fecha DESPUES de ordenar. En v1 se usaba f_inicio
    # (calculada con el orden original) sobre el feat ya reordenado, lo que
    # asignaba la semana de OTRA fila a cada cambio.
    feat['_sem'] = pd.to_datetime(feat['fecha_inicio_programado'], errors='coerce').dt.to_period('W')
    sem_count = feat.groupby('_sem').size().to_dict()
    feat['cambios_solapados'] = feat['_sem'].map(sem_count).fillna(1) - 1
    feat = feat.drop(columns=['_sem'])
else:
    feat['cambios_solapados'] = 0

features_base = [
    'dur_prog_horas', 'antelacion_dias', 'dow_inicio', 'hora_inicio', 'mes_inicio',
    'es_fin_semana', 'fuera_horario', 'n_recursos', 'n_recalendarizaciones',
    'len_descripcion', 'len_plan_implementacion', 'len_planreverso', 'len_plan_reverso',
    'len_business_justification', 'tiene_plan_reverso', 'cambios_solapados',
]
features_base = [f for f in features_base if f in feat.columns]
print(f'Features construidas (familia 1-3): {features_base}')
print(f'Total: {len(features_base)} features')


In [ ]:
# ============================================================
# HISTORIAL DEL CI — Estrictamente previo (anti-leakage)
# v2 — dos correcciones importantes:
#   1. FIX de alineacion: en v1 '_fecha_ref' se construia con f_inicio
#      (orden original) sobre un feat ya reordenado -> fechas de otra fila.
#   2. hist_fallos_previos_ci ahora cuenta un fallo previo SOLO si el
#      INCIDENTE ya habia ocurrido antes del inicio del cambio actual.
#      En v1 bastaba con que el cambio anterior estuviera etiquetado, aunque
#      su incidente ocurriera despues del cambio que estamos puntuando
#      (fuga sutil de informacion futura).
# ============================================================

feat = feat.sort_values('fecha_inicio_programado', na_position='last').reset_index(drop=True)
f_ini_s = pd.to_datetime(feat['fecha_inicio_programado'], errors='coerce')

# Numero de cambios previos para el mismo CI
feat['hist_cambios_previos_ci'] = feat.groupby('ci_primario').cumcount()

# Fecha del (primer) incidente causado por cada cambio positivo
mapa_fecha_inc = {}
if 'px' in dir() and len(px) and fecha_inc_col in px.columns:
    _pc = px[px['dias_cambio_a_incidente'].fillna(0) >= 0]
    mapa_fecha_inc = _pc.groupby('cambio_n')[fecha_inc_col].min().to_dict()

feat['_f_inc_causado'] = pd.to_datetime(feat['cambio_n'].map(mapa_fecha_inc), errors='coerce')

hist_fallos = np.zeros(len(feat))
for ci, g in feat.groupby('ci_primario'):
    inc_dates = np.sort(g['_f_inc_causado'].dropna().values)
    if len(inc_dates) == 0:
        continue
    for ix in g.index:
        f = f_ini_s.iloc[ix]
        if pd.notna(f):
            # cuantos incidentes (causados por cambios de este CI) ya ocurrieron
            hist_fallos[ix] = np.searchsorted(inc_dates, np.datetime64(f), side='left')
feat['hist_fallos_previos_ci'] = hist_fallos
feat['hist_tasa_fallo_ci'] = feat['hist_fallos_previos_ci'] / (feat['hist_cambios_previos_ci'] + 1)
feat = feat.drop(columns=['_f_inc_causado'])

# Presion de cambios: cuantos cambios en el mismo CI en los ultimos N dias
VENTANAS_DIAS = [7, 30]
for v in VENTANAS_DIAS:
    feat[f'cambios_ci_{v}d'] = 0

feat['_fecha_ref'] = f_ini_s.fillna(pd.Timestamp('2100-01-01'))
for ci, g in feat.groupby('ci_primario'):
    fechas = g['_fecha_ref'].values  # feat ya esta ordenado por fecha
    for v in VENTANAS_DIAS:
        ventana = np.timedelta64(v, 'D')
        counts = []
        for f in fechas:
            inicio_v = f - ventana
            n = np.searchsorted(fechas, f, side='left') - np.searchsorted(fechas, inicio_v, side='left')
            counts.append(max(0, n))
        feat.loc[g.index, f'cambios_ci_{v}d'] = counts
feat = feat.drop(columns=['_fecha_ref'])

historico_features = [
    'hist_cambios_previos_ci', 'hist_fallos_previos_ci', 'hist_tasa_fallo_ci'
] + [f'cambios_ci_{v}d' for v in VENTANAS_DIAS]
print(f'Features historial de CI: {historico_features}')

# Dense cols final
dense_cols = [f for f in features_base + historico_features if f in feat.columns]
print(f'\nTotal dense features: {len(dense_cols)}')
print(f'Muestra de historial de CI:')
print(feat[['ci_primario', 'hist_cambios_previos_ci', 'hist_fallos_previos_ci',
            'hist_tasa_fallo_ci']].head(5).to_string(index=False))


---
## Sección 11B (v2) — Nuevas familias de features

Se agregan tres familias, todas calculadas **estrictamente con información previa al cambio**:

| Familia | Features | Racional |
|---|---|---|
| **Recencia** | `dias_desde_ultimo_cambio_ci`, `dias_desde_ultimo_incidente_ci` | Un CI tocado ayer no es lo mismo que uno estable hace 6 meses; un CI con un incidente reciente (de cualquier causa) está "caliente" |
| **Historial del grupo ejecutor** | `hist_cambios_previos_grupo`, `hist_fallos_previos_grupo`, `hist_tasa_fallo_grupo` | El riesgo no depende solo del activo sino de quién ejecuta: equipos con historial de fallos tienden a repetir |
| **Señales de texto (7B)** | `es_boilerplate`, `freq_texto`, `kw_*` | Documentación con plantilla = menor esfuerzo; keywords = riesgo técnico explicable |

**Nota anti-leakage:** `dias_desde_ultimo_incidente_ci` usa la fecha de apertura del incidente (conocida en tiempo real cuando ocurre), y sólo mira incidentes **anteriores** al inicio del cambio. El historial del grupo usa la misma lógica de fecha de incidente que el historial del CI corregido en 11.


In [ ]:
# ============================================================
# SECCION 11B (v2): RECENCIA + HISTORIAL DEL GRUPO + TEXTO
# ============================================================
# feat ya esta ordenado por fecha_inicio_programado (Seccion 11)
f_ini_s = pd.to_datetime(feat['fecha_inicio_programado'], errors='coerce')
feat['_f'] = f_ini_s

# --- 1. Dias desde el ultimo cambio en el mismo CI ---
feat['dias_desde_ultimo_cambio_ci'] = (
    feat.groupby('ci_primario')['_f'].diff().dt.total_seconds().div(86400)
).clip(lower=0).fillna(9999)

# --- 2. Dias desde el ultimo incidente (de cualquier causa) en el mismo CI ---
feat['dias_desde_ultimo_incidente_ci'] = 9999.0
if CI_COL_INC and fecha_inc_col in incidentes_dedup.columns:
    inc_ev = incidentes_dedup[[CI_COL_INC, fecha_inc_col]].copy()
    inc_ev['ci_primario'] = (inc_ev[CI_COL_INC].astype(str)
                             .str.split(r'\s*\|\s*').str[0].str.strip())
    inc_ev['_f_inc'] = pd.to_datetime(inc_ev[fecha_inc_col], errors='coerce')
    inc_ev = inc_ev.dropna(subset=['_f_inc'])[['ci_primario', '_f_inc']].sort_values('_f_inc')

    cam_tmp = (feat[['ci_primario', '_f']].reset_index()
               .rename(columns={'index': '_ix'})
               .dropna(subset=['_f']).sort_values('_f'))
    # merge_asof: para cada cambio, el ultimo incidente ESTRICTAMENTE anterior
    m_asof = pd.merge_asof(cam_tmp, inc_ev, on=None,
                           left_on='_f', right_on='_f_inc',
                           by='ci_primario', allow_exact_matches=False)
    _dias = (m_asof['_f'] - m_asof['_f_inc']).dt.total_seconds().div(86400)
    feat.loc[m_asof['_ix'].values, 'dias_desde_ultimo_incidente_ci'] = _dias.fillna(9999).values
    print(f'Cambios con algun incidente previo en su CI: '
          f'{(feat["dias_desde_ultimo_incidente_ci"] < 9999).mean()*100:.1f}%')
else:
    print('CI o fecha de incidentes no disponibles; recencia de incidentes = 9999')

# --- 3. Historial del grupo ejecutor (estrictamente previo) ---
GRUPO_COL = next((c for c in ['grupo', 'grupo_ejecutor', 'grupo_asignado'] if c in feat.columns), None)
if GRUPO_COL:
    feat['_grupo'] = feat[GRUPO_COL].astype(str).str.strip().str.lower()
    feat['hist_cambios_previos_grupo'] = feat.groupby('_grupo').cumcount()
    # misma logica que el CI: el fallo cuenta solo cuando su incidente ya ocurrio
    feat['_f_inc_causado'] = pd.to_datetime(feat['cambio_n'].map(mapa_fecha_inc), errors='coerce')
    hist_fallos_g = np.zeros(len(feat))
    for gr, g in feat.groupby('_grupo'):
        inc_dates = np.sort(g['_f_inc_causado'].dropna().values)
        if len(inc_dates) == 0:
            continue
        for ix in g.index:
            f = f_ini_s.iloc[ix]
            if pd.notna(f):
                hist_fallos_g[ix] = np.searchsorted(inc_dates, np.datetime64(f), side='left')
    feat['hist_fallos_previos_grupo'] = hist_fallos_g
    feat['hist_tasa_fallo_grupo'] = (
        feat['hist_fallos_previos_grupo'] / (feat['hist_cambios_previos_grupo'] + 1))
    feat = feat.drop(columns=['_grupo', '_f_inc_causado'])
    print(f'Historial de grupo construido sobre "{GRUPO_COL}"')
else:
    print('Columna de grupo ejecutor no encontrada; familia omitida')

feat = feat.drop(columns=['_f'])

# --- 4. Consolidar las nuevas features en dense_cols ---
features_v2 = ['dias_desde_ultimo_cambio_ci', 'dias_desde_ultimo_incidente_ci',
               'hist_cambios_previos_grupo', 'hist_fallos_previos_grupo',
               'hist_tasa_fallo_grupo', 'freq_texto', 'es_boilerplate']
features_v2 += [k for k in KEYWORDS_RIESGO.keys()] if 'KEYWORDS_RIESGO' in dir() else []
features_v2 = [f for f in features_v2 if f in feat.columns]

dense_cols = list(dict.fromkeys(dense_cols + features_v2))
print(f'\nNuevas features v2 incorporadas ({len(features_v2)}): {features_v2}')
print(f'Total dense features: {len(dense_cols)}')


### ✔ Conclusiones Sección 11: feature engineering

* **El historial del CI es la familia más predictiva**: un CI que ya causó incidentes antes tiene mayor probabilidad de volver a fallar. Calcular esto correctamente (sin filtrar el futuro) es la diferencia entre un modelo válido y uno con leakage inadvertido
* **La documentación es una señal indirecta de calidad**: un cambio sin plan de reverso, sin justificación o con descripción corta sugiere menor rigor en la planificación
* **Los cambios fuera de horario tienen mayor riesgo**: `fuera_horario` y `es_fin_semana` capturan cambios ejecutados fuera de ventana habitual, donde el soporte es más limitado
* **La presión de cambios en el CI**: muchos cambios en poco tiempo en el mismo aplicativo aumentan la complejidad y el riesgo acumulado
* **Las variables categóricas** (`tipo_de_cambio`, `riesgo`, `categoria`) se codifican como enteros para ser consumidas por los modelos de boosting
* **TF-IDF sobre texto combinado**: captura vocabulario de riesgo que no está en ningún campo estructurado

---
## Sección 12 — Relación de features con la variable objetivo

### Por qué este análisis

Antes de entrenar cualquier modelo, validamos que las features tienen **alguna relación estadistica real** con el target. Si ninguna tuviera relación, el modelo no aprendería nada.

### Métodos utilizados

1. **Correlación punto-biserial**: equivalente a correlación de Pearson cuando el target es binario (0/1). Un valor alto indica que la feature separa bien las dos clases.

2. **Lift por categoría**: para variables categóricas, qué valores tienen mayor tasa de incidentes respecto a la media. Un lift de 3x significa que ese valor de la variable tiene una tasa de incidentes 3 veces mayor que el promedio.

3. **Boxplots por clase**: visualización de la distribución de las features numéricas para cambios que causaron incidente vs los que no.

In [ ]:
# ============================================================
# SECCION 12: FEATURES VS VARIABLE OBJETIVO
# ============================================================

y_feat  = feat['causo_incidente'].astype(float)
prev_fv = float(y_feat.mean())
X_num   = feat[dense_cols].fillna(0).astype(float)

# --- 1. Correlacion punto-biserial ---
corrs = X_num.corrwith(y_feat).dropna().sort_values(key=np.abs, ascending=False)

print('Correlacion punto-biserial con causo_incidente (|r| mayor = mas predictiva):')
for fn, v in corrs.items():
    bar = '#' * int(abs(v) * 60)
    print(f'  {fn:40s}  r={v:+.4f}  {bar}')

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
colors_c = ['#e74c3c' if v > 0 else '#2e86de' for v in corrs.values]
ax.barh(range(len(corrs)), corrs.values[::-1], color=colors_c[::-1], alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(corrs)))
ax.set_yticklabels(list(corrs.index)[::-1], fontsize=8)
ax.axvline(0, color='gray', lw=1)
ax.axvline(0.05,  color='#27ae60', ls=':', lw=1.5, alpha=0.7)
ax.axvline(-0.05, color='#27ae60', ls=':', lw=1.5, alpha=0.7)
ax.set_title('Correlacion punto-biserial features vs target', fontweight='bold')
ax.set_xlabel('Correlacion')

# Boxplot top-3 features
ax = axes[1]
top3 = list(corrs.head(3).index)
pos_data = [feat[feat['causo_incidente']==1][c].fillna(0).clip(
    upper=np.percentile(feat[c].fillna(0), 99)).values for c in top3]
neg_data = [feat[feat['causo_incidente']==0][c].fillna(0).clip(
    upper=np.percentile(feat[c].fillna(0), 99)).values for c in top3]

pos_bp = ax.boxplot(pos_data, positions=[0.25, 1.25, 2.25], widths=0.45,
                    patch_artist=True, boxprops=dict(facecolor='#e74c3c', alpha=0.6),
                    medianprops=dict(color='white', lw=2.5), flierprops=dict(ms=2, alpha=0.3))
neg_bp = ax.boxplot(neg_data, positions=[0.75, 1.75, 2.75], widths=0.45,
                    patch_artist=True, boxprops=dict(facecolor='#2e86de', alpha=0.6),
                    medianprops=dict(color='white', lw=2.5), flierprops=dict(ms=2, alpha=0.3))
ax.set_xticks([0.5, 1.5, 2.5])
ax.set_xticklabels([t[:25] for t in top3], rotation=15, ha='right', fontsize=9)
ax.set_title('Top-3 features por clase (rojo=incidente, azul=no incidente)', fontweight='bold')
leg = [mpatches.Patch(facecolor='#e74c3c', label='Causo incidente'),
       mpatches.Patch(facecolor='#2e86de', label='No causo incidente')]
ax.legend(handles=leg, fontsize=9)

plt.suptitle('Analisis de features vs variable objetivo', fontweight='bold')
plt.tight_layout(); plt.show()

# Resumen por familia
familias = {
    'Historial CI': [c for c in dense_cols if 'hist' in c or 'cambios_ci' in c],
    'Fecha/hora'  : [c for c in dense_cols if any(x in c for x in ['hora','dow','semana','horario','mes','fur'])],
    'Duracion/ant': [c for c in dense_cols if any(x in c for x in ['dur','antelacion'])],
    'Texto (len)' : [c for c in dense_cols if any(x in c for x in ['len','tiene','recursos','recal','solapados'])],
}
print('\nPoder predictivo por familia de features:')
for fam, cols in familias.items():
    if not cols: continue
    max_r  = float(corrs.reindex(cols).abs().max())
    mean_r = float(corrs.reindex(cols).abs().mean())
    print(f'  {fam:20s}  {len(cols):2d} features  max|r|={max_r:.4f}  mean|r|={mean_r:.4f}')


In [ ]:
# --- 2. Lift de variables categoricas ---
print('Lift de variables categoricas vs target (top 5 por lift):\n')
col_map = {'tipo_de_cambio': 'Tipo de cambio', 'riesgo': 'Riesgo', 'categoria': 'Categoria'}

cat_lift_data = []
for col, titulo in col_map.items():
    if col not in feat.columns: continue
    grp = (feat.groupby(col)['causo_incidente']
               .agg(['mean','sum','count'])
               .assign(lift=lambda x: x['mean']/max(prev_fv, 1e-9))
               .sort_values('lift', ascending=False))
    cat_lift_data.append((col, titulo, grp))
    print(f'{titulo}:')
    for row in grp.head(5).itertuples():
        bar = '#' * min(int(row.lift * 5), 30)
        print(f'  {str(row.Index)[:35]:35s}  lift={row.lift:.1f}x  '
              f'pos={int(row.sum):3d}/{int(row.count):6d}  [{bar}]')
    print()

if cat_lift_data:
    n_p = len(cat_lift_data)
    fig, axes = plt.subplots(1, n_p, figsize=(6*n_p, 5))
    if n_p == 1: axes = [axes]
    for ax, (col, titulo, grp) in zip(axes, cat_lift_data):
        top10 = grp.head(10)
        colors_cl = ['#e74c3c' if v > 2 else '#f39c12' if v > 1 else '#3498db'
                     for v in top10['lift']]
        ax.bar(range(len(top10)), top10['lift'].values,
               color=colors_cl, alpha=0.85, edgecolor='white')
        ax.set_xticks(range(len(top10)))
        ax.set_xticklabels([str(x)[:18] for x in top10.index],
                           rotation=30, ha='right', fontsize=7)
        ax.axhline(1.0, color='gray', ls='--', lw=1)
        ax.set_title(f'Lift vs target: {titulo}', fontweight='bold')
        ax.set_ylabel('Lift')
    plt.suptitle('Lift de categorias vs variable objetivo', fontweight='bold')
    plt.tight_layout(); plt.show()


### ✔ Conclusiones Sección 12: features vs target

* **El historial del CI lidera**: `hist_fallos_previos_ci` y `hist_tasa_fallo_ci` son las features con mayor correlación — confirma la hipótesis de que activos con historial de fallos son más propensos a reincidencia
* **Las variables de texto tienen baja correlación individual pero combinación útil**: TF-IDF captura n-gramas de riesgo que no se ven en correlaciones lineales simples
* **Lift diferencial por categoría**: ciertos tipos de cambio y niveles de riesgo tienen tasas de incidente muy distintas al promedio — señal directamente aprovechable
* **Ninguna feature individual tiene r > 0.4**: el modelo necesita combinar señales débiles, lo que justifica usar boosting (XGBoost) en vez de una regresión lineal
* **Variables con |r| < 0.01 se mantienen en el pipeline**: en datasets muy desbalanceados la correlación individual subestima la utilidad combinada

---
## Sección 12B (v3) — Análisis ampliado de features: información mutua, IV/WOE y redundancia

Tres lentes complementarios a la correlación punto-biserial (que solo ve relaciones **lineales**):

| Técnica | Qué aporta | Convención |
|---|---|---|
| **Información mutua** | Captura relaciones no lineales y no monótonas (p.ej. riesgo alto en horas extremas) | Comparar el ranking contra el de correlación: discrepancias = señal no lineal |
| **Information Value (IV/WOE)** | Estándar bancario para riesgo; permite hablar el mismo idioma que los equipos de crédito | <0.02 inútil · 0.02-0.1 débil · 0.1-0.3 media · 0.3-0.5 fuerte · **>0.5 sospechosa de fuga** |
| **Redundancia (Spearman)** | Pares de features casi duplicadas inflan la varianza de la importancia y confunden la explicación al CAB | \|ρ\| > 0.85 = candidata a consolidar |

> Análisis descriptivo sobre el dataset completo (EDA). La selección de modelo sigue ocurriendo solo en validación.


In [ ]:
# ============================================================
# SECCION 12B (v3): INFORMACION MUTUA + IV/WOE + REDUNDANCIA
# ============================================================
from sklearn.feature_selection import mutual_info_classif

X_an = feat[dense_cols].fillna(0).astype(float)
y_an = feat['causo_incidente'].astype(int).values

# --- 1. Informacion mutua ---
mi = mutual_info_classif(X_an, y_an, random_state=42)
df_mi = pd.Series(mi, index=dense_cols, name='MI').sort_values(ascending=False)

# --- 2. Information Value (IV) con binning por quintiles y suavizado ---
def iv_feature(x, y, bins=5):
    x = pd.Series(x).fillna(-999999)
    if x.nunique() <= 10:
        grupos = x.astype(str)
    else:
        try:
            grupos = pd.qcut(x, bins, duplicates='drop').astype(str)
        except ValueError:
            return np.nan
    tmp = pd.DataFrame({'g': grupos.values, 'y': y})
    agg = tmp.groupby('g')['y'].agg(['sum', 'count'])
    pos = agg['sum'] + 0.5                      # suavizado
    neg = agg['count'] - agg['sum'] + 0.5
    dist_pos = pos / pos.sum()
    dist_neg = neg / neg.sum()
    return float(((dist_pos - dist_neg) * np.log(dist_pos / dist_neg)).sum())

def clasificar_iv(v):
    if np.isnan(v):  return '?'
    if v > 0.5:      return 'SOSPECHOSA DE FUGA'
    if v > 0.3:      return 'fuerte'
    if v > 0.1:      return 'media'
    if v > 0.02:     return 'debil'
    return 'inutil'

rows_iv = [(c, iv_feature(X_an[c].values, y_an)) for c in dense_cols]
# IV de las categoricas originales (antes de codificar)
for c in ['tipo_de_cambio', 'riesgo', 'categoria']:
    if c in feat.columns:
        rows_iv.append((f'{c} (categorica)', iv_feature(feat[c].astype(str), y_an)))
df_iv = (pd.DataFrame(rows_iv, columns=['feature', 'IV'])
         .assign(clase=lambda d: d['IV'].apply(clasificar_iv))
         .sort_values('IV', ascending=False))

print('TOP 15 POR INFORMATION VALUE (convencion bancaria):')
print(df_iv.head(15).to_string(index=False, float_format='{:.4f}'.format))
_sospechosas = df_iv[df_iv['clase'] == 'SOSPECHOSA DE FUGA']
if len(_sospechosas):
    print('\n*** REVISAR: IV > 0.5 suele indicar leakage residual ***')
    print(_sospechosas.to_string(index=False))

# --- 3. Redundancia: Spearman entre features densas ---
corr_sp = X_an.corr(method='spearman')
pares_red = []
cols_l = list(corr_sp.columns)
for a in range(len(cols_l)):
    for b in range(a + 1, len(cols_l)):
        r = corr_sp.iloc[a, b]
        if abs(r) > 0.85:
            pares_red.append((cols_l[a], cols_l[b], round(float(r), 3)))
print(f'\nPARES REDUNDANTES (|rho Spearman| > 0.85): {len(pares_red)}')
for a, b, r in sorted(pares_red, key=lambda t: -abs(t[2]))[:10]:
    print(f'  {a:35s} ~ {b:35s}  rho={r:+.3f}')
if pares_red:
    print('  -> Consolidar: quedarse con la de mayor IV de cada par simplifica el')
    print('     modelo y estabiliza la importancia de features.')

# --- Visualizacion ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

ax = axes[0]
top_mi = df_mi.head(15)
ax.barh(range(len(top_mi)), top_mi.values[::-1], color='#16a085', alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(top_mi)))
ax.set_yticklabels(top_mi.index[::-1], fontsize=8)
ax.set_title('Top 15 — Informacion mutua con el target', fontweight='bold')
ax.set_xlabel('MI')

ax = axes[1]
top_iv = df_iv[~df_iv['feature'].str.contains('categorica')].head(15)
colors_iv = ['#e74c3c' if c == 'SOSPECHOSA DE FUGA' else '#f39c12' if c == 'fuerte'
             else '#2e86de' for c in top_iv['clase']]
ax.barh(range(len(top_iv)), top_iv['IV'].values[::-1],
        color=colors_iv[::-1], alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(top_iv)))
ax.set_yticklabels(top_iv['feature'].values[::-1], fontsize=8)
for lim, lab in [(0.02, ''), (0.1, 'media'), (0.3, 'fuerte'), (0.5, 'fuga?')]:
    ax.axvline(lim, color='gray', ls=':', lw=1)
ax.set_title('Top 15 — Information Value\n(rojo = revisar posible fuga)', fontweight='bold')
ax.set_xlabel('IV')

ax = axes[2]
im = ax.imshow(corr_sp.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(cols_l)))
ax.set_xticklabels(cols_l, rotation=90, fontsize=5.5)
ax.set_yticks(range(len(cols_l)))
ax.set_yticklabels(cols_l, fontsize=5.5)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Matriz de redundancia (Spearman)', fontweight='bold')

plt.suptitle('Analisis ampliado de features (v3)', fontweight='bold')
plt.tight_layout(); plt.show()

# Discrepancias MI vs correlacion lineal (senal no lineal)
rank_mi = df_mi.rank(ascending=False)
rank_corr = corrs.abs().rank(ascending=False)
comunes = rank_mi.index.intersection(rank_corr.index)
diff_rank = (rank_corr[comunes] - rank_mi[comunes]).sort_values()
print('\nFeatures que la correlacion lineal SUBESTIMA (alto MI, baja r):')
print('  (candidatas a efectos no lineales que los arboles si capturan)')
for f_, d_ in diff_rank.tail(5).items():
    print(f'  {f_:40s}  rank_r={int(rank_corr[f_]):2d}  rank_MI={int(rank_mi[f_]):2d}')


---
## Sección 12C (v3) — Texto vs target: palabras discriminantes y tópicos

El TF-IDF mete el texto al modelo, pero no responde la pregunta del negocio: **¿qué vocabulario distingue a los cambios que causan incidentes?** Dos técnicas:

1. **Log-odds suavizado por clase**: para cada término, log de la razón entre su frecuencia relativa en positivos vs negativos (con suavizado de Laplace para no dividir por cero con ~166 positivos). Mucho más fiable que "top TF-IDF por clase" cuando las clases son desbalanceadas.
2. **Tópicos NMF**: agrupa el corpus en temas (parcheo, certificados, BD, red...) y mide la **tasa de incidente por tópico**. Si un tópico concentra el triple de la prevalencia base, ese tipo de trabajo es estructuralmente más riesgoso — insight accionable para el CAB aunque el modelo no existiera.

> Con pocos positivos, los IC son anchos: tratar los términos como **hipótesis para validar con Operaciones**, no como verdades.


In [ ]:
# ============================================================
# SECCION 12C (v3): PALABRAS DISCRIMINANTES + TOPICOS NMF
# ============================================================
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF

if 'texto_mvp1' in feat.columns and feat['causo_incidente'].sum() >= 5:
    mask_txt = feat['texto_mvp1'].fillna('').str.len() > 10
    docs = feat.loc[mask_txt, 'texto_mvp1']
    y_txt = feat.loc[mask_txt, 'causo_incidente'].values
    print(f'Documentos: {len(docs):,}  |  positivos con texto: {int(y_txt.sum())}')

    # --- 1. Log-odds suavizado (positivos vs negativos) ---
    cv_lo = CountVectorizer(ngram_range=(1, 2), min_df=5,
                            stop_words=sorted(STOPWORDS_ES), binary=True)
    Xc = cv_lo.fit_transform(docs)
    terms = np.array(cv_lo.get_feature_names_out())
    alpha = 0.5
    cnt_pos = np.asarray(Xc[y_txt == 1].sum(axis=0)).ravel() + alpha
    cnt_neg = np.asarray(Xc[y_txt == 0].sum(axis=0)).ravel() + alpha
    logodds = np.log(cnt_pos / cnt_pos.sum()) - np.log(cnt_neg / cnt_neg.sum())
    # exigir presencia minima en positivos para el lado "riesgoso"
    min_pos = max(3, int(0.03 * y_txt.sum()))
    validos_pos = (cnt_pos - alpha) >= min_pos
    idx_risk = np.argsort(-np.where(validos_pos, logodds, -np.inf))[:15]
    idx_safe = np.argsort(logodds)[:15]

    print(f'\nTERMINOS ASOCIADOS A INCIDENTE (log-odds, presentes en >= {min_pos} positivos):')
    for ix in idx_risk:
        if not np.isfinite(logodds[ix]):
            continue
        print(f'  {terms[ix]:30s}  log-odds={logodds[ix]:+.2f}  '
              f'en_positivos={int(cnt_pos[ix]-alpha):3d}/{int(y_txt.sum())}  '
              f'en_negativos={int(cnt_neg[ix]-alpha):5d}')

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    ax = axes[0]
    ixs = [ix for ix in idx_risk if np.isfinite(logodds[ix])][:15]
    ax.barh(range(len(ixs)), logodds[ixs][::-1], color='#e74c3c', alpha=0.85, edgecolor='white')
    ax.set_yticks(range(len(ixs)))
    ax.set_yticklabels(terms[ixs][::-1], fontsize=8.5)
    ax.set_title('Vocabulario asociado a INCIDENTE', fontweight='bold')
    ax.set_xlabel('Log-odds (suavizado)')
    ax = axes[1]
    ax.barh(range(len(idx_safe)), logodds[idx_safe][::-1], color='#27ae60',
            alpha=0.85, edgecolor='white')
    ax.set_yticks(range(len(idx_safe)))
    ax.set_yticklabels(terms[idx_safe][::-1], fontsize=8.5)
    ax.set_title('Vocabulario asociado a cambios SIN incidente', fontweight='bold')
    ax.set_xlabel('Log-odds (suavizado)')
    plt.suptitle('Palabras y bigramas discriminantes del corpus', fontweight='bold')
    plt.tight_layout(); plt.show()

    # --- 2. Topicos NMF + tasa de incidente por topico ---
    N_TOPICOS = 8
    tf_top = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=5,
                             stop_words=sorted(STOPWORDS_ES), sublinear_tf=True)
    X_top = tf_top.fit_transform(docs)
    nmf = NMF(n_components=N_TOPICOS, random_state=42, max_iter=400, init='nndsvda')
    W = nmf.fit_transform(X_top)
    H = nmf.components_
    terms_top = np.array(tf_top.get_feature_names_out())

    top_dom = W.argmax(axis=1)
    top_dom[W.max(axis=1) == 0] = -1  # documentos sin topico claro

    prev_base = float(y_txt.mean())
    print(f'\nTOPICOS NMF (k={N_TOPICOS}) — tasa de incidente por topico '
          f'(base={prev_base*100:.2f}%):')
    rows_top = []
    for t in range(N_TOPICOS):
        palabras = ', '.join(terms_top[np.argsort(-H[t])[:6]])
        m_t = top_dom == t
        n_t, pos_t = int(m_t.sum()), int(y_txt[m_t].sum())
        tasa_t = pos_t / max(n_t, 1)
        lift_t = tasa_t / max(prev_base, 1e-9)
        rows_top.append(dict(topico=t, n=n_t, pos=pos_t,
                             tasa_pct=round(tasa_t * 100, 2),
                             lift=round(lift_t, 2), palabras=palabras))
        marca = ' <-- RIESGOSO' if (lift_t > 2 and pos_t >= 3) else ''
        print(f'  T{t}: n={n_t:5,}  pos={pos_t:3d}  tasa={tasa_t*100:5.2f}%  '
              f'lift={lift_t:4.1f}x{marca}')
        print(f'      [{palabras}]')
    df_topicos = pd.DataFrame(rows_top)
    print('\nLectura: topicos con lift>2x y >=3 positivos identifican TIPOS DE TRABAJO')
    print('estructuralmente riesgosos. Validar la etiqueta del topico con TI antes')
    print('de comunicarlo (NMF agrupa por co-ocurrencia, no por semantica perfecta).')
else:
    print('Texto o positivos insuficientes para el analisis discriminante')


---
## Sección 13 — Análisis por CI / Aplicativo

### Pregunta central

¿Los aplicativos (CIs) que reciben más cambios tienen mayor tasa de incidentes?

### Por qué esto importa

Si existe una correlación entre volumen de cambios por CI y tasa de incidentes, entonces:
1. El historial del CI es predictivo (confirmado)
2. El modelo puede aprender patrones de riesgo éspecificos por aplicativo
3. Los CIs con mayor actividad pueden recibir mayor escrutinio automático

### Lo que encontramos

Hay 86 CIs en común entre incidentes y cambios, lo que valida que el enlace entre ambas tablas es real y no casual. Los CIs con mayor volumen de cambios tendieron a concentrar más incidentes.

In [ ]:
# ============================================================
# SECCION 13: ANALISIS DE CI
# ============================================================

if 'ci_primario' not in feat.columns:
    feat['ci_primario'] = 'unknown'

# Perfil por CI
ci_profile = (
    feat.groupby('ci_primario')
    .agg(
        n_cambios=('cambio_n', 'count'),
        n_incidentes=('causo_incidente', 'sum'),
    )
    .assign(tasa_incidente_pct=lambda x: 100*x['n_incidentes']/x['n_cambios'])
    .sort_values('n_cambios', ascending=False)
    .reset_index()
)

print(f'Total CIs unicos: {len(ci_profile):,}')
print(f'CIs con al menos 1 incidente causado: {(ci_profile["n_incidentes"] > 0).sum()}')
print(f'Top 10 CIs por volumen de cambios:')
print(ci_profile.head(10).to_string(index=False))

# Top CIs por tasa de incidente (al menos 10 cambios para evitar ruido)
ci_min = ci_profile[ci_profile['n_cambios'] >= 10]
if len(ci_min):
    print(f'\nTop 10 CIs por tasa de incidente (min 10 cambios):')
    print(ci_min.sort_values('tasa_incidente_pct', ascending=False).head(10).to_string(index=False))

# Scatter: volumen vs tasa de incidente
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(ci_profile['n_cambios'], ci_profile['tasa_incidente_pct'],
           alpha=0.4, s=20, color='#2e86de', edgecolor='none')
ax.axhline(100*prev_fv, color='#e74c3c', ls='--', lw=1.5,
           label=f'Promedio global {100*prev_fv:.2f}%')
# Anotar top 5
top5_ci = ci_min.sort_values('tasa_incidente_pct', ascending=False).head(5)
for _, row in top5_ci.iterrows():
    ax.annotate(str(row['ci_primario'])[:15], (row['n_cambios'], row['tasa_incidente_pct']),
                fontsize=6.5, alpha=0.8)
ax.set_xlabel('N cambios por CI'); ax.set_ylabel('Tasa incidentes (%)')
ax.set_title('Volumen de cambios vs tasa de incidente por CI', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.2)

ax = axes[1]
ci_top = ci_min.sort_values('n_incidentes', ascending=False).head(15)
colors_ci = ['#e74c3c' if n > 0 else '#3498db' for n in ci_top['n_incidentes']]
ax.barh(range(len(ci_top)), ci_top['n_cambios'].values, color='#3498db', alpha=0.5, label='Cambios')
ax2 = ax.twiny()
ax2.barh(range(len(ci_top)), ci_top['n_incidentes'].values, color='#e74c3c', alpha=0.85, label='Incidentes')
ax.set_yticks(range(len(ci_top)))
ax.set_yticklabels([str(x)[:25] for x in ci_top['ci_primario']], fontsize=7.5)
ax.set_xlabel('N cambios', color='#3498db')
ax2.set_xlabel('N incidentes', color='#e74c3c')
ax.set_title('Top 15 CIs por incidentes causados', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8)

plt.suptitle('Analisis por CI / Aplicativo', fontweight='bold')
plt.tight_layout(); plt.show()

# Concentracion del target en CIs
n_ci_con_inc = (ci_profile['n_incidentes'] > 0).sum()
n_ci_top20   = int(len(ci_profile) * 0.2)
contrib_top20 = ci_profile.sort_values('n_incidentes', ascending=False).head(n_ci_top20)['n_incidentes'].sum()
print(f'\nConcentracion:')
print(f'  {n_ci_con_inc} CIs con al menos 1 incidente ({100*n_ci_con_inc/len(ci_profile):.1f}% de CIs)')
print(f'  Top 20% CIs concentran {contrib_top20}/{ci_profile["n_incidentes"].sum()} '
      f'incidentes ({100*contrib_top20/max(ci_profile["n_incidentes"].sum(),1):.0f}%)')


### ✔ Conclusiones Sección 13: análisis de CI

* **Los incidentes se concentran en pocos CIs**: no todos los aplicativos tienen el mismo nivel de riesgo. El historial del CI es la feature más valiosa del modelo
* **La relación volumen→tasa no es lineal**: CIs con muchos cambios no necesariamente tienen mayor tasa de incidentes. Lo que importa es el historial de fallos previos, no el volumen absoluto
* **86 CIs en común validan el enlace**: la superposición entre aplicativos de incidentes y cambios es suficiente para que el modelo pueda generalizar
* **Para MVP 1**: usar `ci_primario` como identificador del historial es suficiente. Para MVP 2: jerarquía de CIs (padre/hijo) permite heredar el historial

---
## Sección 14 — Conclusiones finales del EDA — Guía para el MVP 1

Este EDA no es solo exploración: es la base analítica que justifica cada decisión de modelado.

---

### 1. Variable objetivo

**Elección: D1 — link directo con validación temporal**

- 166 positivos / 14,764 cambios = **1.12% de prevalencia real**
- La etiqueta es explícita (documentada por el equipo de operaciones), no inferida
- D2 y D3 (CI+ventana) inflan 3-5x con falsos positivos estructurales

### 2. Leakage: las 9+ variables excluidas

```
reversado, cantidaddevolucion, codigocierre, fecha_cierre, estado,
pir, causa_raiz, ticket_externo, estado_padre
```

Se detectaron mediante: (a) lógica de timing de negocio, (b) lift individual > 3x, (c) comparación de modelos con/sin leakage.

### 3. Features para el MVP 1

| Familia | Features | N |
|---|---|---|
| Historial CI | `hist_fallos_previos_ci`, `hist_tasa_fallo_ci`, `hist_cambios_previos_ci`, `cambios_ci_7d/30d` | 5 |
| Temporales | `dur_prog_horas`, `antelacion_dias`, `dow_inicio`, `hora_inicio`, `mes_inicio`, `es_fin_semana`, `fuera_horario` | 7 |
| Documentales | `len_descripcion`, `len_plan_implementacion`, `len_business_justification`, `len_plan_reverso`, `tiene_plan_reverso` | 5 |
| Recursos | `n_recursos`, `n_recalendarizaciones`, `cambios_solapados` | 3 |
| TF-IDF texto | 300 n-gramas sobre texto combinado | 300 |
| Categóricas | `tipo_de_cambio_cod`, `riesgo_cod`, `categoria_cod` | 3 |

**Total: ~323 features**

### 4. Modelo recomendado

**XGBoost balanceado** con los parámetros del MVP 1:
- `max_depth=1`, `min_child_weight=500`, `n_estimators=40`
- `scale_pos_weight ≈84` (calculado dinámicamente)
- **ROC-AUC test: 0.8781 | Gini: 0.7562 | KS: 0.7206 | Gap ROC: 0.0153 (✓)**

### 5. Hallazgos clave

| Hallazgo | Implicación |
|---|---|
| Top decil concentra 77.8% del recall | El modelo es útil incluso revisando solo el top 10% |
| Mediana cambio→incidente: 10.8 horas | La señal es temporalmente cercana y válida |
| `business_justification` con lift alto | Monitorear si se llena antes o después del evento |
| Historial CI = familia más predictiva | Los activos problemáticos tienden a reeincidir |
| D2 (CI+7d) infla 3-5x | El target correcto es el link directo, no la proximidad temporal |

### 6. Próximos pasos (MVP 2)

- **Calibración Platt/Isotonic**: las probabilidades del modelo de boosting no están calibradas; para uso operativo se necesita calibración explícita
- **Ampliar cobertura de labels**: identificar los 86 CIs en común para recuperar posibles pares no documentados
- **Embeddings semánticos**: reemplazar TF-IDF por embeddings de texto para capturar sinónimos y contexto
- **Jerarquía de CIs**: usar la relación padre/hijo de CIs para heredar el historial
- **Pilot CAB real**: integrar el modelo en el flujo de revisión del CAB para validar en producción

In [ ]:
# ============================================================
# SECCION 14: RESUMEN EJECUTIVO Y SNAPSHOT DE DATOS
# ============================================================

print('=' * 70)
print('RESUMEN EJECUTIVO DEL EDA')
print('=' * 70)
print()
print(f'Datos originales:')
print(f'  Incidentes:  {len(df_incidentes):8,} filas (crudo) -> {len(incidentes_dedup):,} unicos')
print(f'  Cambios:     {len(df_cambios):8,} filas (crudo) -> {len(cambios_dedup):,} unicos')
print()
print(f'Variable objetivo (causo_incidente):')
print(f'  Positivos: {N_D1:,}  ({100*PREV_D1:.3f}%)')
print(f'  Negativos: {N_TOT-N_D1:,}  ({100*(1-PREV_D1):.3f}%)')
print(f'  Desbalance: 1:{int((N_TOT-N_D1)/max(N_D1,1))}')
print()
print(f'Leakage:')
print(f'  Variables excluidas: {len(FUGA_CONFIRMADAS)}')
print(f'  {FUGA_CONFIRMADAS}')
print()
print(f'Features MVP 1:')
for fn in features_base + historico_features:
    if fn in feat.columns:
        non_null = feat[fn].notna().mean() * 100
        print(f'  {fn:40s}  cobertura={non_null:.1f}%')
print(f'  + TF-IDF texto combinado (300 features)')
print(f'  + Categoricas codificadas (3 features)')
print()
print(f'Estado actual del feat DataFrame:')
print(f'  Filas: {len(feat):,}')
print(f'  Columnas totales: {len(feat.columns)}')
print(f'  Positivos: {int(feat["causo_incidente"].sum()):,}')
print()
print('=' * 70)
print('LISTO PARA PASAR AL PIPELINE DE MODELADO')
print('  -> Archivo de features: feat DataFrame (variable Python)')
print('  -> Target: feat["causo_incidente"]')
print('  -> Features: dense_cols (lista Python)')
print('  -> Texto: feat["texto_mvp1"] para TF-IDF')
print('=' * 70)


In [ ]:
# ============================================================
# EXPORT OPCIONAL A CSV PARA USO EXTERNO
# Descomenta para guardar el dataset de features final
# ============================================================

OUTPUT_PATH = Path('incidentes_cambios_features_eda.csv')

# Columnas a exportar: solo features + target + ID
cols_export = ['cambio_n', 'causo_incidente', 'texto_mvp1'] + dense_cols
cols_export = [c for c in cols_export if c in feat.columns]

# feat[cols_export].to_csv(OUTPUT_PATH, index=False)
print(f'Para exportar descomenta la linea anterior.')
print(f'Archivo de salida: {OUTPUT_PATH.resolve()}')
print(f'Columnas a exportar: {len(cols_export)}')
print(f'  {cols_export}')


---
## Sección 15 — Split temporal y construcción de la matriz de features

### Por qué split temporal y no aleatorio

En problemas de predicción sobre series temporales, el split aleatorio produce **data leakage temporal**: el modelo aprende del futuro para predecir el pasado. Usamos split temporal estricto:

**v2 — split en tres bloques temporales:**

- **Train (64%)**: para entrenar cada candidato
- **Validación (16%)**: para elegir hiperparámetros, umbral de decisión y calibrador
- **Test (20%)**: se toca **una sola vez**, con el modelo ya congelado

En v1 la selección de hiperparámetros y del umbral se hacía mirando el test, lo que convierte al test en un segundo set de validación y produce métricas optimistas. Con el bloque de validación intermedio, el test vuelve a simular producción de forma honesta.

### Construcción de la matriz X

La matriz de features combina tres bloques:
1. **Dense features** (numéricas + categóricas codificadas): `~20 features`
2. **TF-IDF** sobre texto combinado (`descripcion` + `plan_implementacion` + `plan_reverso` + `business_justification`): `300 features`
3. **Total: ~323 features**

### Por qué TF-IDF y no embeddings

El TF-IDF en (1,2)-gramas captura vocabulario de riesgo semántico sin necesidad de GPU ni API. Para MVP 1 es suficiente. Los embeddings quedan para MVP 2.

In [ ]:
# ============================================================
# SECCION 15: MATRIZ DE FEATURES Y SPLIT TEMPORAL
# v2: split TRAIN (64%) / VALIDACION (16%) / TEST (20%).
#   - hiperparametros, umbral y calibracion se deciden en VAL
#   - el TEST se evalua una unica vez al final
#   - TF-IDF con stopwords en espanol (definidas en la Seccion 7B)
# ============================================================
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# --- Asegurar que feat tiene el target ---
if 'causo_incidente' not in feat.columns:
    if 'cambio_n' not in feat.columns:
        feat['cambio_n'] = norm_id(feat['cambio'])
    feat['causo_incidente'] = feat['cambio_n'].isin(cambios_causantes_d1).astype(int)

# --- Columnas categoricas a codificar ---
# Nota v2: codificacion ordinal, valida para arboles. El encoder ve todo el
# dataset pero solo aprende NOMBRES de categorias (no el target): el riesgo de
# fuga es despreciable. Para modelos lineales seria preferible one-hot.
CAT_COLS = ['tipo_de_cambio', 'riesgo', 'categoria']
cat_available = [c for c in CAT_COLS if c in feat.columns]

for c in cat_available:
    le = LabelEncoder()
    feat[c + '_cod'] = le.fit_transform(feat[c].astype(str))

cat_coded = [c + '_cod' for c in cat_available]

# --- Lista final de features densas ---
base_dense = [f for f in dense_cols if f in feat.columns]
all_dense  = base_dense + cat_coded
all_dense  = list(dict.fromkeys(all_dense))  # dedup manteniendo orden

print(f'Features densas: {len(all_dense)}')
print(f'  {all_dense}')

# --- Ordenar por fecha para split temporal ---
if fecha_cam_col in feat.columns:
    feat_sorted = feat.sort_values(fecha_cam_col, na_position='last').reset_index(drop=True)
else:
    feat_sorted = feat.reset_index(drop=True)

# --- Split temporal 64/16/20 ---
N = len(feat_sorted)
y = feat_sorted['causo_incidente'].values

# v3: la particion sigue siendo CRONOLOGICA (bloques contiguos, sin barajar),
# pero los puntos de corte se buscan en una ventana estrecha alrededor de
# 64/80 para que la prevalencia de cada bloque quede lo mas pareja posible
# (regla del 3%, ver Seccion 15C). Nunca se estratifica barajando: eso
# destruiria la naturaleza temporal del problema.
BALANCEAR_PREVALENCIA = True
cut_tr, cut = int(N * 0.64), int(N * 0.80)
if BALANCEAR_PREVALENCIA and y.sum() >= 10:
    prev_g = y.mean()
    mejor_dev = float('inf')
    for ftr in np.arange(0.60, 0.685, 0.005):
        for fva in np.arange(0.14, 0.185, 0.005):
            c1, c2 = int(N * ftr), int(N * (ftr + fva))
            if c2 > int(N * 0.85):
                continue  # el test debe conservar al menos 15%
            p = [y[:c1].mean(), y[c1:c2].mean(), y[c2:].mean()]
            if min(p) == 0:
                continue
            dev = max(abs(pi - prev_g) / prev_g for pi in p)
            if dev < mejor_dev:
                mejor_dev, cut_tr, cut = dev, c1, c2
    print(f'Cortes ajustados por prevalencia: train hasta {cut_tr} ({cut_tr/N:.1%}), '
          f'val hasta {cut} ({cut/N:.1%})')
    print(f'  Desviacion relativa maxima de prevalencia entre bloques: {mejor_dev:.1%}')

X_dense = feat_sorted[all_dense].fillna(0).astype(float).values
XDTR, XDVA, XDTE = X_dense[:cut_tr], X_dense[cut_tr:cut], X_dense[cut:]
YTR,  YVA,  YTE  = y[:cut_tr],       y[cut_tr:cut],       y[cut:]

# TF-IDF block (fit SOLO en train)
texto_col = 'texto_mvp1' if 'texto_mvp1' in feat_sorted.columns else None
if texto_col:
    corpus = feat_sorted[texto_col].fillna('').values
    _sw = sorted(STOPWORDS_ES) if 'STOPWORDS_ES' in dir() else None
    tfidf = TfidfVectorizer(max_features=300, ngram_range=(1, 2),
                            min_df=3, sublinear_tf=True, stop_words=_sw)
    tfidf.fit(corpus[:cut_tr])
    TTFIDF_TR = tfidf.transform(corpus[:cut_tr])
    TTFIDF_VA = tfidf.transform(corpus[cut_tr:cut])
    TTFIDF_TE = tfidf.transform(corpus[cut:])
    XTR_FULL = hstack([csr_matrix(XDTR), TTFIDF_TR]).toarray()
    XVA_FULL = hstack([csr_matrix(XDVA), TTFIDF_VA]).toarray()
    XTE_FULL = hstack([csr_matrix(XDTE), TTFIDF_TE]).toarray()
    print(f'TF-IDF: {TTFIDF_TR.shape[1]} features (stopwords ES: {"si" if _sw else "no"})')
else:
    XTR_FULL, XVA_FULL, XTE_FULL = XDTR.copy(), XDVA.copy(), XDTE.copy()
    TTFIDF_TR = TTFIDF_VA = TTFIDF_TE = None
    print('texto_mvp1 no disponible, usando solo dense features')

print(f'\nSplit temporal 64/16/20:')
print(f'  Train: {len(YTR):,} cambios  |  positivos: {YTR.sum():,} ({100*YTR.mean():.2f}%)')
print(f'  Val  : {len(YVA):,} cambios  |  positivos: {YVA.sum():,} ({100*YVA.mean():.2f}%)')
print(f'  Test : {len(YTE):,} cambios  |  positivos: {YTE.sum():,} ({100*YTE.mean():.2f}%)')
print(f'  X shape train: {XTR_FULL.shape}  |  val: {XVA_FULL.shape}  |  test: {XTE_FULL.shape}')

if fecha_cam_col in feat_sorted.columns:
    fechas_ord = pd.to_datetime(feat_sorted[fecha_cam_col], errors='coerce')
    print(f'  Train: {fechas_ord.iloc[0].date()} — {fechas_ord.iloc[cut_tr-1].date()}')
    print(f'  Val  : {fechas_ord.iloc[cut_tr].date()} — {fechas_ord.iloc[cut-1].date()}')
    print(f'  Test : {fechas_ord.iloc[cut].date()} — {fechas_ord.iloc[-1].date()}')

# SPW para XGBoost (calculado en train)
SPW = int((len(YTR) - YTR.sum()) / max(YTR.sum(), 1))
print(f'\nscale_pos_weight (train): {SPW}')

if YVA.sum() == 0:
    print('AVISO: la validacion no tiene positivos; la seleccion de hiperparametros')
    print('        y umbral no sera fiable. Considera mover los cortes del split.')


---
## Sección 15B (v2) — Métrica de optimización: cuál, y por qué

Con una prevalencia de ~1%, la elección de métrica **es una decisión de diseño, no un detalle**:

| Métrica | Rol en el MVP | Por qué |
|---|---|---|
| **PR-AUC** (average precision) | ⭐ **Selección de modelo** | Mide la calidad del ranking sobre los positivos. Con 99:1, el ROC-AUC premia ordenar bien los negativos (fácil); el PR-AUC castiga cada falsa alarma que el CAB tendría que revisar. Es la métrica que se degrada primero cuando el modelo empeora |
| **Recall@10%** (top decil) | ⭐ **Métrica de negocio** | El CAB tiene capacidad de revisión limitada. La pregunta operativa real es: *"si solo puedo revisar a fondo el 10% más riesgoso, ¿qué fracción de los incidentes atrapo?"*. Es la cifra que se reporta a gerencia |
| ROC-AUC / Gini | Secundaria + anti-overfit | Se mantiene por comparabilidad (estándar bancario) y para el filtro de gap train-val < 3pp. **No** se usa sola para elegir: un modelo puede tener ROC 0.88 y ser inservible si su precisión en el top es nula |
| KS | Informativa | Estándar en riesgo crediticio; útil para comunicar, sensible al mismo sesgo que ROC en desbalance extremo |
| Brier / calibración | Complementaria (21B) | Necesaria si el score se comunica como probabilidad ("este cambio tiene 40% de riesgo") |
| Accuracy | ❌ Nunca | Prediciendo "ningún cambio causa incidente" se obtiene ~99% de accuracy |

**Decisión v2:**
- Hiperparámetros → mayor **PR-AUC en validación** (filtro: gap ROC train-val < 3pp)
- Umbral de decisión → EQopt sobre **validación** (o presupuesto fijo del CAB: top-k)
- Reporte a negocio → **recall@10%** y tabla de deciles sobre **test**


---
## Sección 15C (v3) — Validación del split: la regla del 3% y similitud de poblaciones

Un split temporal solo es utilizable si los tres bloques son **comparables**. Tres verificaciones formales:

1. **Regla del 3% sobre la prevalencia**: la diferencia de tasa de positivos entre cualquier par de bloques no debe exceder **3 puntos porcentuales** — y como aquí la prevalencia es ~1%, se aplica también la versión exigente: la desviación **relativa** de cada bloque frente a la prevalencia global no debería superar ~30%. Si falla, las métricas de validación no anticipan las de test (los IC de Wilson dicen si la diferencia es siquiera distinguible del ruido).
2. **PSI train↔val y train↔test** sobre las features: si la población cambió entre bloques (deriva), la selección de hiperparámetros en validación no transfiere al test.
3. **Test KS** (Kolmogorov-Smirnov) sobre las features numéricas clave: contraste formal de igualdad de distribuciones entre train y test.

**Por qué NO estratificamos barajando:** con datos temporales, estratificar mezclaría futuro y pasado (leakage temporal). El compromiso correcto es el de la Sección 15: mover los **puntos de corte** dentro de una ventana estrecha para equilibrar prevalencias, manteniendo bloques cronológicos contiguos.


In [ ]:
# ============================================================
# SECCION 15C (v3): VALIDACION FORMAL DEL SPLIT
# ============================================================
from scipy.stats import ks_2samp

def _wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    den = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / den
    w = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return max(0.0, c - w), min(1.0, c + w)

bloques = {'train': YTR, 'val': YVA, 'test': YTE}
prev_global = y.mean()

print('=== 1. REGLA DEL 3%: PREVALENCIA POR BLOQUE ===')
print(f'{"Bloque":>6} {"n":>7} {"pos":>5} {"prev%":>7} {"IC95 Wilson":>18} {"dev.rel":>8}')
prevs = {}
for nom, yb in bloques.items():
    p = yb.mean()
    lo, hi = _wilson(int(yb.sum()), len(yb))
    dev = abs(p - prev_global) / max(prev_global, 1e-9)
    prevs[nom] = p
    print(f'{nom:>6} {len(yb):>7,} {int(yb.sum()):>5} {p*100:>7.3f} '
          f'[{lo*100:6.3f}, {hi*100:6.3f}] {dev:>7.1%}')

diff_pp_max = max(abs(prevs[a] - prevs[b]) * 100
                  for a in prevs for b in prevs if a < b)
dev_rel_max = max(abs(p - prev_global) / max(prev_global, 1e-9) for p in prevs.values())

ok_3pp = diff_pp_max <= 3.0
ok_rel = dev_rel_max <= 0.30
print(f'\n  Diferencia maxima entre bloques : {diff_pp_max:.3f} pp  '
      f'[{"OK <= 3pp" if ok_3pp else "FALLA la regla del 3%"}]')
print(f'  Desviacion relativa maxima      : {dev_rel_max:.1%}  '
      f'[{"OK <= 30%" if ok_rel else "ALERTA: bloques poco comparables"}]')
if not ok_rel:
    print('  -> Acciones: revisar la Seccion 9B (deriva del label), ampliar la')
    print('     ventana de busqueda de cortes en la Seccion 15, o reportar las')
    print('     metricas de test con IC bootstrap mas anchos (Seccion 22B).')

# === 2. PSI train<->val y train<->test sobre features densas ===
def _psi_v3(esperado, observado, bins=10):
    esperado = np.asarray(esperado, dtype=float)
    observado = np.asarray(observado, dtype=float)
    qs = np.unique(np.quantile(esperado, np.linspace(0, 1, bins + 1)))
    if len(qs) < 3:
        return 0.0
    qs[0], qs[-1] = -np.inf, np.inf
    e, _ = np.histogram(esperado, bins=qs)
    o, _ = np.histogram(observado, bins=qs)
    e = np.clip(e / max(e.sum(), 1), 1e-6, None)
    o = np.clip(o / max(o.sum(), 1), 1e-6, None)
    return float(np.sum((o - e) * np.log(o / e)))

print('\n=== 2. PSI DE FEATURES ENTRE BLOQUES (deriva de poblacion) ===')
X_df = feat_sorted[all_dense].fillna(0).astype(float)
rows_psi3 = []
for c in all_dense:
    p_va_ = _psi_v3(X_df[c].iloc[:cut_tr], X_df[c].iloc[cut_tr:cut])
    p_te_ = _psi_v3(X_df[c].iloc[:cut_tr], X_df[c].iloc[cut:])
    rows_psi3.append(dict(feature=c, psi_val=round(p_va_, 3), psi_test=round(p_te_, 3)))
df_psi3 = pd.DataFrame(rows_psi3).sort_values('psi_test', ascending=False)
print('Top 8 features con mayor deriva train->test:')
print(df_psi3.head(8).to_string(index=False))
n_drift = int((df_psi3['psi_test'] >= 0.25).sum())
print(f'\nFeatures con PSI(train->test) >= 0.25: {n_drift} de {len(df_psi3)}')
if n_drift:
    print('  -> Estas features cambian de distribucion entre entrenamiento y test.')
    print('     Ojo con las de historial: crecen mecanicamente con el tiempo (el')
    print('     pasado acumulado del test siempre es mayor). No es un error, pero')
    print('     el modelo debe apoyarse en TASAS mas que en CONTEOS absolutos.')

# === 3. KS de las features clave train vs test ===
print('\n=== 3. TEST KS TRAIN vs TEST (features top por importancia/IV) ===')
_feats_ks = [c for c in ['hist_tasa_fallo_ci', 'dur_prog_horas', 'antelacion_dias',
                         'len_plan_implementacion', 'dias_desde_ultimo_cambio_ci']
             if c in X_df.columns]
for c in _feats_ks:
    st, pv = ks_2samp(X_df[c].iloc[:cut_tr], X_df[c].iloc[cut:])
    print(f'  {c:35s}  KS={st:.3f}  p={pv:.2e}  '
          f'[{"distribuciones distintas" if pv < 0.01 else "compatibles"}]')
print('\nNota: con n grande, KS declara "distintas" diferencias minusculas;')
print('usar el PSI como magnitud practica y el KS como confirmacion formal.')

VEREDICTO_SPLIT = ok_3pp and ok_rel and n_drift <= max(2, int(0.15 * len(df_psi3)))
print(f'\nVEREDICTO DEL SPLIT: {"APTO para seleccionar y evaluar" if VEREDICTO_SPLIT else "REVISAR antes de confiar en las metricas"}')


---
## Sección 16 — Cross-validation temporal (TimeSeriesSplit)

### Por qué CV temporal y no K-Fold estándar

K-Fold aleatorio mezcla futuro y pasado en los folds. TimeSeriesSplit respeta la cronología: cada fold entrena con datos más antiguos y valida en datos más nuevos, simulando el uso real del modelo.

```
Fold 1: [---TRAIN---] [VAL]
Fold 2: [---TRAIN------] [VAL]
Fold 3: [---TRAIN---------] [VAL]
Fold 4: [---TRAIN------------] [VAL]
```

### Qué buscamos

- **Estabilidad**: que las métricas no fluctuîn mucho entre folds (alta varianza = modelo inestable)
- **Degradación temporal**: si el último fold es significativamente peor, hay concept drift
- **ROC-AUC vs PR-AUC**: en datasets muy desbalanceados el PR-AUC es más informativo

In [ ]:
# ============================================================
# SECCION 16: CROSS-VALIDATION TEMPORAL (TimeSeriesSplit)
# ============================================================
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

N_FOLDS = 4
tscv = TimeSeriesSplit(n_splits=N_FOLDS)

# Usamos XTR_FULL y YTR (solo el bloque de train para CV)
# Esto evita que el fold de validacion toque el test set
X_cv = XTR_FULL
y_cv = YTR

resultados_cv = {'LR': [], 'XGB': []}

print(f'TimeSeriesSplit {N_FOLDS} folds sobre el set de entrenamiento ({len(y_cv):,} registros)\n')
print(f'{"Fold":>5} {"Modelo":>8} {"TrROC":>7} {"ValROC":>7} {"ValPR":>7} {"ValPos":>7} {"Gap":>7}')
print('-' * 58)

for fold_i, (tr_idx, va_idx) in enumerate(tscv.split(X_cv)):
    Xf_tr, Xf_va = X_cv[tr_idx], X_cv[va_idx]
    yf_tr, yf_va = y_cv[tr_idx], y_cv[va_idx]

    if yf_tr.sum() < 2 or yf_va.sum() < 1:
        print(f'Fold {fold_i+1}: sin positivos suficientes, saltando')
        continue

    spw_fold = int((len(yf_tr)-yf_tr.sum()) / max(yf_tr.sum(),1))

    # --- Logistic Regression ---
    lr_cv = LogisticRegression(C=0.3, penalty='l1', solver='liblinear',
                               class_weight='balanced', max_iter=500, random_state=42)
    lr_cv.fit(Xf_tr, yf_tr)
    pr_lr_tr = roc_auc_score(yf_tr, lr_cv.predict_proba(Xf_tr)[:,1])
    pr_lr_va = roc_auc_score(yf_va, lr_cv.predict_proba(Xf_va)[:,1])
    pr_lr_pr = average_precision_score(yf_va, lr_cv.predict_proba(Xf_va)[:,1])
    resultados_cv['LR'].append({'fold':fold_i+1,'roc_tr':pr_lr_tr,'roc_va':pr_lr_va,'pr_va':pr_lr_pr})
    print(f'{fold_i+1:>5} {"LR":>8} {pr_lr_tr:.4f} {pr_lr_va:.4f} {pr_lr_pr:.4f} '
          f'{yf_va.sum():>7} {pr_lr_tr-pr_lr_va:>+7.4f}')

    # --- XGBoost (config final) ---
    xgb_cv = xgb.XGBClassifier(
        max_depth=1, min_child_weight=500, n_estimators=40,
        subsample=0.5, colsample_bytree=0.5, learning_rate=0.03,
        gamma=0.5, reg_alpha=1.0, reg_lambda=6.0,
        scale_pos_weight=spw_fold,
        eval_metric='logloss',
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_cv.fit(Xf_tr, yf_tr)
    pr_xgb_tr = roc_auc_score(yf_tr, xgb_cv.predict_proba(Xf_tr)[:,1])
    pr_xgb_va = roc_auc_score(yf_va, xgb_cv.predict_proba(Xf_va)[:,1])
    pr_xgb_pr = average_precision_score(yf_va, xgb_cv.predict_proba(Xf_va)[:,1])
    resultados_cv['XGB'].append({'fold':fold_i+1,'roc_tr':pr_xgb_tr,'roc_va':pr_xgb_va,'pr_va':pr_xgb_pr})
    print(f'{fold_i+1:>5} {"XGB":>8} {pr_xgb_tr:.4f} {pr_xgb_va:.4f} {pr_xgb_pr:.4f} '
          f'{yf_va.sum():>7} {pr_xgb_tr-pr_xgb_va:>+7.4f}')
    print()

# Resumen de CV
print('\nRESUMEN CROSS-VALIDATION:')
for nombre, folds in resultados_cv.items():
    if not folds: continue
    df_cv = pd.DataFrame(folds)
    gap_medio = (df_cv['roc_tr'] - df_cv['roc_va']).mean()
    print(f'  {nombre:4s}  ROC-val media={df_cv["roc_va"].mean():.4f} '
          f'std={df_cv["roc_va"].std():.4f}  '
          f'PR-AUC media={df_cv["pr_va"].mean():.4f}  '
          f'gap_tr_val={gap_medio:+.4f}')

# Visualizacion
if resultados_cv['LR'] and resultados_cv['XGB']:
    df_lr_cv  = pd.DataFrame(resultados_cv['LR'])
    df_xgb_cv = pd.DataFrame(resultados_cv['XGB'])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, metrica, titulo in [
        (axes[0], 'roc_va',  'ROC-AUC por fold (validacion)'),
        (axes[1], 'pr_va',   'PR-AUC por fold (validacion)'),
    ]:
        ax.plot(df_lr_cv['fold'],  df_lr_cv[metrica],  'o-b', lw=2, ms=7, label='LR (baseline)')
        ax.plot(df_xgb_cv['fold'], df_xgb_cv[metrica], 's-r', lw=2, ms=7, label='XGB final')
        ax.set_xlabel('Fold (cronologico)'); ax.set_ylabel(titulo.split()[0])
        ax.set_title(titulo, fontweight='bold')
        ax.legend(); ax.grid(alpha=0.3)
        ax.set_xticks(df_lr_cv['fold'])

    plt.suptitle('Cross-validation temporal — estabilidad por fold', fontweight='bold')
    plt.tight_layout(); plt.show()

print('\nInterpretacion:')
print('  - Si ROC-AUC es estable entre folds -> modelo robusto a cambios temporales')
print('  - Si el ultimo fold cae mucho -> posible concept drift (cambia la naturaleza de los cambios)')
print('  - Gap train-val > 0.05 en CV es senal de overfitting')


### ✔ Conclusiones Secciones 15-16: split y CV temporal

* **El split temporal es correcto**: no mezcla futuro en el entrenamiento. El test set simula deployment real
* **La CV temporal confirma estabilidad**: la varianza entre folds es baja en ambos modelos, lo que indica que el modelo no está sobreajustado a un período particular
* **XGB supera consistentemente a LR** en todos los folds: la ganancia no es aleatoria
* **El último fold no cae significativamente**: no hay evidencia de concept drift fuerte en el período analizado
* **El PR-AUC es bajo en todos los folds**: esto es normal con 1% de prevalencia; la métrica importante es el lift en el top decil, no el PR-AUC absoluto

---
## Sección 17 — Entrenamiento de modelos: viaje de iteraciones

No llegamos directamente al modelo final. Recorrimos un camino de hipotesis, pruebas y aprendizajes.

### Iteración 1 — LR baseline
Empezamos con Regresión Logística L1 como punto de referencia. L1 es interpretable, rápida y fuerza la selección de features. C=0.3 introduce regularización moderada. **Sin esta baseline no podríamos saber si XGBoost realmente aporta algo**.

### Iteración 2 — XGBoost con parámetros por defecto
Entrenamos XGBoost sin ajustar. Resultado: ROC-AUC train=0.99, test=0.88, **gap=0.11 → overfitting claro**. El modelo memorizó el train en vez de aprender patrones generalizables.

### Iteración 3 — Búsqueda de hiperparámetros para controlar overfitting
Buscamos configuraciones que:
- Limiten la complejidad (`max_depth`, `min_child_weight`)
- Añadan ruido («regularización estocástica» via `subsample`, `colsample_bytree`)
- Penalicen la complejidad (`gamma`, `reg_alpha`, `reg_lambda`)

**Cambio metodológico v2:** en v1 el "ganador" del grid se elegía mirando el **test** (contaminación del test set: la métrica final queda optimista). Ahora la selección se hace sobre el bloque de **validación**, con dos criterios:
- Filtro anti-overfit: **gap ROC train-validación < 3pp**
- Entre los válidos gana el de mayor **PR-AUC en validación** (ver Sección 15B), desempate por ROC-AUC

### Iteración 4 — Modelo final
El modelo final se entrena en train, se elige en validación y se evalúa **una sola vez** en test. Las cifras de v1 (gap=0.0153, ROC=0.8781) se citaban con el protocolo antiguo: al re-ejecutar con el protocolo v2 los valores pueden variar ligeramente — y serán más honestos.

In [ ]:
# ============================================================
# SECCION 17: ENTRENAMIENTO DE MODELOS
# v2 — cambios metodologicos:
#   1. El grid de hiperparametros se evalua en VALIDACION (en v1 se
#      elegia el ganador mirando el test -> estimacion contaminada).
#   2. Metrica de seleccion: PR-AUC en validacion (ver Seccion 15B),
#      manteniendo el filtro anti-overfit gap ROC train-val < 3pp.
#   3. El test se evalua UNA sola vez con el modelo ya elegido.
# ============================================================
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

def recall_top_k(y_true, proba, k_pct=0.10):
    # Recall capturado revisando solo el top k% de cambios por score
    n_top = max(1, int(len(proba) * k_pct))
    idx = np.argsort(-proba)[:n_top]
    return float(y_true[idx].sum() / max(y_true.sum(), 1))

def metricas(modelo, Xtr, ytr, Xte, yte, nombre):
    proba_tr = modelo.predict_proba(Xtr)[:, 1]
    proba_te = modelo.predict_proba(Xte)[:, 1]
    roc_tr = roc_auc_score(ytr, proba_tr)
    roc_te = roc_auc_score(yte, proba_te)
    pr_tr  = average_precision_score(ytr, proba_tr)
    pr_te  = average_precision_score(yte, proba_te)
    gap    = roc_tr - roc_te
    valido = 'VALIDO <3pp' if gap < 0.03 else 'OVERFIT >3pp'
    print(f'{nombre:50s}  ROC-tr={roc_tr:.4f}  ROC-te={roc_te:.4f}  '
          f'gap={gap:+.4f}  PR-te={pr_te:.4f}  [{valido}]')
    return dict(nombre=nombre, roc_tr=roc_tr, roc_te=roc_te, pr_tr=pr_tr, pr_te=pr_te,
                gap=gap, proba_te=proba_te, proba_tr=proba_tr)

resultados_modelos = []

# ------------------------------------------------------------------
# MODELO 1: Logistic Regression L1 — BASELINE
# ------------------------------------------------------------------
print('=== ITERACION 1: LR BASELINE ===')
LR_BEST = LogisticRegression(C=0.3, penalty='l1', solver='liblinear',
                             class_weight='balanced', max_iter=500, random_state=42)
LR_BEST.fit(XTR_FULL, YTR)
res_lr = metricas(LR_BEST, XTR_FULL, YTR, XTE_FULL, YTE, 'LR L1 C=0.3 balanced')
resultados_modelos.append(res_lr)
print(f'  n_features != 0: {(LR_BEST.coef_[0] != 0).sum()} (L1 seleccion automatica)')

# ------------------------------------------------------------------
# MODELO 2: XGBoost con parametros por defecto — DEMO DE OVERFITTING
# (config fija, no participa en ninguna seleccion)
# ------------------------------------------------------------------
print('\n=== ITERACION 2: XGB POR DEFECTO (OVERFIT) ===')
XGB_OVERFIT = xgb.XGBClassifier(
    max_depth=6, n_estimators=200, learning_rate=0.1,
    scale_pos_weight=SPW, eval_metric='logloss',
    random_state=42, n_jobs=-1, verbosity=0)
XGB_OVERFIT.fit(XTR_FULL, YTR)
res_overfit = metricas(XGB_OVERFIT, XTR_FULL, YTR, XTE_FULL, YTE,
                       'XGB md=6 ne=200 lr=0.1 (OVERFIT)')
resultados_modelos.append(res_overfit)
print(f'  -> Gap ROC={res_overfit["gap"]:.4f}: el modelo memoriza el train set')

# ------------------------------------------------------------------
# ITERACION 3 (v2): BUSQUEDA DE HIPERPARAMETROS EN VALIDACION
# ------------------------------------------------------------------
print('\n=== ITERACION 3 (v2): BUSQUEDA EN VALIDACION ===')
print('Filtro: gap ROC train-val < 3pp. Entre validos gana el mayor PR-AUC val.')
print(f'  {"Config":52s}  {"ROC_tr":>7} {"ROC_va":>7} {"gap":>8} {"PR_va":>7} {"R@10va":>7}')
print('  ' + '-' * 95)

grid_configs = [
    dict(max_depth=1, min_child_weight=100, n_estimators=100, subsample=0.8, learning_rate=0.05),
    dict(max_depth=1, min_child_weight=200, n_estimators=80,  subsample=0.7, learning_rate=0.05),
    dict(max_depth=1, min_child_weight=500, n_estimators=40,  subsample=0.5, learning_rate=0.03),
    dict(max_depth=2, min_child_weight=300, n_estimators=60,  subsample=0.6, learning_rate=0.05),
    dict(max_depth=2, min_child_weight=500, n_estimators=50,  subsample=0.5, learning_rate=0.03),
    dict(max_depth=3, min_child_weight=500, n_estimators=40,  subsample=0.5, learning_rate=0.03),
]

FIXED_PARAMS = dict(colsample_bytree=0.5, gamma=0.5, reg_alpha=1.0, reg_lambda=6.0,
                    eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0)

best_cfg, best_pr_va, best_roc_va = None, -1.0, -1.0
for cfg in grid_configs:
    m = xgb.XGBClassifier(**cfg, **FIXED_PARAMS, scale_pos_weight=SPW)
    m.fit(XTR_FULL, YTR)
    p_tr = m.predict_proba(XTR_FULL)[:, 1]
    p_va = m.predict_proba(XVA_FULL)[:, 1]
    roc_tr_i = roc_auc_score(YTR, p_tr)
    if YVA.sum() == 0:
        print('  Sin positivos en validacion: no se puede seleccionar. Abortando grid.')
        break
    roc_va_i = roc_auc_score(YVA, p_va)
    pr_va_i  = average_precision_score(YVA, p_va)
    rk_va_i  = recall_top_k(YVA, p_va)
    gap_i    = roc_tr_i - roc_va_i
    valido   = gap_i < 0.03
    mejor    = valido and (pr_va_i, roc_va_i) > (best_pr_va, best_roc_va)
    tag = '<-- GANADOR' if mejor else ('' if valido else '(overfit)')
    cfg_str = (f"md={cfg['max_depth']} mcw={cfg['min_child_weight']} "
               f"ne={cfg['n_estimators']} sub={cfg['subsample']} lr={cfg['learning_rate']}")
    print(f'  {cfg_str:52s}  {roc_tr_i:.4f}  {roc_va_i:.4f}  {gap_i:+.4f}  {pr_va_i:.4f}  {rk_va_i:.3f}  {tag}')
    if mejor:
        best_pr_va, best_roc_va, best_cfg = pr_va_i, roc_va_i, cfg

if best_cfg is None:
    print('\n  Ninguna config paso el filtro anti-overfit; se usa la mas conservadora.')
    best_cfg = dict(max_depth=1, min_child_weight=500, n_estimators=40,
                    subsample=0.5, learning_rate=0.03)
print(f'\n  Config elegida (en VALIDACION): {best_cfg}')

# ------------------------------------------------------------------
# MODELO FINAL: entrenado en train, evaluado UNA vez en test
# ------------------------------------------------------------------
print('\n=== MODELO FINAL (v2): XGB CON CONFIG ELEGIDA EN VALIDACION ===')
XGB_FINAL = xgb.XGBClassifier(**best_cfg, **FIXED_PARAMS, scale_pos_weight=SPW)
XGB_FINAL.fit(XTR_FULL, YTR)
res_final = metricas(XGB_FINAL, XTR_FULL, YTR, XTE_FULL, YTE, 'XGB FINAL (seleccion en val)')
resultados_modelos.append(res_final)

# Probabilidades para las secciones siguientes
PRB_TR = res_final['proba_tr']
PRB_TE = res_final['proba_te']
PRB_VA = XGB_FINAL.predict_proba(XVA_FULL)[:, 1]  # para umbral y calibracion

ROC_TR  = res_final['roc_tr']
ROC_TE  = res_final['roc_te']
GINI_TE = 2 * ROC_TE - 1
GINI_TR = 2 * ROC_TR - 1
GAP_ROC = res_final['gap']
print(f'\n  Gini test: {GINI_TE:.4f}  |  Gini train: {GINI_TR:.4f}')
print(f'  Recall@10% test: {recall_top_k(YTE, PRB_TE):.3f}  (metrica de negocio, ver Seccion 15B)')


---
## Sección 17B (v2) — Benchmark de algoritmos

La v1 solo comparó LR vs XGBoost. Para un MVP defendible conviene descartar alternativas con evidencia, siempre con **el mismo protocolo**: entrenar en train, comparar en validación, reportar test solo como referencia.

| Modelo | Por qué probarlo |
|---|---|
| **Dummy (prior)** | Piso absoluto: cualquier modelo debe superarlo con claridad |
| **LR L1** | Baseline lineal interpretable (ya en Sección 17) |
| **Random Forest** | Ensamble por bagging: robusto, poco sensible a hiperparámetros; buen contraste contra boosting |
| **HistGradientBoosting** | Boosting nativo de sklearn (sin dependencia extra), maneja bien tamaños medianos |
| **XGBoost final** | El candidato elegido en la Sección 17 |
| **LightGBM** (opcional) | Boosting con crecimiento leaf-wise; a menudo iguala a XGBoost con menos tuning. Se omite silenciosamente si no está instalado |
| **XGB + SVD(50)** | Con `max_depth` bajo, los árboles apenas explotan 300 columnas TF-IDF dispersas; comprimir a 50 componentes densas (LSA) suele ayudar |


In [ ]:
# ============================================================
# SECCION 17B (v2): BENCHMARK DE ALGORITMOS
# Mismo protocolo para todos: fit en TRAIN, comparacion en VAL,
# test solo como referencia final.
# ============================================================
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.decomposition import TruncatedSVD

bench = {}
bench['Dummy (prior)'] = DummyClassifier(strategy='prior')
bench['LR L1 balanced'] = LogisticRegression(C=0.3, penalty='l1', solver='liblinear',
                                             class_weight='balanced', max_iter=500,
                                             random_state=42)
bench['RandomForest'] = RandomForestClassifier(
    n_estimators=300, max_depth=4, min_samples_leaf=50,
    class_weight='balanced_subsample', random_state=42, n_jobs=-1)
try:
    bench['HistGradBoost'] = HistGradientBoostingClassifier(
        max_depth=2, max_iter=150, learning_rate=0.05,
        min_samples_leaf=100, class_weight='balanced', random_state=42)
except TypeError:
    # sklearn < 1.2 no soporta class_weight en HGB
    bench['HistGradBoost'] = HistGradientBoostingClassifier(
        max_depth=2, max_iter=150, learning_rate=0.05,
        min_samples_leaf=100, random_state=42)
bench['XGB final (Sec 17)'] = xgb.XGBClassifier(**best_cfg, **FIXED_PARAMS,
                                                scale_pos_weight=SPW)
try:
    import lightgbm as lgb
    bench['LightGBM'] = lgb.LGBMClassifier(
        num_leaves=4, n_estimators=80, learning_rate=0.05,
        min_child_samples=100, subsample=0.7, colsample_bytree=0.7,
        scale_pos_weight=SPW, random_state=42, n_jobs=-1, verbosity=-1)
except ImportError:
    print('lightgbm no instalado — se omite (pip install lightgbm)')

rows_bench = []
for nombre, modelo in bench.items():
    try:
        modelo.fit(XTR_FULL, YTR)
        p_va = modelo.predict_proba(XVA_FULL)[:, 1]
        p_te = modelo.predict_proba(XTE_FULL)[:, 1]
        rows_bench.append(dict(
            modelo=nombre,
            pr_va=average_precision_score(YVA, p_va) if YVA.sum() else np.nan,
            roc_va=roc_auc_score(YVA, p_va) if YVA.sum() else np.nan,
            r10_va=recall_top_k(YVA, p_va),
            pr_te=average_precision_score(YTE, p_te) if YTE.sum() else np.nan,
            roc_te=roc_auc_score(YTE, p_te) if YTE.sum() else np.nan,
            r10_te=recall_top_k(YTE, p_te),
        ))
    except Exception as e:
        print(f'  {nombre}: error -> {e}')

# --- Variante: XGB con TF-IDF comprimido via SVD (LSA) ---
if texto_col and TTFIDF_TR is not None and TTFIDF_TR.shape[1] > 50:
    svd = TruncatedSVD(n_components=50, random_state=42)
    svd.fit(TTFIDF_TR)
    XTR_SVD = np.hstack([XDTR, svd.transform(TTFIDF_TR)])
    XVA_SVD = np.hstack([XDVA, svd.transform(TTFIDF_VA)])
    XTE_SVD = np.hstack([XDTE, svd.transform(TTFIDF_TE)])
    m_svd = xgb.XGBClassifier(**best_cfg, **FIXED_PARAMS, scale_pos_weight=SPW)
    m_svd.fit(XTR_SVD, YTR)
    p_va = m_svd.predict_proba(XVA_SVD)[:, 1]
    p_te = m_svd.predict_proba(XTE_SVD)[:, 1]
    rows_bench.append(dict(
        modelo='XGB + SVD(50) texto',
        pr_va=average_precision_score(YVA, p_va) if YVA.sum() else np.nan,
        roc_va=roc_auc_score(YVA, p_va) if YVA.sum() else np.nan,
        r10_va=recall_top_k(YVA, p_va),
        pr_te=average_precision_score(YTE, p_te) if YTE.sum() else np.nan,
        roc_te=roc_auc_score(YTE, p_te) if YTE.sum() else np.nan,
        r10_te=recall_top_k(YTE, p_te),
    ))
    print(f'SVD: varianza explicada por 50 componentes = {svd.explained_variance_ratio_.sum():.1%}')

df_bench = pd.DataFrame(rows_bench).sort_values('pr_va', ascending=False)
print('\nBENCHMARK (ordenado por PR-AUC en VALIDACION — la metrica de seleccion):')
print(df_bench.to_string(index=False, float_format='{:.4f}'.format))

# Visualizacion
fig, ax = plt.subplots(figsize=(10, max(3, 0.6 * len(df_bench))))
ypos = np.arange(len(df_bench))
ax.barh(ypos + 0.2, df_bench['pr_va'], height=0.35, color='#8e44ad',
        alpha=0.85, label='PR-AUC val (seleccion)')
ax.barh(ypos - 0.2, df_bench['pr_te'], height=0.35, color='#e67e22',
        alpha=0.85, label='PR-AUC test (referencia)')
ax.set_yticks(ypos)
ax.set_yticklabels(df_bench['modelo'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('PR-AUC (average precision)')
ax.set_title('Benchmark de algoritmos — mismo protocolo temporal', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print('\nLectura:')
print('  - Si ningun modelo supera con claridad al Dummy en PR-AUC: el problema')
print('    esta en las features o en el label, no en el algoritmo.')
print('  - Si RF o LightGBM superan al XGB elegido en VALIDACION, considerar')
print('    cambiar el campeon (la decision sigue tomandose en val, nunca en test).')
print('  - Si XGB+SVD >= XGB con TF-IDF crudo, preferir SVD: menos dimensiones,')
print('    arboles mas efectivos y pipeline mas rapido.')


---
## Sección 18 — Análisis de overfitting: cómo se detectó y cómo se corrigió

### El overfitting es la segunda trampa más común (después del leakage)

Un modelo que memoriza el set de entrenamiento tiene:
- **ROC-AUC train muy alto** (≥0.99)
- **ROC-AUC test moderado** (≈0.88)
- **Gap train-test > 0.05** (inaceptable para producción)

### Por qué XGBoost con parámetros por defecto overfitta aquí

En datasets muy desbalanceados (1:88), cada positivo es “oro” para el modelo. Con árboles profundos (`max_depth=6`) y muchos estimadores (200), el modelo aprende **reglas muy específicas** que identifican los 166 positivos del train set pero no generalizan.

### Solución: regularización multi-nivel

| Hiperparámetro | Valor final | Efecto |
|---|---|---|
| `max_depth=1` | Stumps (1 nivel) | Fuerza reglas muy simples |
| `min_child_weight=500` | Hoja mínima de 500 muestras | Prohíbe reglas hiper-específicas |
| `n_estimators=40` | Solo 40 árboles | Menos capacidad de memorizar |
| `subsample=0.5` | 50% datos por árbol | Ruido estocástico = robustez |
| `gamma=0.5` | Penalización por split | Exige que cada split valga la pena |
| `reg_alpha=1.0, reg_lambda=6.0` | L1+L2 en pesos | Shrinkage de los pesos |

In [ ]:
# ============================================================
# SECCION 18: ANALISIS DE OVERFITTING
# ============================================================

df_modelos = pd.DataFrame([
    {'Modelo': 'LR L1 C=0.3', 'ROC_tr': res_lr['roc_tr'], 'ROC_te': res_lr['roc_te'],
     'PR_te': res_lr['pr_te'], 'Gap': res_lr['gap']},
    {'Modelo': 'XGB overfit', 'ROC_tr': res_overfit['roc_tr'], 'ROC_te': res_overfit['roc_te'],
     'PR_te': res_overfit['pr_te'], 'Gap': res_overfit['gap']},
    {'Modelo': 'XGB final (ELEGIDO)', 'ROC_tr': res_final['roc_tr'], 'ROC_te': res_final['roc_te'],
     'PR_te': res_final['pr_te'], 'Gap': res_final['gap']},
])

print('Comparacion de modelos:')
print(df_modelos.to_string(index=False, float_format='{:.4f}'.format))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: ROC train vs test por modelo
ax = axes[0]
x_pos = range(len(df_modelos))
w = 0.35
bars_tr = ax.bar([x-w/2 for x in x_pos], df_modelos['ROC_tr'],
                 w, color='#3498db', alpha=0.85, label='Train')
bars_te = ax.bar([x+w/2 for x in x_pos], df_modelos['ROC_te'],
                 w, color='#27ae60', alpha=0.85, label='Test')
for bar_ in list(bars_tr) + list(bars_te):
    h = bar_.get_height()
    ax.text(bar_.get_x()+bar_.get_width()/2, h+0.002, f'{h:.3f}',
            ha='center', fontsize=7.5, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(df_modelos['Modelo'], rotation=10, ha='right', fontsize=8)
ax.set_ylabel('ROC-AUC'); ax.set_ylim(0.7, 1.05)
ax.set_title('ROC-AUC Train vs Test', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
ax.axhline(1.0, color='red', ls=':', lw=1.5, alpha=0.5)

# Panel 2: Gap train-test
ax = axes[1]
colores_gap = ['#27ae60' if g < 0.03 else '#e74c3c' for g in df_modelos['Gap']]
bars_gap = ax.bar(x_pos, df_modelos['Gap'], color=colores_gap, alpha=0.85, edgecolor='white')
for bar_, v in zip(bars_gap, df_modelos['Gap']):
    ax.text(bar_.get_x()+bar_.get_width()/2, bar_.get_height()+0.001,
            f'{v:+.4f}', ha='center', fontsize=9, fontweight='bold')
ax.axhline(0.03, color='#e74c3c', ls='--', lw=2, label='Umbral 3pp')
ax.set_xticks(x_pos)
ax.set_xticklabels(df_modelos['Modelo'], rotation=10, ha='right', fontsize=8)
ax.set_ylabel('Gap ROC-AUC (train - test)')
ax.set_title('Gap ROC: diagnostico de overfitting\n(verde=OK, rojo=overfit)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# Panel 3: Distribucion de probabilidades por clase (modelo final)
ax = axes[2]
proba_pos = PRB_TE[YTE == 1]
proba_neg = PRB_TE[YTE == 0]
bins_dist = np.linspace(0, 1, 31)
ax.hist(proba_neg, bins=bins_dist, color='#3498db', alpha=0.6, label=f'No incidente (n={len(proba_neg):,})')
ax.hist(proba_pos, bins=bins_dist, color='#e74c3c', alpha=0.85, label=f'Causo incidente (n={len(proba_pos)})')
ax.set_xlabel('Probabilidad predicha'); ax.set_ylabel('N cambios')
ax.set_title('Distribucion de probabilidades\n(test set, modelo final)', fontweight='bold')
ax.legend(fontsize=9); ax.set_xlim(-0.02, 1.02)

plt.suptitle('Analisis de overfitting: comparacion de modelos', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nLectura del panel derecho:')
print('  - Si las distribuciones se solapan mucho: mala separacion (modelo debil)')
print('  - Si los positivos (rojo) se concentran a la derecha: buena separacion (modelo util)')
print(f'  - Mediana score positivos: {np.median(proba_pos):.3f}  |  Mediana negativos: {np.median(proba_neg):.3f}')


### ✔ Conclusiones Secciones 17-18: entrenamiento y overfitting

* **El overfitting es diagnosticable y corregible**: el gap ROC de 3pp es un criterio operativo claro
* **XGB con parámetros por defecto hace trampa**: memorización que parece buen modelo en train pero no generaliza
* **La regularización multi-nivel funciona**: `max_depth=1 + min_child_weight=500 + gamma + L1+L2` bajan el gap de 0.11 a 0.015
* **El modelo final tiene buena separación**: los positivos se concentran en probabilidades altas, los negativos en probabilidades bajas
* **LR sigue siendo útil como sanity check**: si XGB no supera a LR significativamente, el problema puede ser de features, no de modelo
* **ROC-AUC test 0.8781 con gap 0.0153**: el último modelo validó el criterio y se convierte en el modelo recomendado para MVP 1

In [ ]:
# ============================================================
# SECCION 19: DASHBOARD DE PERFORMANCE
# ============================================================
from sklearn.metrics import (
    roc_curve, precision_recall_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import learning_curve, TimeSeriesSplit

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Panel 1: Curva ROC ---
ax = axes[0][0]
for proba, y_true, label, color, ls in [
    (PRB_TR, YTR, f'Train (AUC={ROC_TR:.4f})', '#3498db', '--'),
    (PRB_TE, YTE, f'Test  (AUC={ROC_TE:.4f})', '#e74c3c', '-'),
    (res_lr['proba_te'], YTE, f'LR baseline (AUC={res_lr["roc_te"]:.4f})', '#27ae60', ':'),
]:
    fpr, tpr, _ = roc_curve(y_true, proba)
    ax.plot(fpr, tpr, color=color, lw=2.5, ls=ls, label=label)
ax.plot([0,1],[0,1], 'gray', lw=1, ls='--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Curva ROC', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 2: Curva PR ---
ax = axes[0][1]
for proba, y_true, label, color, ls in [
    (PRB_TR, YTR, f'Train (AP={res_final["pr_tr"]:.4f})', '#3498db', '--'),
    (PRB_TE, YTE, f'Test  (AP={res_final["pr_te"]:.4f})', '#e74c3c', '-'),
    (res_lr['proba_te'], YTE, f'LR baseline (AP={res_lr["pr_te"]:.4f})', '#27ae60', ':'),
]:
    prec_c, rec_c, _ = precision_recall_curve(y_true, proba)
    ax.plot(rec_c, prec_c, color=color, lw=2.5, ls=ls, label=label)
ax.axhline(YTE.mean(), color='gray', ls=':', lw=1,
           label=f'Baseline {YTE.mean()*100:.2f}%')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Curva Precision-Recall', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 3: Distribucion de probabilidades ---
ax = axes[0][2]
bins_p = np.linspace(0, 1, 41)
ax.hist(PRB_TE[YTE==0], bins=bins_p, color='#3498db', alpha=0.6, label='No incidente', density=True)
ax.hist(PRB_TE[YTE==1], bins=bins_p, color='#e74c3c', alpha=0.85, label='Causo incidente', density=True)
ax.set_xlabel('Probabilidad predicha')
ax.set_ylabel('Densidad')
ax.set_title('Distribucion de proba por clase (test)', fontweight='bold')
ax.legend(fontsize=9)

# --- Panel 4: Curva KS ---
ax = axes[1][0]
scores = np.sort(np.unique(PRB_TE))
cdf_pos = np.array([(PRB_TE[YTE==1] <= t).mean() for t in scores])
cdf_neg = np.array([(PRB_TE[YTE==0] <= t).mean() for t in scores])
KS_val  = float(np.max(np.abs(cdf_pos - cdf_neg)))
KS_thr  = scores[np.argmax(np.abs(cdf_pos - cdf_neg))]
ax.plot(scores, cdf_pos, 'r-', lw=2.5, label='Positivos (CDF)')
ax.plot(scores, cdf_neg, 'b-', lw=2.5, label='Negativos (CDF)')
ax.axvline(KS_thr, color='gray', ls='--', lw=1.5,
           label=f'KS={KS_val:.4f} @ t={KS_thr:.3f}')
ax.fill_betweenx([0,1], KS_thr, KS_thr, alpha=0.1)
ax.set_xlabel('Umbral de probabilidad')
ax.set_ylabel('CDF acumulada')
ax.set_title(f'Curva KS (Kolmogorov-Smirnov)\nKS={KS_val:.4f}', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# --- Panel 5: Confusion matrix en EQopt ---
ax = axes[1][1]
def eqopt_score(p, r):
    if p+r == 0: return 0
    return 0.55*np.sqrt(p*r) + 0.25*min(p,r) + 0.20*(1-abs(p-r))

# v2: el umbral se selecciona en VALIDACION (en v1 se elegia sobre el
# propio test, lo que da una matriz de confusion optimista)
thresholds = np.linspace(0.01, 0.99, 200)
scores_eq  = []
for t in thresholds:
    yp = (PRB_VA >= t).astype(int)
    tp_ = int(((yp==1) & (YVA==1)).sum())
    fp_ = int(((yp==1) & (YVA==0)).sum())
    fn_ = int(((yp==0) & (YVA==1)).sum())
    p_ = tp_/max(tp_+fp_,1); r_ = tp_/max(tp_+fn_,1)
    scores_eq.append(eqopt_score(p_, r_))

T_EQ = thresholds[np.argmax(scores_eq)]
YP_EQ = (PRB_TE >= T_EQ).astype(int)
cm = confusion_matrix(YTE, YP_EQ)
cmd = ConfusionMatrixDisplay(cm, display_labels=['No incidente','Causo incidente'])
cmd.plot(ax=ax, colorbar=False, cmap='Blues')
TP_= cm[1,1]; FP_= cm[0,1]; FN_= cm[1,0]; TN_= cm[0,0]
PREC_ = TP_/max(TP_+FP_,1); REC_ = TP_/max(TP_+FN_,1)
ax.set_title(f'Confusion Matrix TEST @ EQopt (t={T_EQ:.3f}, elegido en VAL)\nPrec={PREC_:.2%}  Rec={REC_:.2%}',
             fontweight='bold')

# --- Panel 6: Learning curve ---
ax = axes[1][2]
train_sizes = np.linspace(0.1, 1.0, 8)
try:
    # Usamos solo dense (sin TF-IDF) para que la LC sea rapida
    lc_model = xgb.XGBClassifier(
        max_depth=1, min_child_weight=500, n_estimators=40,
        subsample=0.5, colsample_bytree=0.5, learning_rate=0.03,
        gamma=0.5, reg_alpha=1.0, reg_lambda=6.0,
        scale_pos_weight=SPW,
        eval_metric='logloss',
        random_state=42, n_jobs=-1, verbosity=0
    )
    # v2 FIX: cv=3 (KFold aleatorio) mezclaba futuro y pasado dentro de la
    # learning curve; TimeSeriesSplit mantiene la cronologia.
    ts_lc, tr_sc, va_sc = learning_curve(
        lc_model, XDTR, YTR,
        train_sizes=train_sizes, cv=TimeSeriesSplit(n_splits=3),
        scoring='roc_auc', n_jobs=-1
    )
    ax.plot(ts_lc, tr_sc.mean(axis=1), 'o-b', lw=2, ms=5, label='Train ROC-AUC')
    ax.fill_between(ts_lc, tr_sc.mean(1)-tr_sc.std(1), tr_sc.mean(1)+tr_sc.std(1), alpha=0.1, color='b')
    ax.plot(ts_lc, va_sc.mean(axis=1), 'o-r', lw=2, ms=5, label='Val ROC-AUC')
    ax.fill_between(ts_lc, va_sc.mean(1)-va_sc.std(1), va_sc.mean(1)+va_sc.std(1), alpha=0.1, color='r')
except Exception as e:
    ax.text(0.5, 0.5, f'LC error: {e}', ha='center', va='center', transform=ax.transAxes, fontsize=8)
ax.set_xlabel('N muestras de entrenamiento')
ax.set_ylabel('ROC-AUC')
ax.set_title('Curva de aprendizaje (dense features)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle(f'Dashboard de performance — Modelo XGB final\n'
             f'ROC-AUC={ROC_TE:.4f}  Gini={GINI_TE:.4f}  KS={KS_val:.4f}  Gap={GAP_ROC:.4f}',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

print(f'\nResumen de metricas del modelo final:')
print(f'  ROC-AUC  train={ROC_TR:.4f}  test={ROC_TE:.4f}  gap={GAP_ROC:+.4f}')
print(f'  Gini     train={GINI_TR:.4f}  test={GINI_TE:.4f}')
print(f'  KS       test={KS_val:.4f}  @ umbral={KS_thr:.3f}')
print(f'  PR-AUC   train={res_final["pr_tr"]:.4f}  test={res_final["pr_te"]:.4f}')
print(f'  EQopt umbral: {T_EQ:.4f}  Precision={PREC_:.2%}  Recall={REC_:.2%}')

---
## Sección 20 — Importancia de features

### Por qué analizar importancia

La importancia de features hace tres cosas:
1. **Valida el modelo**: si las features más importantes son lógicas de negocio, el modelo aprendió algo real
2. **Detecta anomalías**: si una variable de leakage (que debimos haber excluido) aparece top, hay un bug
3. **Orienta el MVP 2**: saber qué familias dominan indica dónde invertir en feature engineering

### Qué esperamos encontrar

Basado en el EDA:
- **Historial CI** (`hist_fallos_previos_ci`, `hist_tasa_fallo_ci`) deben dominar
- **TF-IDF** debería tener presencia moderada
- **Temporales** (`hora_inicio`, `dow_inicio`, `fuera_horario`) debería contribuir
- Si `business_justification` aparece muy arriba, monitorear timing de llenado

In [ ]:
# ============================================================
# SECCION 20: FEATURE IMPORTANCE
# ============================================================

# ============================================================
# SECCION 20: FEATURE IMPORTANCE
# ============================================================

fi = XGB_FINAL.feature_importances_

# Nombres de features
if texto_col:
    feat_names = all_dense + list(tfidf.get_feature_names_out())
else:
    feat_names = all_dense

# Completar hasta la longitud de fi si hay diferencia
if len(feat_names) < len(fi):
    feat_names = feat_names + [f'f_{i}' for i in range(len(feat_names), len(fi))]
feat_names = feat_names[:len(fi)]

df_fi = pd.DataFrame({'feature': feat_names, 'importance': fi})
df_fi = df_fi.sort_values('importance', ascending=False)

# Clasificar por familia — asignar a df_fi COMPLETO antes de cualquier slice
def familia_feature(nombre):
    if any(x in nombre for x in ['hist_', 'cambios_ci_']): return 'Historial CI'
    if any(x in nombre for x in ['hora', 'dow', 'fin_semana', 'horario', 'mes', 'fur']): return 'Temporal'
    if any(x in nombre for x in ['dur_', 'antelacion']): return 'Duracion'
    if any(x in nombre for x in ['len_', 'tiene_', 'texto', 'bj']): return 'Documental'
    if any(x in nombre for x in ['n_recursos', 'n_recal', 'solapados']): return 'Escala'
    if any(x in nombre for x in ['_cod', 'tipo_', 'riesgo', 'categoria']): return 'Categorica'
    return 'TF-IDF'

df_fi['familia'] = df_fi['feature'].apply(familia_feature)  # en df_fi completo

# Top 20 features: hereda 'familia' de df_fi
top_n = 20
df_top = df_fi.head(top_n).copy()

color_fam = {
    'Historial CI': '#e74c3c',
    'Temporal':     '#3498db',
    'Duracion':     '#2e86de',
    'Documental':   '#f39c12',
    'Escala':       '#9b59b6',
    'Categorica':   '#27ae60',
    'TF-IDF':       '#95a5a6',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Top 20 features individuales
ax = axes[0]
colors_fi = [color_fam.get(f, '#95a5a6') for f in df_top['familia']]
ax.barh(range(len(df_top)), df_top['importance'].values[::-1][::-1],
        color=colors_fi, alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(df_top)))
ax.set_yticklabels([n[:40] for n in df_top['feature']], fontsize=8)
ax.set_xlabel('Importancia (XGBoost gain)')
ax.set_title(f'Top {top_n} features por importancia', fontweight='bold')

patches = [mpatches.Patch(color=v, label=k, alpha=0.85) for k, v in color_fam.items()
           if k in df_top['familia'].values]
ax.legend(handles=patches, fontsize=8, loc='lower right')

# Panel 2: Importancia agregada por familia
ax = axes[1]
# df_fi ya tiene 'familia' asignada arriba, no hace falta copia
fi_fam = df_fi.groupby('familia')['importance'].sum().sort_values(ascending=False)
colors_fam2 = [color_fam.get(f, '#95a5a6') for f in fi_fam.index]
bars_fam = ax.bar(range(len(fi_fam)), fi_fam.values, color=colors_fam2, alpha=0.85, edgecolor='white')
ax.set_xticks(range(len(fi_fam)))
ax.set_xticklabels(fi_fam.index, rotation=20, ha='right', fontsize=9)
for bar_, v in zip(bars_fam, fi_fam.values):
    ax.text(bar_.get_x()+bar_.get_width()/2, bar_.get_height()+0.0005,
            f'{100*v/fi_fam.sum():.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Importancia por familia de features', fontweight='bold')
ax.set_ylabel('Importancia total (XGBoost gain)')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Importancia de features — Modelo XGB final', fontweight='bold')
plt.tight_layout(); plt.show()

print('Top 10 features:')
print(df_fi.head(10)[['feature','importance','familia']].to_string(index=False))
print(f'\nPorcentaje de importancia por familia:')
for fam, v in (fi_fam / fi_fam.sum() * 100).items():
    print(f'  {fam:20s}: {v:.1f}%')

---
## Sección 20B (v3) — Importancia por permutación (validación)

La importancia por *gain* de XGBoost tiene dos sesgos conocidos: favorece features de alta cardinalidad y se calcula sobre el **train** (mide qué usó el modelo, no qué le sirve para generalizar). La importancia por **permutación** rompe cada feature en el set de **validación** y mide cuánto cae la métrica de selección (average precision):

- Feature con gain alto pero permutación ≈ 0 → el modelo la usó para memorizar (ruido).
- Feature con permutación alta → señal que **generaliza**; si además es una feature de historial, refuerza la hipótesis central del proyecto.
- Si una feature domina la permutación de forma desproporcionada → re-auditar su timing (posible fuga residual).


In [ ]:
# ============================================================
# SECCION 20B (v3): PERMUTATION IMPORTANCE EN VALIDACION
# ============================================================
from sklearn.inspection import permutation_importance

if YVA.sum() >= 3:
    r_perm = permutation_importance(
        XGB_FINAL, XVA_FULL, YVA,
        scoring='average_precision', n_repeats=10,
        random_state=42, n_jobs=-1)

    df_perm = pd.DataFrame({
        'feature': feat_names[:XVA_FULL.shape[1]],
        'imp_media': r_perm.importances_mean,
        'imp_std': r_perm.importances_std,
    }).sort_values('imp_media', ascending=False)

    top_perm = df_perm.head(15)
    print('TOP 15 POR PERMUTACION (caida de PR-AUC en validacion):')
    print(top_perm.to_string(index=False, float_format='{:.5f}'.format))

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(range(len(top_perm)), top_perm['imp_media'].values[::-1],
            xerr=top_perm['imp_std'].values[::-1],
            color='#2c3e50', alpha=0.85, edgecolor='white', capsize=3)
    ax.set_yticks(range(len(top_perm)))
    ax.set_yticklabels(top_perm['feature'].values[::-1], fontsize=8.5)
    ax.axvline(0, color='gray', lw=1)
    ax.set_xlabel('Caida media de average precision al permutar (val)')
    ax.set_title('Importancia por permutacion — validacion', fontweight='bold')
    plt.tight_layout(); plt.show()

    # Contraste con la importancia por gain (Seccion 20)
    top_gain_set = set(df_fi.head(15)['feature'])
    top_perm_set = set(top_perm['feature'])
    solo_gain = top_gain_set - top_perm_set
    solo_perm = top_perm_set - top_gain_set
    print(f'\nCoincidencia top-15 gain vs permutacion: '
          f'{len(top_gain_set & top_perm_set)}/15')
    if solo_gain:
        print(f'  Solo en gain (posible memorizacion/ruido): {sorted(solo_gain)}')
    if solo_perm:
        print(f'  Solo en permutacion (senal que generaliza): {sorted(solo_perm)}')

    # Alerta de dominancia (fuga residual)
    if len(df_perm) > 1 and df_perm.iloc[0]['imp_media'] > 0:
        ratio_dom = df_perm.iloc[0]['imp_media'] / max(df_perm.iloc[1]['imp_media'], 1e-9)
        if ratio_dom > 5:
            print(f'\nALERTA: "{df_perm.iloc[0]["feature"]}" domina {ratio_dom:.0f}x sobre')
            print('la segunda feature. Re-auditar su timing de disponibilidad (Seccion 10).')
else:
    print('Muy pocos positivos en validacion para permutation importance')


In [ ]:
# ============================================================
# SECCION 21: THRESHOLD SWEEP + EQOPT
# v2: el umbral se ELIGE en validacion y se EVALUA en test.
# (En v1 el sweep completo se hacia sobre el test.)
#
# EQ(P,R) = 0.55*sqrt(P*R) + 0.25*min(P,R) + 0.20*(1-|P-R|)
# Nota: EQopt es una metrica ad-hoc de equilibrio precision/recall.
# La alternativa operativa mas simple es fijar el presupuesto del CAB
# (p.ej. "revisar el top 10%") y reportar recall a ese presupuesto.
# ============================================================

def sweep_umbral(proba, y_true, thresholds):
    P, R, F1, EQ, NF = [], [], [], [], []
    for t in thresholds:
        yp = (proba >= t).astype(int)
        tp_ = int(((yp == 1) & (y_true == 1)).sum())
        fp_ = int(((yp == 1) & (y_true == 0)).sum())
        fn_ = int(((yp == 0) & (y_true == 1)).sum())
        p = tp_ / max(tp_ + fp_, 1)
        r = tp_ / max(tp_ + fn_, 1)
        P.append(p); R.append(r)
        F1.append(2 * p * r / max(p + r, 1e-9))
        EQ.append(eqopt_score(p, r))
        NF.append(int(yp.sum()))
    return P, R, F1, EQ, NF

thresholds_sw = np.linspace(0.01, 0.99, 300)

# --- 1. Seleccion del umbral en VALIDACION ---
prec_va, rec_va, f1_va, eq_va, nf_va = sweep_umbral(PRB_VA, YVA, thresholds_sw)
T_EQ = float(thresholds_sw[int(np.argmax(eq_va))])
T_F1 = float(thresholds_sw[int(np.argmax(f1_va))])
print(f'Umbral EQopt (elegido en VALIDACION): {T_EQ:.3f}   EQ_va={max(eq_va):.4f}')
print(f'Umbral F1opt (validacion)           : {T_F1:.3f}   F1_va={max(f1_va):.4f}')

# --- 2. Evaluacion honesta en TEST con el umbral congelado ---
prec_te, rec_te, f1_te, eq_te, nf_te = sweep_umbral(PRB_TE, YTE, thresholds_sw)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(thresholds_sw, rec_va,  'b-',  lw=2.5, label='Recall (val)')
ax.plot(thresholds_sw, prec_va, 'r-',  lw=2.5, label='Precision (val)')
ax.plot(thresholds_sw, f1_va,   'g--', lw=2,   label='F1 (val)')
ax.plot(thresholds_sw, eq_va,   'm-',  lw=2,   label='EQopt (val)')
ax.axvline(T_EQ, color='m', ls=':', lw=2, label=f'EQopt t={T_EQ:.3f}')
ax.set_xlabel('Umbral de decision'); ax.set_ylabel('Metrica')
ax.set_title('Sweep en VALIDACION (aqui se elige el umbral)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_xlim(0, 1)

ax = axes[1]
ax2 = ax.twinx()
ax.plot(thresholds_sw, nf_te, 'k-', lw=2.5, label='N flagueados (test)')
ax2.plot(thresholds_sw, rec_te, 'b--', lw=2, label='Recall (test)')
ax.axvline(T_EQ, color='m', ls=':', lw=2, label=f'Umbral congelado t={T_EQ:.3f}')
ax.set_xlabel('Umbral'); ax.set_ylabel('N cambios flagueados (test)')
ax2.set_ylabel('Recall (test)', color='#3498db')
ax.set_title('Trade-off en TEST con el umbral ya congelado', fontweight='bold')
lines1, lab1 = ax.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, lab1 + lab2, fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('Threshold sweep v2 — seleccion en validacion, evaluacion en test', fontweight='bold')
plt.tight_layout(); plt.show()

# --- 3. Metricas operativas en TEST @ umbral elegido ---
yp_eq = (PRB_TE >= T_EQ).astype(int)
tp_eq = int(((yp_eq == 1) & (YTE == 1)).sum())
fp_eq = int(((yp_eq == 1) & (YTE == 0)).sum())
fn_eq = int(((yp_eq == 0) & (YTE == 1)).sum())
print(f'\nEn TEST @ EQopt (t={T_EQ:.3f}, elegido en validacion):')
print(f'  Flagueados: {int(yp_eq.sum())}  TP: {tp_eq}  FP: {fp_eq}  FN: {fn_eq}')
print(f'  Precision : {tp_eq/max(tp_eq+fp_eq,1):.2%}')
print(f'  Recall    : {tp_eq/max(tp_eq+fn_eq,1):.2%}')
print(f'  Por cada cambio riesgoso detectado se revisan {fp_eq/max(tp_eq,1):.1f} falsos positivos')


---
## Sección 21B (v2) — Calibración de probabilidades

La v1 dejaba la calibración para el MVP 2. Se adelanta porque es barata y porque **sin ella el score no es una probabilidad**: si el CAB lee "0.70", debe poder interpretarlo como ~70% de riesgo relativo real, no como un número interno del boosting (que con `scale_pos_weight` alto queda sistemáticamente inflado).

**Método:** los calibradores se ajustan sobre la **validación** (nunca sobre test ni train):
- **Platt (sigmoide)**: robusto con pocos positivos — recomendado aquí
- **Isotónica**: más flexible, pero con ~30 positivos en validación puede sobreajustar

**Nota:** la calibración es monótona — no cambia el ranking, el ROC-AUC ni la tabla de deciles. Solo corrige la escala del score.


In [ ]:
# ============================================================
# SECCION 21B (v2): CALIBRACION DE PROBABILIDADES
# Calibradores ajustados en VALIDACION, evaluados en TEST.
# ============================================================
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

if YVA.sum() >= 5:
    # --- Platt scaling (sigmoide sobre el score) ---
    platt = LogisticRegression(C=1e6, max_iter=1000)
    platt.fit(PRB_VA.reshape(-1, 1), YVA)
    p_platt_te = platt.predict_proba(PRB_TE.reshape(-1, 1))[:, 1]

    # --- Isotonica ---
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(PRB_VA, YVA)
    p_iso_te = iso.predict(PRB_TE)

    print('Brier score en TEST (menor = mejor):')
    print(f'  Sin calibrar : {brier_score_loss(YTE, PRB_TE):.5f}')
    print(f'  Platt        : {brier_score_loss(YTE, p_platt_te):.5f}')
    print(f'  Isotonica    : {brier_score_loss(YTE, p_iso_te):.5f}')
    print(f'  Referencia (predecir prevalencia {YTR.mean():.4f} siempre): '
          f'{brier_score_loss(YTE, np.full(len(YTE), YTR.mean())):.5f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    for proba, label, color in [(PRB_TE, 'Sin calibrar', '#e74c3c'),
                                (p_platt_te, 'Platt', '#27ae60'),
                                (p_iso_te, 'Isotonica', '#3498db')]:
        try:
            frac_pos, mean_pred = calibration_curve(YTE, proba, n_bins=5, strategy='quantile')
            ax.plot(mean_pred, frac_pos, 'o-', lw=2, ms=6, color=color, label=label)
        except Exception as e:
            print(f'  curva {label}: {e}')
    lim = max(PRB_TE.max(), 0.05)
    ax.plot([0, lim], [0, lim], 'k--', lw=1, label='Calibracion perfecta')
    ax.set_xlabel('Probabilidad media predicha (bin)')
    ax.set_ylabel('Fraccion real de positivos (bin)')
    ax.set_title('Curva de calibracion (test, bins por cuantiles)', fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.hist(PRB_TE, bins=40, alpha=0.5, color='#e74c3c', label='Sin calibrar')
    ax.hist(p_platt_te, bins=40, alpha=0.5, color='#27ae60', label='Platt')
    ax.set_yscale('log')
    ax.set_xlabel('Score'); ax.set_ylabel('N cambios (log)')
    ax.set_title('Distribucion del score antes/despues de calibrar', fontweight='bold')
    ax.legend(fontsize=9)

    plt.suptitle('Calibracion de probabilidades — ajustada en validacion', fontweight='bold')
    plt.tight_layout(); plt.show()

    CALIBRADOR_MVP1 = platt  # recomendado: Platt (robusto con pocos positivos)
    print('\nRecomendacion MVP 1: usar Platt. Con ~decenas de positivos en validacion,')
    print('la isotonica tiende a escalones sobreajustados.')
    print('La calibracion NO cambia el ranking: deciles y recall@10% quedan intactos.')
else:
    CALIBRADOR_MVP1 = None
    print('Muy pocos positivos en validacion para calibrar de forma fiable.')


### ✔ Conclusiones Secciones 19-21: performance, features y umbral

**Performance:**
* ROC-AUC=0.8781, Gini=0.7562, KS=0.7206 son métricas sólidas para un dataset con ~1% de prevalencia
* La curva de aprendizaje muestra que aún hay margen: aúdir datos ayudaría más que afinar el modelo

**Features:**
* El historial del CI domina la importancia, confirmando la hipótesis principal del EDA
* TF-IDF contribuye significativamente: hay vocabulario de riesgo captureable semánticamente
* Las variables temporales y documentales aportan en conjunto pero individualmente son débiles

**EQopt:**
* Definimos EQopt como el umbral que maximiza `0.55×√(P×R) + 0.25×min(P,R) + 0.20×(1-|P-R|)`, que pondera más el equilibrio que el F1 puro
* En este dataset EQopt = F1opt (señal de robustez: ambas métricas convergen)
* El trade-off clave: 59% recall al costo de revisar ~5 cambios adicionales por cada incidente detectado
* Umbral conservador (0.90): cero flagueados — inútil operativamente
* Umbral agresivo (0.35): revisa 2,225 cambios para atrapar 26/27 incidentes — sobrecarga el CAB

## Sección 22 — Análisis operacional: deciles, escenarios y simulación CAB

### De las métricas al negocio

Un ROC-AUC de 0.88 no le dice al gerente del CAB cuántos cambios va a revisar extra. Lo que necesita saber es:

1. **¿En qué decil se concentran los incidentes?** (tabla de deciles)
2. **¿Cuánto trabajo extra genera cada umbral?** (escenarios operativos)
3. **¿Qué significa esto en la semana real del CAB?** (simulación semanal)

### La tabla de deciles es la métrica de negocio

El modelo ordena los cambios de mayor a menor riesgo. La tabla de deciles responde: ¿en el 10% más riesgoso (top decil), qué fracción de los incidentes reales está?

**Lift de 7.79x** significa que revisar solo el top 10% captura casi 8 veces más incidentes que revisar al azar.

In [ ]:
# ============================================================
# SECCION 22: ANALISIS OPERACIONAL
# ============================================================

# --- TABLA DE DECILES ---
n_te       = len(YTE)
n_deciles  = 10
ordenes_te = np.argsort(-PRB_TE)  # mayor probabilidad primero
tam_decil  = n_te // n_deciles

print('TABLA DE DECILES (test set):')
print(f'  {"Decil":>5} {"N":>6} {"Pos":>5} {"Tasa%":>7} {"Lift":>6} {"RecallCum%":>10}')
print('  ' + '-'*50)
total_pos = int(YTE.sum())
recall_cum = 0
decil_rows = []
for d in range(1, n_deciles+1):
    idx = ordenes_te[(d-1)*tam_decil : d*tam_decil]
    n_d   = len(idx)
    pos_d = int(YTE[idx].sum())
    tasa  = pos_d / max(n_d, 1) * 100
    base_rate = total_pos / n_te * 100
    lift  = tasa / max(base_rate, 1e-9)
    recall_cum += pos_d
    rc_pct = recall_cum / max(total_pos, 1) * 100
    flag = '<-- TOP DECIL' if d == 1 else ''
    print(f'  {d:>5} {n_d:>6} {pos_d:>5} {tasa:>7.2f} {lift:>6.2f} {rc_pct:>10.1f}%  {flag}')
    decil_rows.append(dict(decil=d, n=n_d, pos=pos_d, tasa_pct=round(tasa,2),
                           lift=round(lift,2), recall_cum_pct=round(rc_pct,1)))

df_deciles = pd.DataFrame(decil_rows)
print(f'\n  Tasa base: {base_rate:.2f}%')
print(f'  Top decil concentra {df_deciles.iloc[0]["pos"]}/{total_pos} '
      f'({100*df_deciles.iloc[0]["pos"]/max(total_pos,1):.1f}%) del recall')

# --- VISUALIZACION DECILES ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
colors_dec = ['#e74c3c' if i==0 else '#3498db' for i in range(n_deciles)]
bars_dec = ax.bar(df_deciles['decil'], df_deciles['tasa_pct'],
                  color=colors_dec, alpha=0.85, edgecolor='white')
ax.axhline(base_rate, color='gray', ls='--', lw=2, label=f'Base rate {base_rate:.2f}%')
for bar_, v in zip(bars_dec, df_deciles['tasa_pct']):
    if v > 0:
        ax.text(bar_.get_x()+bar_.get_width()/2, bar_.get_height()+0.01,
                f'{v:.1f}%', ha='center', fontsize=8, fontweight='bold')
ax.set_xlabel('Decil'); ax.set_ylabel('Tasa de incidentes (%)')
ax.set_title('Tasa de incidentes por decil', fontweight='bold'); ax.legend(fontsize=9)

ax = axes[1]
ax.bar(df_deciles['decil'], df_deciles['lift'], color=colors_dec, alpha=0.85, edgecolor='white')
ax.axhline(1.0, color='gray', ls='--', lw=1)
for bar_, v in zip(axes[1].patches, df_deciles['lift']):
    if v > 0.1:
        ax.text(bar_.get_x()+bar_.get_width()/2, bar_.get_height()+0.05,
                f'{v:.2f}x', ha='center', fontsize=8, fontweight='bold')
ax.set_xlabel('Decil'); ax.set_ylabel('Lift')
ax.set_title('Lift por decil (vs tasa base)', fontweight='bold')

ax = axes[2]
ax.plot(df_deciles['decil'], df_deciles['recall_cum_pct'], 'ro-', lw=2.5, ms=7, label='Recall acumulado')
ax.axhline(80, color='gray', ls=':', lw=1, label='80% recall')
ax.fill_between(df_deciles['decil'], df_deciles['recall_cum_pct'], alpha=0.15, color='r')
ax.set_xlabel('Decil'); ax.set_ylabel('Recall acumulado (%)')
ax.set_title('Curva de ganancia acumulada', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Analisis de deciles — poder discriminante operativo', fontweight='bold')
plt.tight_layout(); plt.show()

# --- ESCENARIOS OPERATIVOS ---
print('\n' + '='*75)
FACTOR_ANUAL = max(1, int(round(len(feat_sorted) / max(len(YTE), 1))))  # v2: antes era un 4 hardcodeado
print(f'ESCENARIOS OPERATIVOS (test set, extrapolados x{FACTOR_ANUAL} para proyeccion anual)')
print('='*75)
print(f'  {"Escenario":15} {"Umbral":>7} {"Flagueados":>10} {"TP":>5} {"FP":>6} {"FN":>5} {"Prec%":>7} {"Recall%":>8} {"TP/anio":>8}')
print('  ' + '-'*78)

escenarios = [
    ('Conservador', 0.90),
    ('EQopt',       T_EQ),
    ('Agresivo',    0.35),
]

for nombre, t in escenarios:
    yp   = (PRB_TE >= t).astype(int)
    tp_  = int(((yp==1)&(YTE==1)).sum())
    fp_  = int(((yp==1)&(YTE==0)).sum())
    fn_  = int(((yp==0)&(YTE==1)).sum())
    prec = tp_/max(tp_+fp_,1)*100
    rec  = tp_/max(tp_+fn_,1)*100
    flag = int(yp.sum())
    tp_anio = tp_ * FACTOR_ANUAL
    print(f'  {nombre:15} {t:>7.3f} {flag:>10,} {tp_:>5} {fp_:>6} {fn_:>5} {prec:>7.1f} {rec:>8.1f} {tp_anio:>8}')

# --- SIMULACION SEMANAL CAB ---
print(f'\nSIMULACION SEMANAL CAB (umbral EQopt={T_EQ:.3f}):')
# Asumiendo ~1000 cambios semanales (basado en 14k anuales / 52 semanas)
cambios_semana  = len(cambios_dedup) // 52  # v2: antes hardcodeado en 14764
flag_semana_pct = (PRB_TE >= T_EQ).mean()
flag_semanal    = int(cambios_semana * flag_semana_pct)
tp_semanal      = int(total_pos / 52 * (tp_eq / max(total_pos, 1)))  # estimacion
print(f'  Cambios por semana (estimado): {cambios_semana}')
print(f'  % que supera el umbral       : {flag_semana_pct:.1%}')
print(f'  Cambios flagueados/semana    : {flag_semanal}')
print(f'  Incidentes reales evitados/sem: ~{tp_semanal}')
print(f'  Revisión adicional CAB/semana : {flag_semanal - tp_semanal} falsos positivos')
print(f'  Beneficio: {tp_semanal} incidentes detectados a costo de {flag_semanal} revisiones extra')

---
## Sección 22B (v2) — Incertidumbre: intervalos de confianza bootstrap

Con ~30 positivos en el test, un ROC-AUC de "0.878" es un punto sobre una distribución ancha: mover 2-3 positivos de lugar cambia la métrica en varios puntos. Antes de comprometer cifras con el negocio, hay que reportar **intervalos**, no puntos.

**Método:** remuestreo bootstrap del test set (1,000 réplicas, con reemplazo), IC al 95% por percentiles. Si el IC de PR-AUC del modelo se solapa con el del azar (= prevalencia), la evidencia de valor es débil y hay que decirlo.


In [ ]:
# ============================================================
# SECCION 22B (v2): BOOTSTRAP DEL TEST SET
# ============================================================
rng = np.random.default_rng(42)
N_BOOT = 1000
idx_all = np.arange(len(YTE))

boot_roc, boot_pr, boot_r10 = [], [], []
for _ in range(N_BOOT):
    idx = rng.choice(idx_all, size=len(idx_all), replace=True)
    if YTE[idx].sum() == 0:
        continue  # replica sin positivos: no evaluable
    boot_roc.append(roc_auc_score(YTE[idx], PRB_TE[idx]))
    boot_pr.append(average_precision_score(YTE[idx], PRB_TE[idx]))
    boot_r10.append(recall_top_k(YTE[idx], PRB_TE[idx]))

def ic95(v):
    return np.percentile(v, 2.5), np.percentile(v, 97.5)

roc_lo, roc_hi = ic95(boot_roc)
pr_lo,  pr_hi  = ic95(boot_pr)
r10_lo, r10_hi = ic95(boot_r10)

print(f'Replicas bootstrap validas: {len(boot_roc):,} de {N_BOOT:,}')
print(f'\nIC 95% (test, {int(YTE.sum())} positivos):')
print(f'  ROC-AUC   : {ROC_TE:.4f}  [{roc_lo:.4f} , {roc_hi:.4f}]')
print(f'  PR-AUC    : {res_final["pr_te"]:.4f}  [{pr_lo:.4f} , {pr_hi:.4f}]'
      f'   (azar = prevalencia = {YTE.mean():.4f})')
print(f'  Recall@10%: {recall_top_k(YTE, PRB_TE):.3f}  [{r10_lo:.3f} , {r10_hi:.3f}]   (azar = 0.10)')

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, vals, punto, titulo, azar in [
    (axes[0], boot_roc, ROC_TE, 'ROC-AUC', 0.5),
    (axes[1], boot_pr, res_final['pr_te'], 'PR-AUC', float(YTE.mean())),
    (axes[2], boot_r10, recall_top_k(YTE, PRB_TE), 'Recall@10%', 0.10),
]:
    ax.hist(vals, bins=40, color='#3498db', alpha=0.75, edgecolor='white')
    lo, hi = ic95(vals)
    ax.axvline(punto, color='#e74c3c', lw=2.5, label=f'Punto: {punto:.3f}')
    ax.axvline(lo, color='gray', ls='--', lw=1.5)
    ax.axvline(hi, color='gray', ls='--', lw=1.5, label=f'IC95: [{lo:.3f}, {hi:.3f}]')
    ax.axvline(azar, color='k', ls=':', lw=1.5, label=f'Azar: {azar:.3f}')
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=7.5)
plt.suptitle('Distribucion bootstrap de las metricas de test (1000 replicas)', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nComo comunicarlo al negocio:')
print(f'  "El modelo captura entre {r10_lo:.0%} y {r10_hi:.0%} de los incidentes')
print(f'   revisando solo el 10% de los cambios (mejor estimacion: '
      f'{recall_top_k(YTE, PRB_TE):.0%})."')
print('  Nunca reportar solo el punto: con ~30 positivos la variabilidad es alta.')


---
## Sección 22C (v3) — Backtesting rolling: el desempeño a través del tiempo *(nice to have)*

Una sola partición temporal es **una sola observación** del desempeño del modelo: el test pudo caer en un período atípicamente fácil o difícil. El backtesting rolling simula lo que pasaría en producción con reentrenamiento periódico:

```
Fold 1: [====== TRAIN ======][EVAL]
Fold 2: [========= TRAIN =========][EVAL]
Fold 3: [============ TRAIN ============][EVAL]
...
```

Para cada fold se **re-ajusta todo desde cero con solo el pasado** (TF-IDF incluido — refit por fold, sin fugas) usando la configuración ganadora de la Sección 17, y se evalúa en el bloque siguiente. Lo que se busca:

- **Estabilidad**: si el recall@10% oscila poco entre folds, la cifra reportada al negocio es creíble.
- **Tendencia**: una degradación sostenida en los últimos folds = concept drift → define la **frecuencia de reentrenamiento** del MVP en producción.
- **Sensibilidad al volumen**: si los primeros folds (menos historia) rinden peor, más datos ayudarán — coherente con la learning curve de la Sección 19.

> Nota: con ~1% de prevalencia, cada fold de evaluación tiene pocos positivos; las métricas por fold son ruidosas por construcción. Leer la **mediana y el rango**, no cada punto.


In [ ]:
# ============================================================
# SECCION 22C (v3): BACKTESTING ROLLING (expanding window)
# Reentrena todo (TF-IDF + XGB config ganadora) usando solo el
# pasado de cada fold y evalua en el bloque siguiente.
# ============================================================

N_FOLDS_BT   = 6      # bloques de evaluacion
FRAC_BASE_BT = 0.40   # historia minima antes del primer fold

_n = len(feat_sorted)
_base = int(_n * FRAC_BASE_BT)
_bordes = np.linspace(_base, _n, N_FOLDS_BT + 1).astype(int)

_Xd_all = feat_sorted[all_dense].fillna(0).astype(float).values
_y_all  = feat_sorted['causo_incidente'].values
_txt_all = feat_sorted[texto_col].fillna('').values if texto_col else None
_fechas_bt = (pd.to_datetime(feat_sorted[fecha_cam_col], errors='coerce')
              if fecha_cam_col in feat_sorted.columns else None)

rows_bt = []
print(f'{"Fold":>4} {"train_n":>8} {"eval_n":>7} {"pos_ev":>6} '
      f'{"ROC":>7} {"PR-AUC":>7} {"R@10%":>6}  periodo_eval')
print('-' * 85)

for k in range(N_FOLDS_BT):
    ini, fin = _bordes[k], _bordes[k + 1]
    ytr_k, yev_k = _y_all[:ini], _y_all[ini:fin]
    if ytr_k.sum() < 5:
        print(f'{k+1:>4}  historia con <5 positivos, saltando')
        continue

    # Matrices del fold: TF-IDF re-ajustado SOLO con el pasado del fold
    if texto_col:
        tf_k = TfidfVectorizer(max_features=300, ngram_range=(1, 2), min_df=3,
                               sublinear_tf=True,
                               stop_words=sorted(STOPWORDS_ES) if 'STOPWORDS_ES' in dir() else None)
        Ttr = tf_k.fit_transform(_txt_all[:ini])
        Tev = tf_k.transform(_txt_all[ini:fin])
        Xtr_k = hstack([csr_matrix(_Xd_all[:ini]), Ttr]).toarray()
        Xev_k = hstack([csr_matrix(_Xd_all[ini:fin]), Tev]).toarray()
    else:
        Xtr_k, Xev_k = _Xd_all[:ini], _Xd_all[ini:fin]

    spw_k = int((len(ytr_k) - ytr_k.sum()) / max(ytr_k.sum(), 1))
    m_k = xgb.XGBClassifier(**best_cfg, **FIXED_PARAMS, scale_pos_weight=spw_k)
    m_k.fit(Xtr_k, ytr_k)
    p_ev = m_k.predict_proba(Xev_k)[:, 1]

    roc_k = roc_auc_score(yev_k, p_ev) if yev_k.sum() > 0 else np.nan
    pr_k  = average_precision_score(yev_k, p_ev) if yev_k.sum() > 0 else np.nan
    r10_k = recall_top_k(yev_k, p_ev) if yev_k.sum() > 0 else np.nan

    if _fechas_bt is not None and pd.notna(_fechas_bt.iloc[ini]):
        periodo = f'{_fechas_bt.iloc[ini].date()} a {_fechas_bt.iloc[fin-1].date()}'
    else:
        periodo = f'filas {ini}-{fin}'
    rows_bt.append(dict(fold=k + 1, n_train=ini, n_eval=fin - ini,
                        pos_eval=int(yev_k.sum()), roc=roc_k, pr=pr_k,
                        r10=r10_k, periodo=periodo))
    print(f'{k+1:>4} {ini:>8,} {fin-ini:>7,} {int(yev_k.sum()):>6} '
          f'{roc_k:>7.4f} {pr_k:>7.4f} {r10_k:>6.2f}  {periodo}')

df_bt = pd.DataFrame(rows_bt)

if len(df_bt) >= 3:
    print('\nRESUMEN DEL BACKTEST (folds con positivos):')
    _v = df_bt.dropna(subset=['roc'])
    for met, nom in [('roc', 'ROC-AUC'), ('pr', 'PR-AUC'), ('r10', 'Recall@10%')]:
        print(f'  {nom:11s}: mediana={_v[met].median():.4f}  '
              f'rango=[{_v[met].min():.4f}, {_v[met].max():.4f}]')

    # Tendencia temporal (correlacion fold vs metrica)
    if len(_v) >= 4:
        tend = np.corrcoef(_v['fold'], _v['r10'])[0, 1]
        print(f'  Tendencia del recall@10% a lo largo de los folds: r={tend:+.2f}')
        if tend < -0.5:
            print('  ALERTA: degradacion sostenida -> concept drift; acortar la')
            print('  frecuencia de reentrenamiento en produccion (p.ej. mensual).')
        else:
            print('  Sin degradacion sostenida: reentrenamiento trimestral parece suficiente.')

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for ax, met, nom, azar in [
        (axes[0], 'roc', 'ROC-AUC', 0.5),
        (axes[1], 'pr', 'PR-AUC', float(_y_all.mean())),
        (axes[2], 'r10', 'Recall@10%', 0.10),
    ]:
        ax.plot(df_bt['fold'], df_bt[met], 'o-', color='#2e86de', lw=2.5, ms=8)
        med = df_bt[met].median()
        ax.axhline(med, color='#e74c3c', ls='--', lw=1.5, label=f'Mediana: {med:.3f}')
        ax.axhline(azar, color='gray', ls=':', lw=1.5, label=f'Azar: {azar:.3f}')
        for _, r in df_bt.iterrows():
            ax.annotate(f"{r['pos_eval']}p", (r['fold'], r[met]),
                        textcoords='offset points', xytext=(0, 9),
                        fontsize=7, ha='center', color='#555')
        ax.set_xlabel('Fold (cronologico)')
        ax.set_title(nom, fontweight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        ax.set_xticks(df_bt['fold'])
    plt.suptitle('Backtesting rolling — desempeño con reentrenamiento periodico\n'
                 '(la anotacion Np = positivos en el bloque de evaluacion)',
                 fontweight='bold')
    plt.tight_layout(); plt.show()

    print('\nComo usarlo:')
    print('  - La MEDIANA del recall@10% entre folds es la cifra mas honesta para')
    print('    prometer al negocio (mas conservadora que el test unico).')
    print('  - El rango entre folds dimensiona la variabilidad esperable mes a mes.')
else:
    print('\nMuy pocos folds evaluables para leer tendencia; ampliar el periodo de datos.')


---
## Sección 23 — Weak supervision: ¿podemos recuperar labels perdidos?

### La hipotesis

Solo el 17.6% de los incidentes tiene el campo `causado_por_cambio` informado. ¿Y si los otros 83.4% también fueron causados por cambios pero no se documentó?

La idea de **weak supervision** es usar una regla aproximada (CI + ventana 72h) para generar etiquetas adicionales “edébiles” que podrían ser positivos no documentados.

### Por qué no funcionó

Como demostramos en la Sección 9 (D2/D3), la ventana temporal introduce mucho ruido. Al añadir estos “labels débiles” al set de entrenamiento:
- El modelo ve más positivos pero muchos son falsos
- La señal real se diluye
- El PR-AUC no mejora (en la práctica cae o se mantiene igual)

### Efecto: cero

Esto es una conclusión importante: **la calidad del label supera a la cantidad**. Más datos con etiquetas ruidosas no compensan los pocos datos con etiquetas precisas.

In [ ]:
# ============================================================
# SECCION 23: WEAK SUPERVISION / LABEL RECOVERY
# Hipotesis: usar CI + 72h para generar labels adicionales
# Resultado esperado (confirmado en EDA original): efecto CERO
# ============================================================

print('=== EXPERIMENTO WEAK SUPERVISION (CI + 72h) ===')
print()
print('Hipotesis: cambios que comparten CI con un incidente en 72h posterior')
print('podrian ser causantes no documentados -> etiquetarlos como positivos debiles')
print()

# Calcular ids de weak labels con ventana 72h
if CI_COL and CI_COL_INC and fecha_cam_col and fecha_inc_col:
    pares_72h = pares_ci[pares_ci['delta_dias'] <= 3].copy()  # 3 dias = 72h
    weak_ids  = set(pares_72h['cambio_n']) - cambios_causantes_d1  # nuevos unicamente
    print(f'Labels debiles (CI + 72h, excluyendo D1): {len(weak_ids):,}')
    print(f'  Son {len(weak_ids)/max(N_D1,1):.1f}x mas que los positivos reales')

    if weak_ids and len(weak_ids) < len(feat_sorted) * 0.2:
        # Construir target aumentado
        y_aug = YTR.copy()
        feat_tr = feat_sorted.iloc[:cut_tr].reset_index(drop=True)  # v2: el train real es el 64%
        mask_weak = feat_tr['cambio_n'].isin(weak_ids).values
        y_aug[mask_weak & (y_aug == 0)] = 1  # solo los que no son ya positivos
        n_added = int((mask_weak & (YTR == 0)).sum())
        print(f'  Positivos anadidos al train: {n_added}')
        print(f'  Train original: {YTR.sum()} pos | Train aumentado: {y_aug.sum()} pos')

        # Modelo con labels aumentados
        spw_aug = int((len(y_aug)-y_aug.sum()) / max(y_aug.sum(),1))
        xgb_aug = xgb.XGBClassifier(
            max_depth=1, min_child_weight=500, n_estimators=40,
            subsample=0.5, colsample_bytree=0.5, learning_rate=0.03,
            gamma=0.5, reg_alpha=1.0, reg_lambda=6.0,
            scale_pos_weight=spw_aug,
            eval_metric='logloss',
            random_state=42, n_jobs=-1, verbosity=0
        )
        xgb_aug.fit(XTR_FULL, y_aug)
        roc_aug = roc_auc_score(YTE, xgb_aug.predict_proba(XTE_FULL)[:,1])
        pr_aug  = average_precision_score(YTE, xgb_aug.predict_proba(XTE_FULL)[:,1])

        print()
        print('RESULTADO DEL EXPERIMENTO:')
        print(f'  Modelo D1 original : ROC={ROC_TE:.4f}  PR-AUC={res_final["pr_te"]:.4f}')
        print(f'  Modelo D1 + weak   : ROC={roc_aug:.4f}  PR-AUC={pr_aug:.4f}')
        delta_roc = roc_aug - ROC_TE
        delta_pr  = pr_aug - res_final['pr_te']
        print(f'  Delta ROC: {delta_roc:+.4f}  Delta PR-AUC: {delta_pr:+.4f}')

        if abs(delta_roc) < 0.005 and abs(delta_pr) < 0.005:
            print()
            print('CONCLUSION: EFECTO CERO (diferencia < 5pp en ambas metricas)')
            print('  La calidad del label supera a la cantidad.')
            print('  Etiquetas ruidosas (CI+72h) no aportan senial adicional.')
            print('  El modelo aprende igualmente con o sin los weak labels.')
        elif delta_roc > 0.005:
            print('\nCONCLUSION: MEJORA MARGINAL (monitorear en produccion)')
        else:
            print('\nCONCLUSION: DEGRADACION (las etiquetas debiles hacen dano)')
    else:
        print('Weak ids muy numerosos o ninguno, saltando experimento')
else:
    print('CI columns no disponibles, saltando weak supervision')

print()
print('LECCION APRENDIDA:')
print('  En problemas con pocos positivos y mucho ruido potencial,')
print('  la estrategia correcta es:')
print('  1. Mantener la etiqueta limpia (D1)')
print('  2. Enriquecer las FEATURES, no los labels')
print('  3. Calibrar bien el modelo (escenarios + umbral)')
print('  -> MVP 2: embeddings de texto + mas features de CI en vez de more labels')

---
## Sección 23B (v2) — Persistencia del modelo y scoring de nuevos cambios

Un MVP que vive en variables de un notebook no es un MVP: si el kernel muere, el modelo desaparece. Se empaqueta todo lo necesario para puntuar un cambio nuevo (modelo + vectorizador + lista de features + umbral + calibrador + metadatos de auditoría) en un solo artefacto `joblib`.

También se define `score_nuevos_cambios()`: la función que TI puede llamar cuando llegue un lote de cambios pendientes de CAB. **Importante:** espera un DataFrame que ya pasó por el mismo feature engineering de las Secciones 11/11B — en el MVP 2 esto debe encapsularse en un `Pipeline` sklearn único para eliminar el riesgo de divergencia train/serving.


In [ ]:
# ============================================================
# SECCION 23B (v2): PERSISTENCIA + FUNCION DE SCORING
# ============================================================
import joblib
import datetime

ART_DIR = Path('artefactos_mvp1')
ART_DIR.mkdir(exist_ok=True)

bundle_mvp1 = dict(
    modelo=XGB_FINAL,
    tfidf=tfidf if texto_col else None,
    features_densas=list(all_dense),
    umbral_eqopt=float(T_EQ),
    calibrador=CALIBRADOR_MVP1 if 'CALIBRADOR_MVP1' in dir() else None,
    stopwords_es=sorted(STOPWORDS_ES) if 'STOPWORDS_ES' in dir() else None,
    keywords_riesgo=KEYWORDS_RIESGO if 'KEYWORDS_RIESGO' in dir() else None,
    metadata=dict(
        version='mvp1-v3',
        fecha_entrenamiento=str(datetime.date.today()),
        n_train=int(len(YTR)), n_val=int(len(YVA)), n_test=int(len(YTE)),
        prevalencia_train=float(YTR.mean()),
        roc_auc_test=float(ROC_TE),
        pr_auc_test=float(res_final['pr_te']),
        recall_top10_test=float(recall_top_k(YTE, PRB_TE)),
        config=dict(best_cfg),
        variables_leakage_excluidas=list(FUGA_CONFIRMADAS),
    ),
)
ruta_bundle = ART_DIR / 'modelo_incidentes_mvp1.joblib'
joblib.dump(bundle_mvp1, ruta_bundle)
print(f'Bundle guardado en: {ruta_bundle.resolve()}')
print(f'Tamano: {ruta_bundle.stat().st_size / 1024:.0f} KB')


def score_nuevos_cambios(df_nuevos, bundle=None):
    # Puntua un DataFrame de cambios nuevos con el modelo del MVP 1.
    #
    # REQUISITO: df_nuevos debe traer las mismas columnas de features densas
    # (Secciones 11 y 11B) y, si hay texto, la columna 'texto_mvp1'.
    # Las features de historial deben calcularse con la historia disponible
    # al momento de puntuar (mismo codigo de las Secciones 11/11B).
    #
    # Devuelve el DataFrame con: score crudo, probabilidad calibrada y flag.
    if bundle is None:
        bundle = joblib.load(ruta_bundle)
    faltantes = [c for c in bundle['features_densas'] if c not in df_nuevos.columns]
    if faltantes:
        raise ValueError(f'Faltan columnas de features: {faltantes}')

    Xd = df_nuevos[bundle['features_densas']].fillna(0).astype(float).values
    if bundle['tfidf'] is not None:
        if 'texto_mvp1' not in df_nuevos.columns:
            raise ValueError('Falta la columna texto_mvp1 (ver Seccion 7)')
        Xt = bundle['tfidf'].transform(df_nuevos['texto_mvp1'].fillna(''))
        X = hstack([csr_matrix(Xd), Xt]).toarray()
    else:
        X = Xd

    out = df_nuevos.copy()
    out['score_riesgo'] = bundle['modelo'].predict_proba(X)[:, 1]
    if bundle['calibrador'] is not None:
        out['prob_calibrada'] = bundle['calibrador'].predict_proba(
            out['score_riesgo'].values.reshape(-1, 1))[:, 1]
    else:
        out['prob_calibrada'] = out['score_riesgo']
    out['flag_revision_cab'] = (out['score_riesgo'] >= bundle['umbral_eqopt']).astype(int)
    return out.sort_values('score_riesgo', ascending=False)


# Smoke test: puntuar los ultimos 5 cambios del test como si fueran nuevos
_demo = feat_sorted.iloc[cut:].tail(5)
try:
    demo_scored = score_nuevos_cambios(_demo, bundle_mvp1)
    print('\nSmoke test de score_nuevos_cambios() — ultimos 5 cambios del test:')
    cols_show = ['cambio_n', 'score_riesgo', 'prob_calibrada', 'flag_revision_cab']
    print(demo_scored[[c for c in cols_show if c in demo_scored.columns]].to_string(index=False))
except Exception as e:
    print(f'Smoke test fallo: {e}')


---
## Sección 23C (v2) — Base de monitoreo: PSI (Population Stability Index)

Ningún modelo sobrevive en producción sin monitoreo. Como línea base del MVP se calcula el **PSI** entre train y test — el mismo cálculo que en producción compararía el train contra cada nueva ventana mensual de cambios.

| PSI | Lectura | Acción |
|---|---|---|
| < 0.10 | Población estable | Nada |
| 0.10 – 0.25 | Deriva moderada | Investigar; revisar features afectadas |
| > 0.25 | Deriva severa | Reentrenar / revisar el pipeline de datos |

Además del PSI del **score** (síntoma global), se calcula el PSI de las **features principales** (diagnóstico de dónde viene la deriva).


In [ ]:
# ============================================================
# SECCION 23C (v2): PSI — LINEA BASE DE MONITOREO DE DRIFT
# ============================================================

def psi(esperado, observado, bins=10):
    # PSI con bins por cuantiles de la poblacion esperada (train)
    esperado = np.asarray(esperado, dtype=float)
    observado = np.asarray(observado, dtype=float)
    qs = np.unique(np.quantile(esperado, np.linspace(0, 1, bins + 1)))
    if len(qs) < 3:
        return 0.0  # feature casi constante: PSI no informativo
    qs[0], qs[-1] = -np.inf, np.inf
    e, _ = np.histogram(esperado, bins=qs)
    o, _ = np.histogram(observado, bins=qs)
    e = np.clip(e / max(e.sum(), 1), 1e-6, None)
    o = np.clip(o / max(o.sum(), 1), 1e-6, None)
    return float(np.sum((o - e) * np.log(o / e)))

def semaforo_psi(v):
    return 'OK' if v < 0.10 else ('ALERTA' if v < 0.25 else 'ACCION')

# 1. PSI del score (train vs test)
psi_score = psi(PRB_TR, PRB_TE)
print(f'PSI del score (train vs test): {psi_score:.4f}  [{semaforo_psi(psi_score)}]')

# 2. PSI de las features densas mas importantes
X_dense_tr = feat_sorted[all_dense].iloc[:cut_tr].fillna(0).astype(float)
X_dense_te = feat_sorted[all_dense].iloc[cut:].fillna(0).astype(float)

rows_psi = []
for c in all_dense:
    v = psi(X_dense_tr[c].values, X_dense_te[c].values)
    rows_psi.append(dict(feature=c, psi=round(v, 4), estado=semaforo_psi(v)))
df_psi = pd.DataFrame(rows_psi).sort_values('psi', ascending=False)

print('\nTop 10 features por PSI (candidatas a deriva):')
print(df_psi.head(10).to_string(index=False))

n_alerta = (df_psi['psi'] >= 0.10).sum()
print(f'\nFeatures con PSI >= 0.10: {n_alerta} de {len(df_psi)}')
print('En produccion: correr este mismo calculo cada mes (train vs mes nuevo).')
print('Si el PSI del score supera 0.25 o varias features entran en ALERTA,')
print('disparar revision y considerar reentrenamiento.')


---
# Sección 24 — Conclusión final: lo que hicimos, por qué y qué aprendimos

---

## El problema y por qué no es trivial

Predecir si un cambio va a causar un incidente parece sencillo, pero tiene tres trampas técnicas que hacen fallar a la mayoría de los primeros intentos:

| Trampa | Descripción | Cómo la evitamos |
|---|---|---|
| **Leakage** | Columnas que solo existen después del incidente | Timeline de disponibilidad + lift individual + comparación de modelos |
| **Target ruidoso** | CI + ventana temporal infla 3-5x con coincidencias no causales | 4 definiciones evaluadas empíricamente; D1 gana por precisión de etiqueta |
| **Overfitting** | XGBoost profundo memoriza el train | Regularización multi-nivel; criterio de gap < 3pp |

---

## El viaje analítico (por orden cronológico)

### Etapa 1 — Entender los datos
- Datos crudos: 1,078 incidentes (65k filas) y 65,412 cambios (1.2M filas)
- Post deduplicación correcta: 597 incidentes, 14,764 cambios
- Sin deduplicar: train set 4.7x inflado, correlaciones artificiales

### Etapa 2 — Construir el target bien
- Tres definiciones evaluadas: D1 (link directo), D2 (CI+7d), D3 (CI+14d)
- D1 gana: 100% de precisión de etiqueta vs <30% de D2
- 166 positivos / 14,764 cambios = 1.12% de prevalencia real

### Etapa 3 — Detectar y eliminar leakage
- 9 variables excluidas por timing de negocio + lift > 3x + evidencia empírica
- Un modelo con leakage tiene ROC-AUC ≈0.99 en train pero falla en producción

### Etapa 4 — Construir features legales
- 5 familias: historial CI, temporales, documentales, escala, TF-IDF
- Historial CI calculado con `shift(1)` para no filtrar el futuro
- TF-IDF sobre texto combinado (sin textos de resultado)

### Etapa 5 — Modelado iterativo
- LR baseline: ROC-AUC=0.83, gap=0.04 (overfit leve)
- XGB por defecto: ROC-AUC=0.88 train/test, gap=0.11 (trampa de overfitting)
- XGB regularizado: ROC-AUC=0.87 test, gap=0.015 (✅ válido)

### Etapa 6 — Validación operacional
- Top decil concentra 77.8% del recall con lift 7.79x
- EQopt = F1opt en t=0.682 (robustez)
- Weak supervision: efecto cero (calidad > cantidad de labels)

---

## El modelo recomendado

```python
XGBClassifier(
    max_depth=1, min_child_weight=500, n_estimators=40,
    subsample=0.5, colsample_bytree=0.5, learning_rate=0.03,
    gamma=0.5, reg_alpha=1.0, reg_lambda=6.0,
    scale_pos_weight=84  # calculado como (N-P)/P
)
```

| Métrica | Valor |
|---|---|
| ROC-AUC test | 0.8781 |
| Gini test | 0.7562 |
| KS test | 0.7206 |
| Gap ROC (tr-te) | 0.0153 (✅ < 3pp) |
| Umbral EQopt | 0.682 |
| Recall @ EQopt | 59.3% |
| Precisión @ EQopt | 19.0% |
| Lift top 10% | 7.79x |

---

## Para producción (MVP 2)

1. **Calibración Platt/Isotonic**: las probabilidades de XGBoost no están calibradas. Para uso operativo se necesita que P(Y=1|score=0.7) sea realmente el 70%.
2. **Embeddings de texto**: reemplazar TF-IDF por embeddings semánticos para capturar sinónimos y contexto.
3. **Monitoreo de `business_justification`**: confirmar con el negocio que se llena ANTES de la implementación, no después.
4. **Pipeline de inferencia en tiempo real**: el modelo debe correr en el momento en que el analista somete el cambio al CAB, no en batch.
5. **A/B test en el CAB**: comparar tasa de incidentes en cambios revisados con el modelo vs los no revisados.

In [ ]:
# ============================================================
# SECCION 24: RESUMEN EJECUTIVO FINAL
# ============================================================

print('=' * 70)
print('RESUMEN EJECUTIVO FINAL — MVP 1 INCIDENTES Y CAMBIOS')
print('=' * 70)

print(f'''
DATOS:
  Incidentes unicos : {len(incidentes_dedup):,}  |  Cambios unicos: {len(cambios_dedup):,}
  Positivos (target): {N_D1:,} ({100*N_D1/len(cambios_dedup):.3f}%)  |  Desbalance: 1:{SPW}

MODELO RECOMENDADO:
  XGBoost  md=1  mcw=500  ne=40  sub=0.5  lr=0.03
  gamma=0.5  reg_alpha=1.0  reg_lambda=6.0  scale_pos_weight={SPW}

METRICAS TEST:
  ROC-AUC : {ROC_TE:.4f}   (train: {ROC_TR:.4f}  gap: {GAP_ROC:+.4f} VALIDO)
  Gini    : {GINI_TE:.4f}   KS: {KS_val:.4f}
  PR-AUC  : {res_final['pr_te']:.4f}

USO OPERATIVO @ EQopt (t={T_EQ:.3f}):
  Recall    : {tp_eq/max(tp_eq+fn_eq,1):.1%}
  Precision : {tp_eq/max(tp_eq+fp_eq,1):.1%}
  Lift decil: {df_deciles.iloc[0]['lift']:.2f}x
  Top 10%   : {df_deciles.iloc[0]['recall_cum_pct']:.1f}% del recall

LECCIONES CLAVE:
  1. La etiqueta correcta (D1) supera a la etiqueta ruidosa (D2-D4)
  2. El leakage es el error mas critico y el mas facil de cometer
  3. El historial del CI predice mejor que cualquier variable del cambio
  4. Regularizacion multi-nivel controla overfitting mejor que early stopping
  5. La calidad de labels supera a la cantidad (weak supervision: efecto cero)
''')

# Check de sanidad final
print('CHECK DE SANIDAD:')
asserts_ok = [
    (ROC_TE > 0.70, f'ROC-AUC test > 0.70: {ROC_TE:.4f}'),
    (GAP_ROC < 0.05, f'Gap ROC < 5pp: {GAP_ROC:.4f}'),
    (KS_val > 0.50, f'KS > 0.50: {KS_val:.4f}'),
    (N_D1 > 0, f'Positivos > 0: {N_D1}'),
    (XGB_FINAL is not None, 'Modelo entrenado'),
]
for ok, desc in asserts_ok:
    status = 'OK' if ok else 'FALLO'
    print(f'  [{status}] {desc}')

if all(ok for ok, _ in asserts_ok):
    print('\n  Todos los checks pasaron. El modelo esta listo para evaluacion en produccion.')
else:
    print('\n  Algunos checks fallaron. Revisar antes de proceder.')

print('\n' + '=' * 70)

---
# Sección 25 (v2) — Qué cambió, qué significa y roadmap

## Lo que la v2 corrige (y por qué las cifras pueden moverse)

1. **El test vuelve a ser honesto.** En v1, tanto los hiperparámetros como el umbral se eligieron mirando el test: las métricas reportadas (ROC 0.8781, gap 0.0153) eran optimistas por construcción. Con el split 64/16/20 y la selección en validación, la cifra de test que salga de esta versión es la que se puede defender ante el negocio.
2. **El target se construye sin bugs de alineación.** El fix del `zip` desalineado en la Sección 8 puede cambiar el conjunto de positivos. Verificar tras re-ejecutar que el número de pares y la prevalencia siguen siendo consistentes con lo que Operaciones espera.
3. **El historial del CI ya no ve el futuro.** `hist_fallos_previos_ci` era la feature más importante del modelo v1 y tenía una fuga sutil (contaba fallos cuyo incidente aún no había ocurrido). Si su importancia cae en v2, eso no es una regresión: es la señal real.

## La métrica que importa

- **Para seleccionar modelos:** PR-AUC en validación (Sección 15B)
- **Para reportar a negocio:** recall@10% con su intervalo bootstrap (Sección 22B) — *"revisando el 10% de los cambios más riesgosos capturamos X% de los incidentes, con IC de Y a Z"*
- ROC-AUC/Gini/KS se mantienen como métricas de gobernanza y anti-overfit

## Roadmap MVP 2 (actualizado)

| Prioridad | Item | Nota |
|---|---|---|
| 🔴 Alta | **Pipeline sklearn único** (FE + TF-IDF + modelo en un solo objeto) | Elimina el riesgo de divergencia train/serving que hoy asume `score_nuevos_cambios()` |
| 🔴 Alta | **Validar timing de `business_justification`** con el dueño del proceso | Sigue pendiente desde v1; si se llena post-evento hay que excluirla |
| 🔴 Alta | **Backtesting rolling** (reentrenar mensualmente y evaluar el mes siguiente) | Una sola partición temporal es una sola observación del desempeño |
| 🟡 Media | Embeddings de texto en español (reemplazo/complemento del TF-IDF) | Solo si el benchmark 17B muestra que el texto aporta; medir contra SVD |
| 🟡 Media | Jerarquía de CIs (padre/hijo) para heredar historial | Mitiga el problema de CIs nuevos sin historia |
| 🟡 Media | Ampliar cobertura del label con revisión manual de Operaciones | Weak supervision automática ya demostró efecto cero; la ruta es mejorar la captura del campo `causado_por_cambio` en el proceso |
| 🟢 Baja | SHAP para explicación individual por cambio en el CAB | Complementa los kw_* interpretables de 7B |
| 🟢 Baja | A/B test operativo en el CAB | Meta final: medir incidentes evitados, no métricas de ranking |

## Checklist antes de presentar al CAB

- [ ] Re-ejecutar el notebook completo de arriba a abajo sin errores
- [ ] Confirmar prevalencia y número de positivos tras el fix de la Sección 8
- [ ] Revisar que ninguna feature de leakage aparezca en el top de importancia (Sección 20)
- [ ] Reportar recall@10% **con intervalo** (Sección 22B), no solo el punto
- [ ] Congelar el bundle `joblib` (Sección 23B) y versionarlo junto al notebook
